In [2]:
import pandas as pd
import numpy as np

# Load the frozen final dataset
df = pd.read_csv("../data/himalayan_temporal_final_v3.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (2250, 15)

Columns:
['latitude', 'longitude', 'reference_date', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D']


In [3]:
FEATURES = [
    "RAIN_1D",
    "RAIN_3D",
    "RAIN_7D",
    "RAIN_15D",
    "RAIN_30D",
    "ELEVATION",
    "SLOPE",
    "LANDCOVER_CODE"
]

TARGETS = [
    "NEXT_1D",
    "NEXT_2D",
    "NEXT_3D"
]

X = df[FEATURES].copy()

print("Features:")
print(FEATURES)

print("\nTargets:")
print(TARGETS)

print("\nX shape:", X.shape)

for target in TARGETS:
    print(f"{target}: {df[target].sum()} positive / {len(df) - df[target].sum()} negative")

Features:
['RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE']

Targets:
['NEXT_1D', 'NEXT_2D', 'NEXT_3D']

X shape: (2250, 8)
NEXT_1D: 375 positive / 1875 negative
NEXT_2D: 750 positive / 1500 negative
NEXT_3D: 1125 positive / 1125 negative


In [4]:
# Create 0.5° spatial blocks

df["lat_block"] = np.floor(df["latitude"] / 0.5)
df["lon_block"] = np.floor(df["longitude"] / 0.5)

df["spatial_block"] = (
    df["lat_block"].astype(str)
    + "_"
    + df["lon_block"].astype(str)
)

print("Number of spatial blocks:", df["spatial_block"].nunique())

print("\nSamples per spatial block:")
print(df["spatial_block"].value_counts().describe())

Number of spatial blocks: 58

Samples per spatial block:
count     58.000000
mean      38.793103
std       30.634666
min        1.000000
25%       14.500000
50%       33.500000
75%       54.750000
max      118.000000
Name: count, dtype: float64


In [5]:
# Inspect how spatial blocks are distributed across time

block_years = (
    df.groupby("spatial_block")["reference_date"]
      .agg(["min", "max", "count"])
      .sort_values("min")
)

print(block_years.head(10))
print("\n...")
print(block_years.tail(10))

                      min         max  count
spatial_block                               
61.0_159.0     2007-07-27  2018-09-01     61
58.0_159.0     2007-07-27  2018-09-24     33
61.0_156.0     2007-07-27  2018-08-12    118
59.0_157.0     2007-07-27  2016-08-14     47
64.0_154.0     2007-07-27  2017-06-03     84
60.0_158.0     2007-07-27  2018-08-03     89
60.0_157.0     2007-07-28  2018-09-21     79
62.0_155.0     2007-07-29  2018-09-27     60
61.0_153.0     2007-07-29  2017-06-28     19
59.0_158.0     2007-08-08  2018-08-10     35

...
                      min         max  count
spatial_block                               
66.0_152.0     2010-08-07  2018-08-09      6
58.0_157.0     2010-08-07  2017-04-04      4
66.0_153.0     2010-09-16  2017-02-19     10
65.0_156.0     2011-05-30  2016-07-01      5
64.0_157.0     2011-08-05  2017-02-20      5
63.0_151.0     2013-07-04  2013-07-30      3
60.0_154.0     2013-07-04  2013-07-04      1
57.0_158.0     2013-07-22  2018-08-05      2
57.0_

In [6]:
# Convert reference_date to datetime
df["reference_date"] = pd.to_datetime(df["reference_date"])

# Count samples by year
year_counts = (
    df.groupby(df["reference_date"].dt.year)
      .size()
      .reset_index(name="samples")
)

print(year_counts)
print("\nTotal samples:", year_counts["samples"].sum())

    reference_date  samples
0             2007       42
1             2008       54
2             2009       48
3             2010      528
4             2011      294
5             2012       90
6             2013      480
7             2014       42
8             2015      234
9             2016      144
10            2017      138
11            2018      156

Total samples: 2250


In [7]:
# Check possible temporal train/test cutoffs

for cutoff in [2013, 2014, 2015, 2016, 2017]:
    train = df[df["reference_date"].dt.year < cutoff]
    test = df[df["reference_date"].dt.year >= cutoff]

    print(
        f"Before {cutoff}: "
        f"train={len(train):4d}, "
        f"test={len(test):4d}, "
        f"test%={len(test)/len(df)*100:.1f}%"
    )

Before 2013: train=1056, test=1194, test%=53.1%
Before 2014: train=1536, test= 714, test%=31.7%
Before 2015: train=1578, test= 672, test%=29.9%
Before 2016: train=1812, test= 438, test%=19.5%
Before 2017: train=1956, test= 294, test%=13.1%


In [8]:
# Initial temporal split
# Training: before 2016
# Testing: 2016 onward

train_time = df[df["reference_date"].dt.year < 2016].copy()
test_time = df[df["reference_date"].dt.year >= 2016].copy()

print("Training samples:", len(train_time))
print("Testing samples:", len(test_time))

print("\nTraining date range:")
print(train_time["reference_date"].min(), "→", train_time["reference_date"].max())

print("\nTesting date range:")
print(test_time["reference_date"].min(), "→", test_time["reference_date"].max())

Training samples: 1812
Testing samples: 438

Training date range:
2007-07-27 00:00:00 → 2015-08-21 00:00:00

Testing date range:
2016-05-19 00:00:00 → 2018-10-08 00:00:00


In [9]:
train_blocks = set(train_time["spatial_block"])
test_blocks = set(test_time["spatial_block"])

overlap_blocks = train_blocks.intersection(test_blocks)

print("Training spatial blocks:", len(train_blocks))
print("Testing spatial blocks:", len(test_blocks))
print("Blocks appearing in BOTH:", len(overlap_blocks))

print("\nOverlapping blocks:")
print(sorted(overlap_blocks))

Training spatial blocks: 57
Testing spatial blocks: 56
Blocks appearing in BOTH: 55

Overlapping blocks:
['57.0_158.0', '57.0_159.0', '57.0_160.0', '58.0_157.0', '58.0_158.0', '58.0_159.0', '58.0_160.0', '59.0_155.0', '59.0_156.0', '59.0_157.0', '59.0_158.0', '59.0_159.0', '59.0_160.0', '60.0_155.0', '60.0_156.0', '60.0_157.0', '60.0_158.0', '60.0_159.0', '60.0_160.0', '60.0_161.0', '61.0_153.0', '61.0_154.0', '61.0_155.0', '61.0_156.0', '61.0_157.0', '61.0_158.0', '61.0_159.0', '61.0_160.0', '62.0_152.0', '62.0_153.0', '62.0_154.0', '62.0_155.0', '62.0_156.0', '62.0_157.0', '63.0_152.0', '63.0_153.0', '63.0_154.0', '63.0_155.0', '63.0_156.0', '63.0_157.0', '64.0_151.0', '64.0_152.0', '64.0_153.0', '64.0_154.0', '64.0_155.0', '64.0_156.0', '64.0_157.0', '65.0_151.0', '65.0_152.0', '65.0_153.0', '65.0_154.0', '65.0_155.0', '65.0_156.0', '66.0_152.0', '66.0_153.0']


In [10]:
print("TRAIN TARGET DISTRIBUTION")
print(train_time[TARGETS].sum())

print("\nTEST TARGET DISTRIBUTION")
print(test_time[TARGETS].sum())

print("\nTest samples by LANDSLIDE:")
print(test_time["LANDSLIDE"].value_counts())

TRAIN TARGET DISTRIBUTION
NEXT_1D    302
NEXT_2D    604
NEXT_3D    906
dtype: int64

TEST TARGET DISTRIBUTION
NEXT_1D     73
NEXT_2D    146
NEXT_3D    219
dtype: int64

Test samples by LANDSLIDE:
LANDSLIDE
1    219
0    219
Name: count, dtype: int64


In [11]:
# Freeze the temporal holdout

train_idx = train_time.index
test_idx = test_time.index

print("Train indices:", len(train_idx))
print("Test indices:", len(test_idx))

# Verify there is no overlap
print("Index overlap:", len(set(train_idx) & set(test_idx)))

Train indices: 1812
Test indices: 438
Index overlap: 0


In [12]:
# Prepare 1-day prediction data

X_train = train_time[FEATURES].copy()
X_test = test_time[FEATURES].copy()

y_train = train_time["NEXT_1D"].copy()
y_test = test_time["NEXT_1D"].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

X_train: (1812, 8)
X_test : (438, 8)

y_train distribution:
NEXT_1D
0    1510
1     302
Name: count, dtype: int64

y_test distribution:
NEXT_1D
0    365
1     73
Name: count, dtype: int64


In [13]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

print("1-day Logistic Regression baseline trained successfully.")

1-day Logistic Regression baseline trained successfully.


In [14]:
# Generate 1-day predictions

y_prob = baseline_model.predict_proba(X_test)[:, 1]
y_pred = baseline_model.predict(X_test)

print("First 10 predicted probabilities:")
print(y_prob[:10])

print("\nFirst 10 predicted classes:")
print(y_pred[:10])

First 10 predicted probabilities:
[0.64623373 0.65247291 0.51890499 0.56676273 0.55812391 0.59237716
 0.55027251 0.51320088 0.49436418 0.56149649]

First 10 predicted classes:
[1 1 1 1 1 1 1 1 0 1]


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)

print("1-DAY LOGISTIC REGRESSION")
print("-------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

1-DAY LOGISTIC REGRESSION
-------------------------
Accuracy : 0.5205
Precision: 0.2060
Recall   : 0.6575
F1 Score : 0.3137
ROC-AUC  : 0.5949
PR-AUC   : 0.2338

Confusion Matrix:
[[180 185]
 [ 25  48]]


In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("1-day Random Forest trained successfully.")

1-day Random Forest trained successfully.


In [17]:
# Random Forest predictions

rf_prob = rf_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict(X_test)

accuracy = accuracy_score(y_test, rf_pred)
precision = precision_score(y_test, rf_pred)
recall = recall_score(y_test, rf_pred)
f1 = f1_score(y_test, rf_pred)
roc_auc = roc_auc_score(y_test, rf_prob)
pr_auc = average_precision_score(y_test, rf_prob)

cm = confusion_matrix(y_test, rf_pred)

print("1-DAY RANDOM FOREST")
print("-------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

1-DAY RANDOM FOREST
-------------------
Accuracy : 0.7146
Precision: 0.2547
Recall   : 0.3699
F1 Score : 0.3017
ROC-AUC  : 0.6408
PR-AUC   : 0.2566

Confusion Matrix:
[[286  79]
 [ 46  27]]


In [18]:
from sklearn.model_selection import train_test_split

X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    stratify=y_train,
    random_state=42
)

print("Training subset:", X_train_sub.shape)
print("Validation set :", X_val.shape)

print("\nTraining target:")
print(y_train_sub.value_counts())

print("\nValidation target:")
print(y_val.value_counts())

Training subset: (1449, 8)
Validation set : (363, 8)

Training target:
NEXT_1D
0    1207
1     242
Name: count, dtype: int64

Validation target:
NEXT_1D
0    303
1     60
Name: count, dtype: int64


In [19]:
rf_val_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_val_model.fit(X_train_sub, y_train_sub)

print("Validation Random Forest trained successfully.")

Validation Random Forest trained successfully.


In [20]:
# Validation probabilities

val_prob = rf_val_model.predict_proba(X_val)[:, 1]

print("First 10 validation probabilities:")
print(val_prob[:10])

print("\nProbability range:")
print("Minimum:", val_prob.min())
print("Maximum:", val_prob.max())

First 10 validation probabilities:
[0.14700261 0.49994761 0.34364175 0.3411827  0.34901092 0.54245742
 0.3879685  0.49759441 0.45798142 0.27382102]

Probability range:
Minimum: 0.032684674744436946
Maximum: 0.7758490163561943


In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("Threshold | Precision | Recall | F1")
print("-------------------------------------")

for threshold in thresholds:
    val_pred = (val_prob >= threshold).astype(int)

    precision = precision_score(y_val, val_pred, zero_division=0)
    recall = recall_score(y_val, val_pred, zero_division=0)
    f1 = f1_score(y_val, val_pred, zero_division=0)

    print(
        f"{threshold:9.2f} | "
        f"{precision:9.3f} | "
        f"{recall:6.3f} | "
        f"{f1:5.3f}"
    )

Threshold | Precision | Recall | F1
-------------------------------------
     0.20 |     0.186 |  0.950 | 0.311
     0.25 |     0.196 |  0.933 | 0.325
     0.30 |     0.202 |  0.883 | 0.329
     0.35 |     0.197 |  0.750 | 0.311
     0.40 |     0.163 |  0.483 | 0.244
     0.45 |     0.146 |  0.333 | 0.203
     0.50 |     0.118 |  0.183 | 0.144


In [22]:
from sklearn.metrics import roc_auc_score, average_precision_score

val_roc_auc = roc_auc_score(y_val, val_prob)
val_pr_auc = average_precision_score(y_val, val_prob)

print("Validation ROC-AUC:", round(val_roc_auc, 4))
print("Validation PR-AUC :", round(val_pr_auc, 4))

Validation ROC-AUC: 0.5222
Validation PR-AUC : 0.1585


In [23]:
from sklearn.model_selection import StratifiedGroupKFold

# Training pool only — final 2016–2018 test remains untouched
cv_data = train_time.copy()

cv_X = cv_data[FEATURES]
cv_y = cv_data["NEXT_1D"]
cv_groups = cv_data["spatial_block"]

print("CV samples:", len(cv_data))
print("Spatial groups:", cv_groups.nunique())
print("Positive:", cv_y.sum())
print("Negative:", (cv_y == 0).sum())

CV samples: 1812
Spatial groups: 57
Positive: 302
Negative: 1510


In [24]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_info = []

for fold, (train_idx_cv, val_idx_cv) in enumerate(
    cv.split(cv_X, cv_y, groups=cv_groups), start=1
):
    y_tr = cv_y.iloc[train_idx_cv]
    y_va = cv_y.iloc[val_idx_cv]

    fold_info.append({
        "fold": fold,
        "train_samples": len(train_idx_cv),
        "validation_samples": len(val_idx_cv),
        "train_positive": int(y_tr.sum()),
        "validation_positive": int(y_va.sum()),
        "validation_groups": cv_groups.iloc[val_idx_cv].nunique()
    })

fold_df = pd.DataFrame(fold_info)

print(fold_df.to_string(index=False))

 fold  train_samples  validation_samples  train_positive  validation_positive  validation_groups
    1           1451                 361             245                   57                 10
    2           1445                 367             236                   66                 12
    3           1454                 358             246                   56                 11
    4           1452                 360             244                   58                 12
    5           1446                 366             237                   65                 12


In [25]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

fold_results = []

for fold, (train_idx_cv, val_idx_cv) in enumerate(
    cv.split(cv_X, cv_y, groups=cv_groups), start=1
):
    
    X_tr = cv_X.iloc[train_idx_cv]
    X_va = cv_X.iloc[val_idx_cv]
    y_tr = cv_y.iloc[train_idx_cv]
    y_va = cv_y.iloc[val_idx_cv]

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_tr, y_tr)

    prob = model.predict_proba(X_va)[:, 1]
    pred = (prob >= 0.5).astype(int)

    fold_results.append({
        "fold": fold,
        "ROC_AUC": roc_auc_score(y_va, prob),
        "PR_AUC": average_precision_score(y_va, prob),
        "Precision": precision_score(y_va, pred, zero_division=0),
        "Recall": recall_score(y_va, pred, zero_division=0),
        "F1": f1_score(y_va, pred, zero_division=0)
    })

cv_results = pd.DataFrame(fold_results)

print(cv_results.round(4).to_string(index=False))

print("\nMean:")
print(cv_results.drop(columns="fold").mean().round(4))

print("\nStd:")
print(cv_results.drop(columns="fold").std().round(4))

 fold  ROC_AUC  PR_AUC  Precision  Recall     F1
    1   0.6427  0.2238     0.2394  0.2982 0.2656
    2   0.6437  0.2655     0.2967  0.4091 0.3439
    3   0.6841  0.2618     0.2556  0.4107 0.3151
    4   0.6093  0.2272     0.1970  0.2241 0.2097
    5   0.6232  0.2341     0.2576  0.2615 0.2595

Mean:
ROC_AUC      0.6406
PR_AUC       0.2425
Precision    0.2492
Recall       0.3207
F1           0.2788
dtype: float64

Std:
ROC_AUC      0.0282
PR_AUC       0.0197
Precision    0.0360
Recall       0.0855
F1           0.0522
dtype: float64


In [26]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

oof_prob = np.zeros(len(cv_data))
oof_true = cv_y.to_numpy()

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx_cv, val_idx_cv) in enumerate(
    cv.split(cv_X, cv_y, groups=cv_groups), start=1
):
    
    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        cv_X.iloc[train_idx_cv],
        cv_y.iloc[train_idx_cv]
    )

    oof_prob[val_idx_cv] = model.predict_proba(
        cv_X.iloc[val_idx_cv]
    )[:, 1]

print("Out-of-fold ROC-AUC:",
      round(roc_auc_score(oof_true, oof_prob), 4))

print("Out-of-fold PR-AUC:",
      round(average_precision_score(oof_true, oof_prob), 4))

print("\nProbability range:")
print("Min:", round(oof_prob.min(), 4))
print("Max:", round(oof_prob.max(), 4))

Out-of-fold ROC-AUC: 0.6377
Out-of-fold PR-AUC: 0.2354

Probability range:
Min: 0.0185
Max: 0.7393


In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.05, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:
    pred = (oof_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision_score(oof_true, pred, zero_division=0),
        "Recall": recall_score(oof_true, pred, zero_division=0),
        "F1": f1_score(oof_true, pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))

 Threshold  Precision  Recall     F1
      0.05     0.1679  0.9934 0.2872
      0.10     0.1747  0.9834 0.2967
      0.15     0.1837  0.9768 0.3092
      0.20     0.1891  0.9503 0.3154
      0.25     0.1993  0.9272 0.3281
      0.30     0.2069  0.8576 0.3333
      0.35     0.2132  0.7384 0.3309
      0.40     0.2217  0.6159 0.3260
      0.45     0.2356  0.4603 0.3117
      0.50     0.2526  0.3212 0.2828


In [28]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(oof_true, oof_prob)

prob_true, prob_pred = calibration_curve(
    oof_true,
    oof_prob,
    n_bins=5,
    strategy="quantile"
)

print("Brier score:", round(brier, 4))

print("\nCalibration:")
for p_pred, p_true in zip(prob_pred, prob_true):
    print(
        f"Predicted: {p_pred:.3f}  |  "
        f"Observed: {p_true:.3f}"
    )

Brier score: 0.1824

Calibration:
Predicted: 0.137  |  Observed: 0.055
Predicted: 0.292  |  Observed: 0.141
Predicted: 0.383  |  Observed: 0.171
Predicted: 0.462  |  Observed: 0.213
Predicted: 0.568  |  Observed: 0.253


In [29]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(oof_true, oof_prob)

prob_true, prob_pred = calibration_curve(
    oof_true,
    oof_prob,
    n_bins=5,
    strategy="quantile"
)

print("Brier score:", round(brier, 4))

print("\nCalibration:")
for p_pred, p_true in zip(prob_pred, prob_true):
    print(
        f"Predicted: {p_pred:.3f}  |  "
        f"Observed: {p_true:.3f}"
    )

Brier score: 0.1824

Calibration:
Predicted: 0.137  |  Observed: 0.055
Predicted: 0.292  |  Observed: 0.141
Predicted: 0.383  |  Observed: 0.171
Predicted: 0.462  |  Observed: 0.213
Predicted: 0.568  |  Observed: 0.253


In [30]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

calibrated_rf = CalibratedClassifierCV(
    estimator=RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    method="sigmoid",
    cv=5
)

calibrated_rf.fit(X_train, y_train)

cal_prob = calibrated_rf.predict_proba(X_train)[:, 1]

print("Training calibrated probability range:")
print("Min:", round(cal_prob.min(), 4))
print("Max:", round(cal_prob.max(), 4))

Training calibrated probability range:
Min: 0.0998
Max: 0.3811


In [31]:
from sklearn.model_selection import StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss

cal_oof_prob = np.zeros(len(cv_data))

cal_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for train_idx_cal, val_idx_cal in cal_cv.split(cv_X, cv_y):

    model = CalibratedClassifierCV(
        estimator=RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        method="sigmoid",
        cv=3
    )

    model.fit(
        cv_X.iloc[train_idx_cal],
        cv_y.iloc[train_idx_cal]
    )

    cal_oof_prob[val_idx_cal] = model.predict_proba(
        cv_X.iloc[val_idx_cal]
    )[:, 1]

print("Calibrated OOF Brier score:",
      round(brier_score_loss(cv_y, cal_oof_prob), 4))

print("Probability range:")
print("Min:", round(cal_oof_prob.min(), 4))
print("Max:", round(cal_oof_prob.max(), 4))

Calibrated OOF Brier score: 0.1446
Probability range:
Min: 0.0921
Max: 0.298


In [32]:
from sklearn.calibration import calibration_curve

cal_true, cal_pred = calibration_curve(
    cv_y,
    cal_oof_prob,
    n_bins=5,
    strategy="quantile"
)

print("Calibrated OOF calibration:")

for predicted, observed in zip(cal_pred, cal_true):
    print(
        f"Predicted: {predicted:.3f}  |  "
        f"Observed: {observed:.3f}"
    )

Calibrated OOF calibration:
Predicted: 0.127  |  Observed: 0.168
Predicted: 0.150  |  Observed: 0.202
Predicted: 0.169  |  Observed: 0.235
Predicted: 0.195  |  Observed: 0.171
Predicted: 0.246  |  Observed: 0.058


In [33]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Train calibrated model using ONLY the training period
final_model = CalibratedClassifierCV(
    estimator=RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    method="sigmoid",
    cv=5
)

final_model.fit(X_train, y_train)

# Final untouched test predictions
test_prob = final_model.predict_proba(X_test)[:, 1]

# Frozen threshold from training/OOF analysis
TEST_THRESHOLD = 0.30
test_pred = (test_prob >= TEST_THRESHOLD).astype(int)

print("FINAL TEST RESULTS")
print("------------------")
print("ROC-AUC :", round(roc_auc_score(y_test, test_prob), 4))
print("PR-AUC  :", round(average_precision_score(y_test, test_prob), 4))
print("Brier   :", round(brier_score_loss(y_test, test_prob), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1       :", round(f1_score(y_test, test_pred, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nProbability range:")
print("Min:", round(test_prob.min(), 4))
print("Max:", round(test_prob.max(), 4))

FINAL TEST RESULTS
------------------
ROC-AUC : 0.3771
PR-AUC  : 0.1404
Brier   : 0.1612
Precision: 0.0672
Recall   : 0.1233
F1       : 0.087

Confusion Matrix:
[[240 125]
 [ 64   9]]

Probability range:
Min: 0.1177
Max: 0.3777


In [34]:
# Compare training-period and future-test feature distributions

comparison = pd.DataFrame({
    "Train_mean": X_train.mean(),
    "Test_mean": X_test.mean(),
    "Train_median": X_train.median(),
    "Test_median": X_test.median(),
    "Train_std": X_train.std(),
    "Test_std": X_test.std()
})

comparison["Mean_difference"] = (
    comparison["Test_mean"] - comparison["Train_mean"]
)

comparison["Mean_difference_%"] = (
    comparison["Mean_difference"]
    / comparison["Train_mean"].replace(0, np.nan)
) * 100

print(comparison.round(3).to_string())

                Train_mean  Test_mean  Train_median  Test_median  Train_std  Test_std  Mean_difference  Mean_difference_%
RAIN_1D             10.942     11.466         0.000        0.000     22.325    19.807            0.524              4.787
RAIN_3D             35.693     30.890        19.678       15.663     46.758    39.096           -4.803            -13.456
RAIN_7D             84.494     60.532        69.222       45.026     76.071    60.930          -23.962            -28.359
RAIN_15D           175.873    141.443       161.555      124.959    125.169   117.878          -34.430            -19.576
RAIN_30D           310.174    274.794       292.323      256.140    214.591   206.604          -35.380            -11.406
ELEVATION         2142.591   2110.059      1689.500     1616.000   1469.826  1526.464          -32.531             -1.518
SLOPE               25.554     25.304        25.996       25.599     13.691    14.397           -0.249             -0.976
LANDCOVER_CODE      30.4

In [35]:
from scipy.stats import ks_2samp

shift_results = []

for feature in FEATURES:
    train_values = X_train[feature].dropna()
    test_values = X_test[feature].dropna()

    statistic, p_value = ks_2samp(
        train_values,
        test_values
    )

    shift_results.append({
        "Feature": feature,
        "KS_statistic": statistic,
        "p_value": p_value
    })

shift_df = pd.DataFrame(shift_results)

print(shift_df.sort_values(
    "KS_statistic",
    ascending=False
).round(4).to_string(index=False))

       Feature  KS_statistic  p_value
       RAIN_7D        0.1577   0.0000
      RAIN_15D        0.1339   0.0000
      RAIN_30D        0.0968   0.0025
     ELEVATION        0.0886   0.0073
       RAIN_1D        0.0557   0.2145
       RAIN_3D        0.0528   0.2680
         SLOPE        0.0517   0.2909
LANDCOVER_CODE        0.0407   0.5869


In [36]:
for feature in FEATURES:
    print(f"\n===== {feature} =====")

    train_summary = train_time.groupby("NEXT_1D")[feature].agg(
        ["count", "mean", "median"]
    )

    test_summary = test_time.groupby("NEXT_1D")[feature].agg(
        ["count", "mean", "median"]
    )

    print("\nTRAIN (2007-2015)")
    print(train_summary.round(3))

    print("\nTEST (2016-2018)")
    print(test_summary.round(3))


===== RAIN_1D =====

TRAIN (2007-2015)
         count    mean  median
NEXT_1D                       
0         1510  10.679     0.0
1          302  12.254     0.0

TEST (2016-2018)
         count    mean  median
NEXT_1D                       
0          365  11.337     0.0
1           73  12.108     0.0

===== RAIN_3D =====

TRAIN (2007-2015)
         count    mean  median
NEXT_1D                       
0         1510  35.806  19.574
1          302  35.131  19.736

TEST (2016-2018)
         count    mean  median
NEXT_1D                       
0          365  30.786  13.248
1           73  31.415  19.881

===== RAIN_7D =====

TRAIN (2007-2015)
         count    mean  median
NEXT_1D                       
0         1510  82.992  68.734
1          302  92.003  71.424

TEST (2016-2018)
         count    mean  median
NEXT_1D                       
0          365  59.782  44.948
1           73  64.284  54.534

===== RAIN_15D =====

TRAIN (2007-2015)
         count     mean   median
NEXT_1D 

In [37]:
rows = []

for feature in FEATURES:
    for target in [0, 1]:

        train_values = train_time.loc[
            train_time["NEXT_1D"] == target, feature
        ]

        test_values = test_time.loc[
            test_time["NEXT_1D"] == target, feature
        ]

        rows.append({
            "Feature": feature,
            "Target": target,
            "Train_mean": train_values.mean(),
            "Test_mean": test_values.mean(),
            "Train_median": train_values.median(),
            "Test_median": test_values.median()
        })

relationship_df = pd.DataFrame(rows)

print(
    relationship_df.round(2).to_string(index=False)
)

       Feature  Target  Train_mean  Test_mean  Train_median  Test_median
       RAIN_1D       0       10.68      11.34          0.00         0.00
       RAIN_1D       1       12.25      12.11          0.00         0.00
       RAIN_3D       0       35.81      30.79         19.57        13.25
       RAIN_3D       1       35.13      31.41         19.74        19.88
       RAIN_7D       0       82.99      59.78         68.73        44.95
       RAIN_7D       1       92.00      64.28         71.42        54.53
      RAIN_15D       0      172.98     139.06        159.30       120.24
      RAIN_15D       1      190.35     153.34        179.46       135.36
      RAIN_30D       0      304.37     270.76        283.77       253.42
      RAIN_30D       1      339.21     294.96        316.58       284.15
     ELEVATION       0     2221.72    2183.40       1741.00      1623.00
     ELEVATION       1     1746.95    1743.37       1552.50      1369.00
         SLOPE       0       25.51      25.03      

In [38]:
# Compare positive and negative spatial locations

from sklearn.metrics import pairwise_distances

positive_coords = train_time[
    train_time["NEXT_1D"] == 1
][["latitude", "longitude"]].to_numpy()

negative_coords = train_time[
    train_time["NEXT_1D"] == 0
][["latitude", "longitude"]].to_numpy()

print("Positive locations:", len(positive_coords))
print("Negative locations:", len(negative_coords))

print("\nLatitude:")
print(
    "Positive:",
    round(positive_coords[:, 0].min(), 3),
    "to",
    round(positive_coords[:, 0].max(), 3)
)
print(
    "Negative:",
    round(negative_coords[:, 0].min(), 3),
    "to",
    round(negative_coords[:, 0].max(), 3)
)

print("\nLongitude:")
print(
    "Positive:",
    round(positive_coords[:, 1].min(), 3),
    "to",
    round(positive_coords[:, 1].max(), 3)
)
print(
    "Negative:",
    round(negative_coords[:, 1].min(), 3),
    "to",
    round(negative_coords[:, 1].max(), 3)
)

Positive locations: 302
Negative locations: 1510

Latitude:
Positive: 28.866 to 32.757
Negative: 28.778 to 33.157

Longitude:
Positive: 75.957 to 80.858
Negative: 75.83 to 80.858


In [39]:
from sklearn.neighbors import BallTree
import numpy as np

# Coordinates in radians
positive_rad = np.radians(positive_coords)
negative_rad = np.radians(negative_coords)

# BallTree with Haversine distance
tree = BallTree(positive_rad, metric="haversine")

distances, _ = tree.query(
    negative_rad,
    k=1
)

# Convert radians → km
nearest_positive_km = distances[:, 0] * 6371.0

print("Nearest positive distance for negative samples")
print("----------------------------------------------")
print("Minimum :", round(nearest_positive_km.min(), 2), "km")
print("Median  :", round(np.median(nearest_positive_km), 2), "km")
print("Mean    :", round(nearest_positive_km.mean(), 2), "km")
print("Maximum :", round(nearest_positive_km.max(), 2), "km")

print("\nPercentiles:")
for p in [25, 50, 75, 90, 95]:
    print(
        f"{p}th percentile:",
        round(np.percentile(nearest_positive_km, p), 2),
        "km"
    )

Nearest positive distance for negative samples
----------------------------------------------
Minimum : 0.0 km
Median  : 6.38 km
Mean    : 10.15 km
Maximum : 62.19 km

Percentiles:
25th percentile: 0.0 km
50th percentile: 6.38 km
75th percentile: 16.57 km
90th percentile: 27.32 km
95th percentile: 35.13 km


In [40]:
# Check coordinate overlap between positive and negative samples

positive_set = set(
    zip(
        train_time.loc[train_time["NEXT_1D"] == 1, "latitude"].round(6),
        train_time.loc[train_time["NEXT_1D"] == 1, "longitude"].round(6)
    )
)

negative_set = set(
    zip(
        train_time.loc[train_time["NEXT_1D"] == 0, "latitude"].round(6),
        train_time.loc[train_time["NEXT_1D"] == 0, "longitude"].round(6)
    )
)

overlap = positive_set.intersection(negative_set)

print("Unique positive locations:", len(positive_set))
print("Unique negative locations:", len(negative_set))
print("Shared locations:", len(overlap))

print(
    "Percentage of positive locations also used as negative:",
    round(100 * len(overlap) / len(positive_set), 2),
    "%"
)

Unique positive locations: 301
Unique negative locations: 1207
Shared locations: 301
Percentage of positive locations also used as negative: 100.0 %


In [41]:
# Check whether negative samples occur very close to positive events
# in both space and time.

from sklearn.neighbors import BallTree
import numpy as np
import pandas as pd

# Positive event-like samples in training period
pos = train_time[train_time["NEXT_1D"] == 1].copy()
neg = train_time[train_time["NEXT_1D"] == 0].copy()

# Convert coordinates to radians
pos_coords = np.radians(
    pos[["latitude", "longitude"]].to_numpy()
)

neg_coords = np.radians(
    neg[["latitude", "longitude"]].to_numpy()
)

tree = BallTree(pos_coords, metric="haversine")

dist_rad, nearest_idx = tree.query(
    neg_coords,
    k=1
)

dist_km = dist_rad[:, 0] * 6371.0

nearest_pos = pos.iloc[nearest_idx[:, 0]].reset_index(drop=True)
neg_reset = neg.reset_index(drop=True)

date_diff = (
    nearest_pos["reference_date"] -
    neg_reset["reference_date"]
).abs().dt.days

audit = pd.DataFrame({
    "distance_km": dist_km,
    "date_difference_days": date_diff
})

print("Negative samples:", len(audit))
print("Within 1 km of a positive:", (audit["distance_km"] <= 1).sum())
print("Within 5 km:", (audit["distance_km"] <= 5).sum())
print("Within 10 km:", (audit["distance_km"] <= 10).sum())

print("\nWithin 1 km AND 3 days:",
      ((audit["distance_km"] <= 1) &
       (audit["date_difference_days"] <= 3)).sum())

print("Within 5 km AND 3 days:",
      ((audit["distance_km"] <= 5) &
       (audit["date_difference_days"] <= 3)).sum())

Negative samples: 1510
Within 1 km of a positive: 604
Within 5 km: 700
Within 10 km: 891

Within 1 km AND 3 days: 602
Within 5 km AND 3 days: 607


In [42]:
# Check exact coordinate overlap among the close negative samples

pos_coords_set = set(
    zip(
        pos["latitude"].round(6),
        pos["longitude"].round(6)
    )
)

neg_coords_df = neg.copy()

neg_coords_df["exact_same_location"] = list(
    zip(
        neg_coords_df["latitude"].round(6),
        neg_coords_df["longitude"].round(6)
    )
)

neg_coords_df["exact_same_location"] = (
    neg_coords_df["exact_same_location"].isin(pos_coords_set)
)

print(
    "Negative samples at exact positive locations:",
    neg_coords_df["exact_same_location"].sum()
)

print(
    "Percentage:",
    round(
        100 * neg_coords_df["exact_same_location"].mean(),
        2
    ),
    "%"
)

Negative samples at exact positive locations: 604
Percentage: 40.0 %


In [43]:
negative_features = pd.read_csv("../data/negative_features_v3.csv")

print(negative_features.columns.tolist())
print(negative_features.head())


['latitude', 'longitude', 'reference_date', 'RAIN_15D', 'RAIN_1D', 'RAIN_30D', 'RAIN_3D', 'RAIN_7D', 'ELEVATION', 'LANDCOVER_CODE', 'SLOPE']
    latitude  longitude reference_date    RAIN_15D  RAIN_1D    RAIN_30D  \
0  32.175084  78.428219     2011-08-11   31.763581      0.0  123.756759   
1  31.016665  77.063216     2011-08-10  163.937138      0.0  387.437702   
2  31.687979  77.495389     2011-08-09  288.352900      0.0  422.845484   
3  31.918540  77.859082     2014-07-22   70.704994      0.0   86.017031   
4  31.640529  77.262023     2014-07-21  112.768549      0.0  138.818800   

     RAIN_3D    RAIN_7D  ELEVATION  LANDCOVER_CODE      SLOPE  
0   0.000000   0.000000       5552              70   4.471054  
1  30.211802  60.423603       1444              10  24.461479  
2  83.160906  83.160906       2919              30  31.920373  
3   0.000000  49.211721       4916              70   5.652798  
4   0.000000  62.629190       2807              10  30.560307  


In [44]:
candidate_pool = pd.read_csv(
    "../data/negative_safe_candidate_pool_v2.csv"
)

print("Shape:", candidate_pool.shape)
print("Columns:", candidate_pool.columns.tolist())
print(candidate_pool.head())

Shape: (11901, 3)
Columns: ['latitude', 'longitude', 'nearest_event_km']
    latitude  longitude  nearest_event_km
0  33.160126  76.456053         66.844491
1  33.158295  76.463849         66.273663
2  33.154722  76.465480         65.858360
3  33.148490  76.496609         63.716602
4  33.123340  76.457237         63.420684


In [45]:
positive_data = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)

positive_data["reference_date"] = pd.to_datetime(
    positive_data["reference_date"]
)

print("Shape:", positive_data.shape)
print("Columns:", positive_data.columns.tolist())

print(
    positive_data[
        ["latitude", "longitude", "reference_date", "event_date", "positive_horizon"]
    ].head()
)

Shape: (1125, 17)
Columns: ['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE']
   latitude  longitude reference_date  event_date  positive_horizon
0   28.8665    79.5682     2011-08-11  2011-08-12                 1
1   28.8665    79.5682     2011-08-10  2011-08-12                 2
2   28.8665    79.5682     2011-08-09  2011-08-12                 3
3   29.1585    79.1464     2014-07-22  2014-07-23                 1
4   29.1585    79.1464     2014-07-21  2014-07-23                 2


In [46]:
from sklearn.neighbors import BallTree
import numpy as np

# Use unique positive event locations
positive_locations = (
    positive_data[
        ["latitude", "longitude"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

candidate_coords = np.radians(
    candidate_pool[["latitude", "longitude"]].to_numpy()
)

positive_coords = np.radians(
    positive_locations[["latitude", "longitude"]].to_numpy()
)

candidate_tree = BallTree(
    candidate_coords,
    metric="haversine"
)

# Find candidates within 25 km of each positive location
distances, _ = candidate_tree.query_radius(
    positive_coords,
    r=25 / 6371.0,
    return_distance=True
)

counts = np.array([
    len(d) for d in distances
])

print("Positive locations:", len(positive_locations))

print("\nSafe candidates within 25 km:")
print("Minimum:", counts.min())
print("Median :", int(np.median(counts)))
print("Mean   :", round(counts.mean(), 1))
print("Maximum:", counts.max())

print("\nLocations with:")
print("0 candidates :", (counts == 0).sum())
print("1+ candidates:", (counts > 0).sum())
print("5+ candidates:", (counts >= 5).sum())
print("10+ candidates:", (counts >= 10).sum())

Positive locations: 374

Safe candidates within 25 km:
Minimum: 96
Median : 215
Mean   : 210.6
Maximum: 264

Locations with:
0 candidates : 0
1+ candidates: 374
5+ candidates: 374
10+ candidates: 374


In [47]:
# Find the nearest safe candidate for each positive location

distances, indices = candidate_tree.query(
    positive_coords,
    k=1
)

nearest_distances_km = distances[:, 0] * 6371.0
nearest_indices = indices[:, 0]

print("Nearest candidate distance:")
print("Minimum:", round(nearest_distances_km.min(), 3), "km")
print("Median :", round(np.median(nearest_distances_km), 3), "km")
print("Mean   :", round(nearest_distances_km.mean(), 3), "km")
print("Maximum:", round(nearest_distances_km.max(), 3), "km")

print("\nUnique candidate locations selected:",
      len(set(nearest_indices)))

print("Positive locations:", len(positive_locations))

Nearest candidate distance:
Minimum: 1.014 km
Median : 2.024 km
Mean   : 2.151 km
Maximum: 4.572 km

Unique candidate locations selected: 260
Positive locations: 374


In [48]:
# Count safe candidates within 10 km of each positive location

distances_10, _ = candidate_tree.query_radius(
    positive_coords,
    r=10 / 6371.0,
    return_distance=True
)

counts_10 = np.array([len(d) for d in distances_10])

print("Safe candidates within 10 km:")
print("Minimum:", counts_10.min())
print("Median :", int(np.median(counts_10)))
print("Mean   :", round(counts_10.mean(), 1))
print("Maximum:", counts_10.max())

print("\nLocations with:")
print("0 candidates :", (counts_10 == 0).sum())
print("1+ candidates:", (counts_10 > 0).sum())
print("5+ candidates:", (counts_10 >= 5).sum())
print("10+ candidates:", (counts_10 >= 10).sum())

Safe candidates within 10 km:
Minimum: 16
Median : 33
Mean   : 33.1
Maximum: 52

Locations with:
0 candidates : 0
1+ candidates: 374
5+ candidates: 374
10+ candidates: 374


In [49]:
selected_controls = []
used_candidates = set()

distances_10, indices_10 = candidate_tree.query_radius(
    positive_coords,
    r=10 / 6371.0,
    return_distance=True,
    sort_results=True
)

# Process locations with the fewest options first
order = sorted(
    range(len(indices_10)),
    key=lambda i: len(indices_10[i])
)

for pos_idx in order:

    found = False

    for cand_id in indices_10[pos_idx]:
        cand_id = int(cand_id)

        if cand_id not in used_candidates:
            used_candidates.add(cand_id)

            selected_controls.append({
                "positive_location_index": pos_idx,
                "control_candidate_index": cand_id
            })

            found = True
            break

    if not found:
        print("Could not assign control to positive location:", pos_idx)
        print("Candidates available:", len(indices_10[pos_idx]))
        print("Controls already assigned:", len(selected_controls))
        break

print("Controls successfully assigned:", len(selected_controls))
print("Unique controls:", len(used_candidates))

Could not assign control to positive location: 108
Candidates available: 17
Controls already assigned: 1
Controls successfully assigned: 1
Unique controls: 1


In [50]:
from scipy.optimize import linear_sum_assignment
import numpy as np
import pandas as pd

# Get 50 nearest safe candidates for every positive location
distances, indices = candidate_tree.query(
    positive_coords,
    k=50
)

# Convert distances from radians to km
distances_km = distances * 6371.0

# Only allow candidates within 10 km
cost = distances_km.copy()
cost[cost > 10] = 1e6

# Solve one-to-one assignment
row_ind, col_ind = linear_sum_assignment(cost)

# Check whether every positive location received a valid candidate
assigned_distances = cost[row_ind, col_ind]

print("Positive locations:", len(positive_locations))
print("Assignments:", len(row_ind))
print("Invalid assignments:", (assigned_distances >= 1e6).sum())

print("\nAssignment distance:")
valid = assigned_distances < 1e6
print("Minimum:", round(assigned_distances[valid].min(), 3), "km")
print("Median :", round(np.median(assigned_distances[valid]), 3), "km")
print("Mean   :", round(assigned_distances[valid].mean(), 3), "km")
print("Maximum:", round(assigned_distances[valid].max(), 3), "km")

print("\nUnique candidates:",
      len(set(indices[row_ind, col_ind][valid])))

Positive locations: 374
Assignments: 50
Invalid assignments: 0

Assignment distance:
Minimum: 1.014 km
Median : 6.883 km
Mean   : 6.473 km
Maximum: 9.974 km

Unique candidates: 48


In [51]:
from scipy.optimize import linear_sum_assignment
import numpy as np
import pandas as pd

# Get 500 nearest safe candidates for every positive location
distances, indices = candidate_tree.query(
    positive_coords,
    k=500
)

# Convert radians to kilometers
distances_km = distances * 6371.0

# Candidates farther than 10 km are not allowed
cost = distances_km.copy()
cost[cost > 10] = 1e6

# Find globally optimal one-to-one assignment
row_ind, col_ind = linear_sum_assignment(cost)

assigned_distances = cost[row_ind, col_ind]

valid = assigned_distances < 1e6

print("Positive locations:", len(positive_locations))
print("Assignments:", len(row_ind))
print("Invalid assignments:", (~valid).sum())

if valid.all():
    print("\nAssignment distance:")
    print("Minimum:", round(assigned_distances.min(), 3), "km")
    print("Median :", round(np.median(assigned_distances), 3), "km")
    print("Mean   :", round(assigned_distances.mean(), 3), "km")
    print("Maximum:", round(assigned_distances.max(), 3), "km")

    print("\nUnique candidates:",
          len(set(indices[row_ind, col_ind])))

Positive locations: 374
Assignments: 374
Invalid assignments: 323


In [52]:

from scipy.optimize import linear_sum_assignment
import numpy as np

for radius in [10, 15, 20, 25]:
    
    distances, indices = candidate_tree.query(
        positive_coords,
        k=500
    )

    distances_km = distances * 6371.0

    cost = distances_km.copy()
    cost[cost > radius] = 1e6

    row_ind, col_ind = linear_sum_assignment(cost)

    assigned = cost[row_ind, col_ind]
    valid = assigned < 1e6

    print(
        f"{radius} km -> "
        f"valid assignments: {valid.sum()}/{len(positive_locations)}, "
        f"max valid distance: "
        f"{assigned[valid].max() if valid.any() else 'N/A':.2f} km"
    )

10 km -> valid assignments: 51/374, max valid distance: 9.98 km
15 km -> valid assignments: 100/374, max valid distance: 15.00 km
20 km -> valid assignments: 164/374, max valid distance: 19.99 km
25 km -> valid assignments: 249/374, max valid distance: 25.00 km


In [53]:
# Nearest safe candidate for each positive location

nearest_distances, nearest_indices = candidate_tree.query(
    positive_coords,
    k=1
)

nearest_distances_km = nearest_distances[:, 0] * 6371.0
nearest_indices = nearest_indices[:, 0]

print("Positive locations:", len(positive_locations))
print("Unique nearest controls:", len(np.unique(nearest_indices)))

# How often is each candidate reused?
usage = pd.Series(nearest_indices).value_counts()

print("\nCandidate reuse:")
print("Maximum reuse:", usage.max())
print("Median reuse :", usage.median())
print("Candidates used once:", (usage == 1).sum())
print("Candidates used 2+ times:", (usage >= 2).sum())
print("Candidates used 5+ times:", (usage >= 5).sum())

print("\nDistance:")
print("Minimum:", round(nearest_distances_km.min(), 3), "km")
print("Median :", round(np.median(nearest_distances_km), 3), "km")
print("Mean   :", round(nearest_distances_km.mean(), 3), "km")
print("Maximum:", round(nearest_distances_km.max(), 3), "km")

Positive locations: 374
Unique nearest controls: 260

Candidate reuse:
Maximum reuse: 12
Median reuse : 1.0
Candidates used once: 202
Candidates used 2+ times: 58
Candidates used 5+ times: 8

Distance:
Minimum: 1.014 km
Median : 2.024 km
Mean   : 2.151 km
Maximum: 4.572 km


In [54]:
# Build nearest-control definitions

control_definitions = positive_locations.copy()

control_definitions["control_latitude"] = candidate_pool.iloc[
    nearest_indices
]["latitude"].to_numpy()

control_definitions["control_longitude"] = candidate_pool.iloc[
    nearest_indices
]["longitude"].to_numpy()

control_definitions["control_distance_km"] = nearest_distances_km

print("Control definitions:", control_definitions.shape)
print("Unique control locations:",
      control_definitions[
          ["control_latitude", "control_longitude"]
      ].drop_duplicates().shape[0])

print("\nDistance summary:")
print(control_definitions["control_distance_km"].describe())

Control definitions: (374, 5)
Unique control locations: 260

Distance summary:
count    374.000000
mean       2.151025
std        0.770413
min        1.014452
25%        1.557518
50%        2.024019
75%        2.616676
max        4.572007
Name: control_distance_km, dtype: float64


In [55]:
from pathlib import Path

data_dir = Path("../data")

for path in sorted(data_dir.iterdir()):
    if any(word in path.name.lower() for word in ["event", "hma", "landslide"]):
        print(path.name)

himalayan_event_rainfall.csv
himalayan_landslide_v1_dataset.csv
HMA_LS_Cat_point_v02.0
HMA_LS_Cat_point_v02.0.zip


In [56]:
from pathlib import Path

event_dir = Path("../data/HMA_LS_Cat_point_v02.0")

for path in event_dir.rglob("*"):
    if path.is_file():
        print(path.name)

HMA_LS_Cat_point_v02.0.cpg
HMA_LS_Cat_point_v02.0.dbf
HMA_LS_Cat_point_v02.0.prj
HMA_LS_Cat_point_v02.0.sbn
HMA_LS_Cat_point_v02.0.sbx
HMA_LS_Cat_point_v02.0.shp
HMA_LS_Cat_point_v02.0.shp.xml
HMA_LS_Cat_point_v02.0.shx
HMA_LS_Cat_point_v02.0.xml


In [57]:
import geopandas as gpd
import pandas as pd

shp_path = (
    "../data/HMA_LS_Cat_point_v02.0/"
    "HMA_LS_Cat_point_v02.0.shp"
)

events = gpd.read_file(shp_path)

# Keep Uttarakhand and Himachal Pradesh events
study_events = events[
    events["div_name"].isin(
        ["Uttaranchal", "Himachal Pradesh"]
    )
].copy()

study_events["state"] = study_events["div_name"].replace({
    "Uttaranchal": "Uttarakhand"
})

# Keep rainfall-related events
rain_triggers = [
    "downpour",
    "rain",
    "continuous_rain",
    "monsoon"
]

rain_events = study_events[
    study_events["ls_trig"].isin(rain_triggers)
].copy()

rain_events["event_date"] = pd.to_datetime(
    rain_events["ev_date"]
).dt.normalize()

event_history = (
    rain_events[["latitude", "longitude", "event_date", "state"]]
    .dropna(subset=["latitude", "longitude", "event_date"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Rainfall-related event records:", len(event_history))
print("Unique event locations:",
      event_history[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Date range:",
      event_history["event_date"].min(),
      "to",
      event_history["event_date"].max())

Rainfall-related event records: 375
Unique event locations: 374
Date range: 2007-07-30 00:00:00 to 2018-10-09 00:00:00


In [58]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# Coordinates of recorded rainfall-related events
event_coords = np.radians(
    event_history[["latitude", "longitude"]].to_numpy()
)

event_dates = pd.to_datetime(
    event_history["event_date"]
).to_numpy()

event_tree = BallTree(event_coords, metric="haversine")

flagged = []

for i, row in control_definitions.iterrows():

    control_coord = np.radians([[
        row["control_latitude"],
        row["control_longitude"]
    ]])

    # Find events within 1 km
    nearby = event_tree.query_radius(
        control_coord,
        r=1 / 6371.0
    )[0]

    # The corresponding positive location's event date
    pos_lat = row["latitude"]
    pos_lon = row["longitude"]

    matching = event_history[
        np.isclose(event_history["latitude"], pos_lat) &
        np.isclose(event_history["longitude"], pos_lon)
    ]

    for date in pd.to_datetime(matching["event_date"]):
        reference_date = date - pd.Timedelta(days=3)
        window_end = reference_date + pd.Timedelta(days=3)

        for event_idx in nearby:
            nearby_date = pd.Timestamp(event_dates[event_idx])

            if reference_date < nearby_date <= window_end:
                flagged.append({
                    "control_index": i,
                    "control_latitude": row["control_latitude"],
                    "control_longitude": row["control_longitude"],
                    "reference_date": reference_date,
                    "nearby_event_date": nearby_date
                })

flagged_df = pd.DataFrame(flagged).drop_duplicates()

print("Controls checked:", len(control_definitions))
print("Flagged control-event matches:", len(flagged_df))
print("Unique flagged controls:",
      flagged_df["control_index"].nunique() if not flagged_df.empty else 0)

Controls checked: 374
Flagged control-event matches: 0
Unique flagged controls: 0


In [59]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# Match each event location to its selected control
controls = control_definitions.copy()

controls["_lat_key"] = controls["latitude"].round(6)
controls["_lon_key"] = controls["longitude"].round(6)

events_for_controls = event_history.copy()
events_for_controls["_lat_key"] = (
    events_for_controls["latitude"].round(6)
)
events_for_controls["_lon_key"] = (
    events_for_controls["longitude"].round(6)
)

matched = events_for_controls.merge(
    controls[
        [
            "_lat_key", "_lon_key",
            "control_latitude", "control_longitude",
            "control_distance_km"
        ]
    ],
    on=["_lat_key", "_lon_key"],
    how="left",
    validate="many_to_one"
)

assert matched["control_latitude"].notna().all(), (
    "Some event locations have no matched control."
)

# Create one control sample for each event and horizon
samples = []

for _, row in matched.iterrows():
    for horizon in [1, 2, 3]:
        samples.append({
            "latitude": row["control_latitude"],
            "longitude": row["control_longitude"],
            "reference_date": (
                row["event_date"] - pd.Timedelta(days=horizon)
            ),
            "event_date": row["event_date"],
            "positive_horizon": horizon,
            "control_distance_km": row["control_distance_km"]
        })

negative_definitions = pd.DataFrame(samples)

# Find catalog events within 1 km of each control
event_tree = BallTree(
    np.radians(
        event_history[["latitude", "longitude"]].to_numpy()
    ),
    metric="haversine"
)

event_dates = pd.to_datetime(
    event_history["event_date"]
).to_numpy()

flagged_rows = []

for idx, row in negative_definitions.iterrows():
    coord = np.radians([[
        row["latitude"], row["longitude"]
    ]])

    nearby = event_tree.query_radius(
        coord,
        r=1 / 6371.0
    )[0]

    ref = pd.Timestamp(row["reference_date"])
    end = ref + pd.Timedelta(days=3)

    for event_idx in nearby:
        recorded_date = pd.Timestamp(event_dates[event_idx])

        if ref < recorded_date <= end:
            flagged_rows.append({
                "sample_index": idx,
                "reference_date": ref,
                "recorded_event_date": recorded_date
            })
            break

flagged_df = pd.DataFrame(flagged_rows)

print("Positive event records:", len(event_history))
print("Control samples created:", len(negative_definitions))
print("Missing control coordinates:",
      negative_definitions["latitude"].isna().sum())
print("Flagged control samples:", len(flagged_df))
print("Unique control locations:",
      negative_definitions[
          ["latitude", "longitude"]
      ].drop_duplicates().shape[0])

Positive event records: 375
Control samples created: 1125
Missing control coordinates: 0
Flagged control samples: 0
Unique control locations: 260


In [60]:
# Check control reuse and horizon coverage

print("Total control samples:", len(negative_definitions))

print(
    "Unique location-date pairs:",
    negative_definitions[
        ["latitude", "longitude", "reference_date"]
    ].drop_duplicates().shape[0]
)

print(
    "Unique reference dates:",
    negative_definitions["reference_date"].nunique()
)

print("\nSamples by horizon:")
print(negative_definitions["positive_horizon"].value_counts().sort_index())

print("\nControl-location reuse:")
usage = negative_definitions.groupby(
    ["latitude", "longitude"]
).size()

print("Unique locations:", len(usage))
print("Maximum reuse:", usage.max())
print("Locations used once:", (usage == 1).sum())
print("Locations used 2+ times:", (usage >= 2).sum())

print("\nDuplicate location-date rows:",
      negative_definitions.duplicated(
          ["latitude", "longitude", "reference_date"]
      ).sum())

Total control samples: 1125
Unique location-date pairs: 1110
Unique reference dates: 477

Samples by horizon:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64

Control-location reuse:
Unique locations: 260
Maximum reuse: 36
Locations used once: 0
Locations used 2+ times: 260

Duplicate location-date rows: 15


In [61]:
duplicate_controls = negative_definitions[
    negative_definitions.duplicated(
        ["latitude", "longitude", "reference_date"],
        keep=False
    )
].sort_values(
    ["latitude", "longitude", "reference_date"]
)

print("Rows involved in duplicates:", len(duplicate_controls))

display(
    duplicate_controls[
        [
            "latitude",
            "longitude",
            "reference_date",
            "event_date",
            "positive_horizon"
        ]
    ]
)

Rows involved in duplicates: 30


,latitude,longitude,reference_date,event_date,positive_horizon
176,30.261491,79.237310,2013-07-21,2013-07-24,3
521,30.261491,79.237310,2013-07-21,2013-07-24,3
175,30.261491,79.237310,2013-07-22,2013-07-24,2
520,30.261491,79.237310,2013-07-22,2013-07-24,2
174,30.261491,79.237310,2013-07-23,2013-07-24,1
519,30.261491,79.237310,2013-07-23,2013-07-24,1
269,31.049537,77.399516,2011-08-04,2011-08-07,3
317,31.049537,77.399516,2011-08-04,2011-08-07,3
268,31.049537,77.399516,2011-08-05,2011-08-07,2
316,31.049537,77.399516,2011-08-05,2011-08-07,2


In [62]:
# Check duplicate event locations and dates in the source catalog

event_keys = ["latitude", "longitude", "event_date"]

print("Total event records:", len(event_history))
print(
    "Unique location-date combinations:",
    event_history[event_keys].drop_duplicates().shape[0]
)

print(
    "Duplicate location-date rows:",
    event_history.duplicated(event_keys, keep=False).sum()
)

print("\nRepeated event dates and locations:")
display(
    event_history[
        event_history.duplicated(event_keys, keep=False)
    ].sort_values(event_keys)
)

Total event records: 375
Unique location-date combinations: 375
Duplicate location-date rows: 0

Repeated event dates and locations:


,latitude,longitude,event_date,state


In [63]:
print("Positive locations:", len(positive_locations))
print(
    "Unique positive coordinates:",
    positive_locations[
        ["latitude", "longitude"]
    ].drop_duplicates().shape[0]
)

print("\nControl definitions:")
print("Rows:", len(control_definitions))
print(
    "Unique positive coordinates in controls:",
    control_definitions[
        ["latitude", "longitude"]
    ].drop_duplicates().shape[0]
)

print("\nRepeated positive coordinates in control definitions:")
display(
    control_definitions[
        control_definitions.duplicated(
            ["latitude", "longitude"], keep=False
        )
    ].sort_values(["latitude", "longitude"])
)

Positive locations: 374
Unique positive coordinates: 374

Control definitions:
Rows: 374
Unique positive coordinates in controls: 374

Repeated positive coordinates in control definitions:


,latitude,longitude,control_latitude,control_longitude,control_distance_km


In [64]:
# Check whether multiple positive events share the same control
# location and reference date.

mapping_check = negative_definitions.merge(
    control_definitions[
        [
            "latitude",
            "longitude",
            "control_latitude",
            "control_longitude"
        ]
    ],
    on=["latitude", "longitude"],
    how="left",
    suffixes=("", "_mapped")
)

print("Negative samples:", len(negative_definitions))
print("Mapping rows:", len(mapping_check))

print("\nRepeated control location-date pairs:")
display(
    negative_definitions[
        negative_definitions.duplicated(
            ["latitude", "longitude", "reference_date"],
            keep=False
        )
    ].sort_values(
        ["latitude", "longitude", "reference_date"]
    )
)

Negative samples: 1125
Mapping rows: 1125

Repeated control location-date pairs:


,latitude,longitude,reference_date,event_date,positive_horizon,control_distance_km
176,30.261491,79.237310,2013-07-21,2013-07-24,3,1.749857
521,30.261491,79.237310,2013-07-21,2013-07-24,3,1.376828
175,30.261491,79.237310,2013-07-22,2013-07-24,2,1.749857
520,30.261491,79.237310,2013-07-22,2013-07-24,2,1.376828
174,30.261491,79.237310,2013-07-23,2013-07-24,1,1.749857
519,30.261491,79.237310,2013-07-23,2013-07-24,1,1.376828
269,31.049537,77.399516,2011-08-04,2011-08-07,3,3.109394
317,31.049537,77.399516,2011-08-04,2011-08-07,3,3.102326
268,31.049537,77.399516,2011-08-05,2011-08-07,2,3.109394
316,31.049537,77.399516,2011-08-05,2011-08-07,2,3.102326


In [65]:
# Inspect control assignments for one duplicated event

event_lat = 30.261491
event_lon = 79.237310

display(
    control_definitions[
        np.isclose(control_definitions["latitude"], event_lat) &
        np.isclose(control_definitions["longitude"], event_lon)
    ]
)

,latitude,longitude,control_latitude,control_longitude,control_distance_km


In [66]:
print("Columns:", control_definitions.columns.tolist())
display(control_definitions.head())

Columns: ['latitude', 'longitude', 'control_latitude', 'control_longitude', 'control_distance_km']


,latitude,longitude,control_latitude,control_longitude,control_distance_km
0,28.866500,79.568200,28.879174,79.586699,2.287052
1,29.158500,79.146400,29.160073,79.157099,1.053550
2,29.295586,79.540431,29.287731,79.564620,2.503172
3,29.330000,80.100000,29.340694,80.088381,1.637800
4,29.337636,79.483545,29.335886,79.466921,1.623110


In [67]:
# Check whether the control mapping is unique for each event

print("Control definitions:", len(control_definitions))
print(
    "Unique positive coordinates:",
    control_definitions[
        ["latitude", "longitude"]
    ].drop_duplicates().shape[0]
)

print("\nEvent catalog:")
print("Rows:", len(event_history))
print(
    "Unique event coordinates and dates:",
    event_history[
        ["latitude", "longitude", "event_date"]
    ].drop_duplicates().shape[0]
)

# Check exact event-to-control join cardinality
event_keys = event_history[
    ["latitude", "longitude", "event_date"]
].copy()

control_keys = control_definitions[
    ["latitude", "longitude"]
].copy()

event_keys["lat_key"] = event_keys["latitude"].round(6)
event_keys["lon_key"] = event_keys["longitude"].round(6)

control_keys["lat_key"] = control_keys["latitude"].round(6)
control_keys["lon_key"] = control_keys["longitude"].round(6)

join_counts = event_keys.merge(
    control_keys[["lat_key", "lon_key"]],
    on=["lat_key", "lon_key"],
    how="left"
)

print("\nRows after join:", len(join_counts))
print("Rows before join:", len(event_keys))

Control definitions: 374
Unique positive coordinates: 374

Event catalog:
Rows: 375
Unique event coordinates and dates: 375

Rows after join: 375
Rows before join: 375


In [68]:
# Find the location with multiple recorded events

event_counts = (
    event_history.groupby(["latitude", "longitude"])
    .size()
    .reset_index(name="event_count")
)

repeated_locations = event_counts[
    event_counts["event_count"] > 1
]

print("Locations with multiple events:", len(repeated_locations))
display(repeated_locations)

if not repeated_locations.empty:
    display(
        event_history.merge(
            repeated_locations[["latitude", "longitude"]],
            on=["latitude", "longitude"],
            how="inner"
        ).sort_values("event_date")
    )

Locations with multiple events: 1


,latitude,longitude,event_count
145,30.6738,77.3544,2


,latitude,longitude,event_date,state
1,30.6738,77.3544,2013-06-16,Himachal Pradesh
0,30.6738,77.3544,2013-07-14,Himachal Pradesh


In [69]:
# Inspect the two events at the repeated location
# and their assigned control coordinates.

repeated = event_history[
    np.isclose(event_history["latitude"], 30.6738) &
    np.isclose(event_history["longitude"], 77.3544)
].copy()

display(repeated)

display(
    control_definitions[
        np.isclose(control_definitions["latitude"], 30.6738) &
        np.isclose(control_definitions["longitude"], 77.3544)
    ]
)

,latitude,longitude,event_date,state
51,30.6738,77.3544,2013-07-14,Himachal Pradesh
198,30.6738,77.3544,2013-06-16,Himachal Pradesh


,latitude,longitude,control_latitude,control_longitude,control_distance_km
145,30.6738,77.3544,30.677672,77.338888,1.544752


In [70]:
# Compare the duplicate rows directly

duplicate_rows = negative_definitions[
    negative_definitions.duplicated(
        ["latitude", "longitude", "reference_date"],
        keep=False
    )
].copy()

print("Duplicate location-date groups:",
      duplicate_rows.groupby(
          ["latitude", "longitude", "reference_date"]
      ).ngroups)

print("\nUnique event dates in duplicate rows:",
      duplicate_rows["event_date"].nunique())

print("\nUnique positive horizons in duplicate rows:",
      sorted(duplicate_rows["positive_horizon"].unique()))

display(
    duplicate_rows[
        [
            "latitude", "longitude", "reference_date",
            "event_date", "positive_horizon",
            "control_distance_km"
        ]
    ].sort_values(
        ["latitude", "longitude", "reference_date"]
    )
)

Duplicate location-date groups: 15

Unique event dates in duplicate rows: 5

Unique positive horizons in duplicate rows: [1, 2, 3]


,latitude,longitude,reference_date,event_date,positive_horizon,control_distance_km
176,30.261491,79.237310,2013-07-21,2013-07-24,3,1.749857
521,30.261491,79.237310,2013-07-21,2013-07-24,3,1.376828
175,30.261491,79.237310,2013-07-22,2013-07-24,2,1.749857
520,30.261491,79.237310,2013-07-22,2013-07-24,2,1.376828
174,30.261491,79.237310,2013-07-23,2013-07-24,1,1.749857
519,30.261491,79.237310,2013-07-23,2013-07-24,1,1.376828
269,31.049537,77.399516,2011-08-04,2011-08-07,3,3.109394
317,31.049537,77.399516,2011-08-04,2011-08-07,3,3.102326
268,31.049537,77.399516,2011-08-05,2011-08-07,2,3.109394
316,31.049537,77.399516,2011-08-05,2011-08-07,2,3.102326


In [71]:
# Rebuild the control mapping from the original nearest-candidate result

control_definitions = positive_locations.copy().reset_index(drop=True)

control_definitions["control_latitude"] = (
    candidate_pool.iloc[nearest_indices]["latitude"].to_numpy()
)

control_definitions["control_longitude"] = (
    candidate_pool.iloc[nearest_indices]["longitude"].to_numpy()
)

control_definitions["control_distance_km"] = nearest_distances_km

print("Control definitions:", len(control_definitions))
print(
    "Unique positive coordinates:",
    control_definitions[["latitude", "longitude"]]
    .drop_duplicates().shape[0]
)

Control definitions: 374
Unique positive coordinates: 374


In [72]:
# Identify events without exactly one matching control

event_check = event_history.copy()

control_counts = []

for _, event in event_check.iterrows():
    matches = control_definitions[
        np.isclose(control_definitions["latitude"], event["latitude"]) &
        np.isclose(control_definitions["longitude"], event["longitude"])
    ]

    control_counts.append(len(matches))

event_check["control_matches"] = control_counts

print(event_check["control_matches"].value_counts().sort_index())

print("\nEvents without exactly one control:")
display(
    event_check[event_check["control_matches"] != 1]
)

control_matches
1    366
2      6
3      3
Name: count, dtype: int64

Events without exactly one control:


,latitude,longitude,event_date,state,control_matches
43,30.2851,78.9829,2010-09-03,Uttarakhand,2
47,30.7333,78.4403,2010-09-12,Uttarakhand,3
56,32.3710,77.2468,2010-08-06,Himachal Pradesh,2
89,31.0766,77.3913,2011-08-07,Himachal Pradesh,2
105,31.0766,77.3916,2011-08-07,Himachal Pradesh,2
115,30.7332,78.4401,2010-09-20,Uttarakhand,3
127,30.7333,78.4399,2011-07-10,Uttarakhand,3
223,30.2853,78.9829,2012-09-14,Uttarakhand,2
324,32.3712,77.2466,2008-06-18,Himachal Pradesh,2


In [73]:
# Check nearest positive location for every event record

from sklearn.neighbors import BallTree
import numpy as np

positive_coords_rad = np.radians(
    positive_locations[["latitude", "longitude"]].to_numpy()
)

event_coords_rad = np.radians(
    event_history[["latitude", "longitude"]].to_numpy()
)

positive_tree = BallTree(
    positive_coords_rad,
    metric="haversine"
)

distances, indices = positive_tree.query(
    event_coords_rad,
    k=1
)

print("Event records:", len(event_history))
print("Unique positive locations:", len(positive_locations))
print(
    "Maximum event-to-positive-location distance:",
    round((distances.max() * 6371), 4),
    "km"
)

Event records: 375
Unique positive locations: 374
Maximum event-to-positive-location distance: 0.0 km


In [74]:
# Rebuild negative samples using verified nearest-location indices

negative_samples = []

for event_idx, event in event_history.reset_index(drop=True).iterrows():

    positive_idx = int(indices[event_idx, 0])
    control = control_definitions.iloc[positive_idx]

    for horizon in [1, 2, 3]:
        negative_samples.append({
            "latitude": control["control_latitude"],
            "longitude": control["control_longitude"],
            "reference_date": (
                pd.Timestamp(event["event_date"])
                - pd.Timedelta(days=horizon)
            ),
            "event_date": pd.Timestamp(event["event_date"]),
            "positive_horizon": horizon,
            "control_distance_km": control["control_distance_km"]
        })

negative_definitions = pd.DataFrame(negative_samples)

print("Negative samples:", len(negative_definitions))
print(
    "Duplicate location-date pairs:",
    negative_definitions.duplicated(
        ["latitude", "longitude", "reference_date"]
    ).sum()
)
print(
    "Unique reference dates:",
    negative_definitions["reference_date"].nunique()
)

Negative samples: 1125
Duplicate location-date pairs: 15
Unique reference dates: 477


In [75]:
duplicate_pairs = negative_definitions[
    negative_definitions.duplicated(
        ["latitude", "longitude", "reference_date"],
        keep=False
    )
].sort_values(
    ["latitude", "longitude", "reference_date"]
)

print("Rows involved:", len(duplicate_pairs))
print(
    "Duplicate location-date groups:",
    duplicate_pairs.groupby(
        ["latitude", "longitude", "reference_date"]
    ).ngroups
)

display(
    duplicate_pairs[
        [
            "latitude",
            "longitude",
            "reference_date",
            "event_date",
            "positive_horizon",
            "control_distance_km"
        ]
    ]
)

Rows involved: 30
Duplicate location-date groups: 15


,latitude,longitude,reference_date,event_date,positive_horizon,control_distance_km
176,30.261491,79.237310,2013-07-21,2013-07-24,3,1.749857
521,30.261491,79.237310,2013-07-21,2013-07-24,3,1.376828
175,30.261491,79.237310,2013-07-22,2013-07-24,2,1.749857
520,30.261491,79.237310,2013-07-22,2013-07-24,2,1.376828
174,30.261491,79.237310,2013-07-23,2013-07-24,1,1.749857
519,30.261491,79.237310,2013-07-23,2013-07-24,1,1.376828
269,31.049537,77.399516,2011-08-04,2011-08-07,3,3.109394
317,31.049537,77.399516,2011-08-04,2011-08-07,3,3.102326
268,31.049537,77.399516,2011-08-05,2011-08-07,2,3.109394
316,31.049537,77.399516,2011-08-05,2011-08-07,2,3.102326


In [76]:
# Check whether different positive locations were assigned
# the same nearest-candidate index.

assignment_check = pd.DataFrame({
    "positive_index": np.arange(len(positive_locations)),
    "candidate_index": nearest_indices,
    "distance_km": nearest_distances_km
})

print("Positive locations:", len(assignment_check))
print("Unique candidate indices:",
      assignment_check["candidate_index"].nunique())

print("\nCandidates assigned more than once:")
display(
    assignment_check[
        assignment_check.duplicated(
            "candidate_index", keep=False
        )
    ].sort_values("candidate_index")
)

Positive locations: 374
Unique candidate indices: 260

Candidates assigned more than once:


,positive_index,candidate_index,distance_km
347,347,10890,4.029366
346,346,10890,3.982300
138,138,11250,3.655494
146,146,11250,3.218866
344,344,11270,4.276688
...,...,...,...
327,327,11896,1.038170
140,140,11897,1.025572
137,137,11897,1.380677
136,136,11897,1.847502


In [77]:
# Inspect the source positive events behind one duplicate control pair

target_date = pd.Timestamp("2013-07-21")
target_lat = 30.261491
target_lon = 79.237310

# Find the control coordinates used in the duplicate rows
dup = negative_definitions[
    (negative_definitions["reference_date"] == target_date) &
    np.isclose(negative_definitions["latitude"], target_lat) &
    np.isclose(negative_definitions["longitude"], target_lon)
]

print("Matching negative rows:", len(dup))
display(dup)

# Find positive events whose controls use these coordinates
for _, row in dup.iterrows():
    print(
        "Control coordinates:",
        row["latitude"], row["longitude"],
        "| distance:", row["control_distance_km"]
    )

Matching negative rows: 2


,latitude,longitude,reference_date,event_date,positive_horizon,control_distance_km
176,30.261491,79.23731,2013-07-21,2013-07-24,3,1.749857
521,30.261491,79.23731,2013-07-21,2013-07-24,3,1.376828


Control coordinates: 30.26149070014138 79.23731004062324 | distance: 1.7498570199782237
Control coordinates: 30.26149070014138 79.23731004062324 | distance: 1.3768276845851273


In [78]:
# Rebuild one authoritative mapping from positive locations to controls

control_definitions = positive_locations.copy().reset_index(drop=True)

control_definitions["candidate_index"] = nearest_indices

control_definitions["control_latitude"] = (
    candidate_pool.iloc[nearest_indices]["latitude"].to_numpy()
)

control_definitions["control_longitude"] = (
    candidate_pool.iloc[nearest_indices]["longitude"].to_numpy()
)

control_definitions["control_distance_km"] = (
    nearest_distances_km
)

# Verify each candidate's distance against the coordinates
from sklearn.metrics.pairwise import haversine_distances

pos_rad = np.radians(
    control_definitions[["latitude", "longitude"]].to_numpy()
)
ctrl_rad = np.radians(
    control_definitions[
        ["control_latitude", "control_longitude"]
    ].to_numpy()
)

control_definitions["verified_distance_km"] = (
    np.array([
        haversine_distances(pos_rad[i:i+1], ctrl_rad[i:i+1])[0, 0]
        for i in range(len(control_definitions))
    ]) * 6371.0
)

print("Rows:", len(control_definitions))
print(
    "Maximum distance discrepancy (km):",
    round(
        np.abs(
            control_definitions["control_distance_km"] -
            control_definitions["verified_distance_km"]
        ).max(),
        8
    )
)

Rows: 374
Maximum distance discrepancy (km): 0.0


In [79]:
negative_samples = []

for event_idx, event in event_history.reset_index(drop=True).iterrows():
    positive_idx = int(indices[event_idx, 0])
    control = control_definitions.iloc[positive_idx]

    for horizon in [1, 2, 3]:
        negative_samples.append({
            "latitude": control["control_latitude"],
            "longitude": control["control_longitude"],
            "reference_date": (
                pd.Timestamp(event["event_date"])
                - pd.Timedelta(days=horizon)
            ),
            "event_date": pd.Timestamp(event["event_date"]),
            "positive_horizon": horizon,
            "control_distance_km": control["verified_distance_km"]
        })

negative_definitions = pd.DataFrame(negative_samples)

print("Negative samples:", len(negative_definitions))
print(
    "Duplicate location-date pairs:",
    negative_definitions.duplicated(
        ["latitude", "longitude", "reference_date"]
    ).sum()
)

Negative samples: 1125
Duplicate location-date pairs: 15


In [80]:
duplicate_pairs = negative_definitions[
    negative_definitions.duplicated(
        ["latitude", "longitude", "reference_date"],
        keep=False
    )
].copy()

summary = (
    duplicate_pairs.groupby(
        ["latitude", "longitude", "reference_date"]
    )
    .agg(
        rows=("event_date", "size"),
        event_dates=("event_date", "nunique"),
        horizons=("positive_horizon", "nunique")
    )
    .reset_index()
)

print("Duplicate location-date groups:", len(summary))
display(summary)

Duplicate location-date groups: 15


,latitude,longitude,reference_date,rows,event_dates,horizons
0,30.261491,79.237310,2013-07-21,2,1,1
1,30.261491,79.237310,2013-07-22,2,1,1
2,30.261491,79.237310,2013-07-23,2,1,1
3,31.049537,77.399516,2011-08-04,2,1,1
4,31.049537,77.399516,2011-08-05,2,1,1
5,31.049537,77.399516,2011-08-06,2,1,1
6,31.081884,77.184508,2010-07-22,2,1,1
7,31.081884,77.184508,2010-07-23,2,1,1
8,31.081884,77.184508,2010-07-24,2,1,1
9,31.687057,76.941333,2013-07-04,2,1,1


In [81]:
negative_definitions_clean = negative_definitions.drop_duplicates(
    subset=["latitude", "longitude", "reference_date"],
    keep="first"
).reset_index(drop=True)

print("Original samples:", len(negative_definitions))
print("Clean samples:", len(negative_definitions_clean))
print(
    "Remaining duplicate location-date pairs:",
    negative_definitions_clean.duplicated(
        ["latitude", "longitude", "reference_date"]
    ).sum()
)

Original samples: 1125
Clean samples: 1110
Remaining duplicate location-date pairs: 0


In [82]:
print([
    name for name in globals()
    if "positive" in name.lower() or "horizon" in name.lower()
])

['positive_coords', 'positive_rad', 'nearest_positive_km', 'positive_set', 'positive_data', 'positive_locations', 'horizon', 'positive_coords_rad', 'positive_tree', 'positive_idx']


In [83]:
# Compare positive and negative samples by location and date

positive_keys = positive_data[
    ["latitude", "longitude", "reference_date"]
].copy()

negative_keys = negative_definitions_clean[
    ["latitude", "longitude", "reference_date"]
].copy()

positive_keys["label"] = 1
negative_keys["label"] = 0

combined_keys = pd.concat(
    [positive_keys, negative_keys],
    ignore_index=True
)

combined_keys["latitude"] = combined_keys["latitude"].round(6)
combined_keys["longitude"] = combined_keys["longitude"].round(6)

conflicts = combined_keys[
    combined_keys.duplicated(
        ["latitude", "longitude", "reference_date"],
        keep=False
    )
]

conflicting_groups = conflicts.groupby(
    ["latitude", "longitude", "reference_date"]
).filter(lambda group: group["label"].nunique() > 1)

print(
    "Location-date groups with conflicting labels:",
    conflicting_groups.groupby(
        ["latitude", "longitude", "reference_date"]
    ).ngroups
)

display(conflicting_groups.head(20))

Location-date groups with conflicting labels: 0


,latitude,longitude,reference_date,label


In [84]:
print("Positive samples:", len(positive_data))
print("Negative samples:", len(negative_definitions_clean))

print("\nNegative samples by horizon:")
print(
    negative_definitions_clean["positive_horizon"]
    .value_counts()
    .sort_index()
)

print("\nNegative reference-date range:")
print(
    negative_definitions_clean["reference_date"].min(),
    "to",
    negative_definitions_clean["reference_date"].max()
)

Positive samples: 1125
Negative samples: 1110

Negative samples by horizon:
positive_horizon
1    370
2    370
3    370
Name: count, dtype: int64

Negative reference-date range:
2007-07-27 00:00:00 to 2018-10-08 00:00:00


In [85]:
from scipy.optimize import linear_sum_assignment
import numpy as np
import pandas as pd

# Find up to 500 nearest safe candidates per positive location
distances, candidate_indices = candidate_tree.query(
    positive_coords,
    k=500
)

distances_km = distances * 6371.0

# Disallow candidates farther than 10 km
cost = distances_km.copy()
cost[cost > 10] = 1e6

# Globally assign unique controls
row_ind, col_ind = linear_sum_assignment(cost)

assigned_distances = cost[row_ind, col_ind]
valid = assigned_distances < 1e6

print("Positive locations:", len(positive_locations))
print("Unique assignments:", valid.sum())
print("Assignments outside 10 km:", (~valid).sum())

Positive locations: 374
Unique assignments: 51
Assignments outside 10 km: 323


In [86]:
# Find alternative safe candidates within 10 km
# for the five affected positive locations.

duplicate_groups = negative_definitions[
    negative_definitions.duplicated(
        ["latitude", "longitude", "reference_date"],
        keep=False
    )
]

affected_events = (
    duplicate_groups[
        ["event_date"]
    ]
    .drop_duplicates()
    .sort_values("event_date")
    .reset_index(drop=True)
)

print("Affected event dates:", len(affected_events))
display(affected_events)

Affected event dates: 5


,event_date
0,2010-07-25
1,2011-08-07
2,2013-07-07
3,2013-07-24
4,2017-07-02


In [87]:
affected_dates = pd.to_datetime(
    affected_events["event_date"]
)

affected_positives = event_history[
    pd.to_datetime(event_history["event_date"]).isin(affected_dates)
].copy()

print("Affected event records:", len(affected_positives))

display(
    affected_positives[
        ["latitude", "longitude", "event_date", "state"]
    ].sort_values("event_date")
)

Affected event records: 26


,latitude,longitude,event_date,state
291,30.558900,77.288900,2010-07-25,Himachal Pradesh
46,30.904000,77.096900,2010-07-25,Himachal Pradesh
260,31.093100,77.167500,2010-07-25,Himachal Pradesh
74,32.206700,76.364100,2010-07-25,Himachal Pradesh
141,31.105400,77.172700,2010-07-25,Himachal Pradesh
299,31.714568,77.022394,2011-08-07,Himachal Pradesh
89,31.076600,77.391300,2011-08-07,Himachal Pradesh
227,31.708300,76.931600,2011-08-07,Himachal Pradesh
105,31.076600,77.391600,2011-08-07,Himachal Pradesh
109,31.240300,77.925800,2011-08-07,Himachal Pradesh


In [88]:
# Count safe candidates within 10 km for affected events

affected_coords = np.radians(
    affected_positives[["latitude", "longitude"]].to_numpy()
)

candidate_coords_rad = np.radians(
    candidate_pool[["latitude", "longitude"]].to_numpy()
)

candidate_tree = BallTree(
    candidate_coords_rad,
    metric="haversine"
)

nearby_indices = candidate_tree.query_radius(
    affected_coords,
    r=10 / 6371.0
)

counts = np.array([len(x) for x in nearby_indices])

print("Affected event records:", len(affected_positives))
print("Minimum candidates:", counts.min())
print("Median candidates:", int(np.median(counts)))
print("Events with 10+ candidates:", (counts >= 10).sum())

Affected event records: 26
Minimum candidates: 20
Median candidates: 30
Events with 10+ candidates: 26


In [89]:
# Find alternative safe locations for the affected events

candidate_coords_rad = np.radians(
    candidate_pool[["latitude", "longitude"]].to_numpy()
)

candidate_tree = BallTree(candidate_coords_rad, metric="haversine")

affected_coords_rad = np.radians(
    affected_positives[["latitude", "longitude"]].to_numpy()
)

# Find all safe candidates within 10 km
nearby_indices = candidate_tree.query_radius(
    affected_coords_rad,
    r=10 / 6371.0
)

# Show the first few alternatives for each affected event
for i, (_, event) in enumerate(affected_positives.iterrows()):
    options = candidate_pool.iloc[nearby_indices[i]].copy()

    distances, _ = candidate_tree.query(
        affected_coords_rad[i:i+1],
        k=len(options)
    )

    options["distance_km"] = distances[0] * 6371.0
    options = options.sort_values("distance_km")

    print(
        f"Event {i+1}: {event['event_date'].date()} | "
        f"{len(options)} candidates"
    )
    print(options.head(3)[
        ["latitude", "longitude", "distance_km"]
    ].to_string(index=False))

Event 1: 2017-07-02 | 29 candidates
 latitude  longitude  distance_km
32.284985  78.010007     4.276688
32.181764  78.062148     4.382411
32.248536  78.139195     4.421926
Event 2: 2017-07-02 | 32 candidates
 latitude  longitude  distance_km
32.284985  78.010007     4.029366
32.181764  78.062148     4.298999
32.248536  78.139195     5.191767
Event 3: 2017-07-02 | 30 candidates
 latitude  longitude  distance_km
32.284985  78.010007     4.353497
32.181764  78.062148     4.640792
32.248536  78.139195     5.090488
Event 4: 2013-07-07 | 20 candidates
 latitude  longitude  distance_km
31.971905  76.747112     1.838033
31.897726  76.868924     2.839669
31.882229  76.868922     4.599214
Event 5: 2010-07-25 | 40 candidates
 latitude  longitude  distance_km
30.871120  77.017190     1.935092
30.928679  77.092368     2.174741
30.954959  77.092008     2.778015
Event 6: 2013-07-24 | 32 candidates
 latitude  longitude  distance_km
30.221767  79.295674     1.749857
30.207699  79.206316     2.914923
30

In [90]:
# Rebuild negative assignments and resolve location-date collisions

event_rows = event_history.reset_index(drop=True).copy()

# Map each event to its positive-location index
event_positive_indices = indices[:, 0]

# Start with the nearest safe candidate for each event
event_candidate_indices = np.array([
    nearest_indices[idx] for idx in event_positive_indices
])

# Track which control locations are already used on each reference date
used = set()

# Sort events by date for deterministic assignment
event_order = sorted(
    range(len(event_rows)),
    key=lambda i: (event_rows.iloc[i]["event_date"], i)
)

for i in event_order:
    event = event_rows.iloc[i]
    positive_idx = event_positive_indices[i]
    current_candidate = event_candidate_indices[i]

    horizons = [1, 2, 3]
    ref_dates = [
        (pd.Timestamp(event["event_date"]) - pd.Timedelta(days=h)).date()
        for h in horizons
    ]

    def conflicts(candidate_idx):
        point = candidate_pool.iloc[candidate_idx]
        coord = (round(point["latitude"], 6), round(point["longitude"], 6))
        return any((coord, date) in used for date in ref_dates)

    if conflicts(current_candidate):
        pos_coord = np.radians([[
            positive_locations.iloc[positive_idx]["latitude"],
            positive_locations.iloc[positive_idx]["longitude"]
        ]])

        options = candidate_tree.query_radius(
            pos_coord, r=10 / 6371.0
        )[0]

        distances = np.array([
            np.linalg.norm(
                np.radians(candidate_pool.iloc[j][["latitude", "longitude"]].to_numpy())
                - pos_coord[0]
            ) * 6371.0
            for j in options
        ])
        options = options[np.argsort(distances)]

        alternatives = [j for j in options if not conflicts(j)]

        if not alternatives:
            raise ValueError(
                f"No collision-free candidate found for event {i}, "
                f"date {event['event_date']}"
            )

        event_candidate_indices[i] = alternatives[0]

    chosen = candidate_pool.iloc[event_candidate_indices[i]]
    coord = (
        round(chosen["latitude"], 6),
        round(chosen["longitude"], 6)
    )

    for date in ref_dates:
        used.add((coord, date))

# Build the revised negative samples
negative_samples = []

for i, event in event_rows.iterrows():
    candidate = candidate_pool.iloc[event_candidate_indices[i]]

    for horizon in [1, 2, 3]:
        negative_samples.append({
            "latitude": candidate["latitude"],
            "longitude": candidate["longitude"],
            "reference_date": (
                pd.Timestamp(event["event_date"])
                - pd.Timedelta(days=horizon)
            ),
            "event_date": event["event_date"],
            "positive_horizon": horizon
        })

negative_definitions = pd.DataFrame(negative_samples)

duplicate_count = negative_definitions.duplicated(
    ["latitude", "longitude", "reference_date"]
).sum()

print("Negative samples:", len(negative_definitions))
print("Unique coordinates:", len(
    negative_definitions[["latitude", "longitude"]].drop_duplicates()
))
print("Duplicate location-date rows:", duplicate_count)

Negative samples: 1125
Unique coordinates: 264
Duplicate location-date rows: 0


In [91]:
# Audit revised negative samples against recorded event locations

negative_coords = np.radians(
    negative_definitions[["latitude", "longitude"]].to_numpy()
)

event_coords = np.radians(
    event_history[["latitude", "longitude"]].to_numpy()
)

event_tree = BallTree(event_coords, metric="haversine")

nearby_events = event_tree.query_radius(
    negative_coords,
    r=1 / 6371.0
)

flagged = []

for i, event_indices in enumerate(nearby_events):
    sample = negative_definitions.iloc[i]
    start_date = pd.Timestamp(sample["reference_date"])
    end_date = start_date + pd.Timedelta(days=3)

    for j in event_indices:
        event_date = pd.Timestamp(
            event_history.iloc[j]["event_date"]
        )

        if start_date < event_date <= end_date:
            flagged.append(i)
            break

print("Negative samples audited:", len(negative_definitions))
print("Samples flagged:", len(set(flagged)))

Negative samples audited: 1125
Samples flagged: 0


In [92]:
negative_definitions.to_csv(
    "../data/negative_location_definitions_v4.csv",
    index=False
)

print("Saved revised negative definitions.")
print("Rows:", len(negative_definitions))
print("Missing values:", negative_definitions.isnull().sum().sum())

Saved revised negative definitions.
Rows: 1125
Missing values: 0


In [93]:
import pandas as pd

positive = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)

negative = pd.read_csv(
    "../data/negative_features_v4.csv"
)

# Add negative labels
negative["LANDSLIDE"] = 0
negative["NEXT_1D"] = 0
negative["NEXT_2D"] = 0
negative["NEXT_3D"] = 0

# Remove positive-only metadata columns from positive samples
positive = positive.drop(
    columns=["event_date", "positive_horizon"],
    errors="ignore"
)

# Keep only columns shared by both datasets, in the same order
feature_columns = [
    "latitude", "longitude", "reference_date",
    "RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]

target_columns = ["LANDSLIDE", "NEXT_1D", "NEXT_2D", "NEXT_3D"]

positive = positive[feature_columns + target_columns]
negative = negative[feature_columns + target_columns]

final_df = pd.concat([positive, negative], ignore_index=True)

final_df.to_csv(
    "../data/himalayan_temporal_final_v4.csv",
    index=False
)

print("Shape:", final_df.shape)
print("\nLANDSLIDE counts:")
print(final_df["LANDSLIDE"].value_counts())
print("\nMissing values:", final_df.isnull().sum().sum())
print(
    "Duplicate location-date pairs:",
    final_df.duplicated(
        ["latitude", "longitude", "reference_date"]
    ).sum()
)
print("\nTarget totals:")
print(final_df[target_columns[1:]].sum())

Shape: (2250, 15)

LANDSLIDE counts:
LANDSLIDE
1    1125
0    1125
Name: count, dtype: int64

Missing values: 0
Duplicate location-date pairs: 0

Target totals:
NEXT_1D     375
NEXT_2D     750
NEXT_3D    1125
dtype: int64


In [94]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

FEATURES = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]

train_time = df[df["reference_date"].dt.year < 2016].copy()
test_time = df[df["reference_date"].dt.year >= 2016].copy()

print("Train shape:", train_time.shape)
print("Test shape:", test_time.shape)
print("\nTrain date range:",
      train_time["reference_date"].min(),
      "to", train_time["reference_date"].max())
print("Test date range:",
      test_time["reference_date"].min(),
      "to", test_time["reference_date"].max())

Train shape: (1812, 15)
Test shape: (438, 15)

Train date range: 2007-07-27 00:00:00 to 2015-08-21 00:00:00
Test date range: 2016-05-19 00:00:00 to 2018-10-08 00:00:00


In [95]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)

# Prepare features and 1-day target
X_train = train_time[FEATURES]
X_test = test_time[FEATURES]

y_train = train_time["NEXT_1D"]
y_test = test_time["NEXT_1D"]

# Train model
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

lr_model.fit(X_train, y_train)

# Evaluate on the untouched future test set
prob = lr_model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))
print("PR-AUC:", round(average_precision_score(y_test, prob), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))

Accuracy: 0.5753
Precision: 0.2042
Recall: 0.5342
F1: 0.2955
ROC-AUC: 0.5748
PR-AUC: 0.2171
Confusion matrix:
 [[213 152]
 [ 34  39]]


In [96]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

prob = rf_model.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 4))
print("PR-AUC:", round(average_precision_score(y_test, prob), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))

Accuracy: 0.7123
Precision: 0.2427
Recall: 0.3425
F1: 0.2841
ROC-AUC: 0.6142
PR-AUC: 0.2475
Confusion matrix:
 [[287  78]
 [ 48  25]]


In [97]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

# Create geographic blocks of approximately 50–55 km
cv_data = train_time.copy()

cv_data["lat_block"] = np.floor(cv_data["latitude"] / 0.5)
cv_data["lon_block"] = np.floor(cv_data["longitude"] / 0.5)

cv_data["spatial_block"] = (
    cv_data["lat_block"].astype(str)
    + "_"
    + cv_data["lon_block"].astype(str)
)

print("Training samples:", len(cv_data))
print("Spatial blocks:", cv_data["spatial_block"].nunique())
print("Positive samples:", int(cv_data["NEXT_1D"].sum()))
print("Negative samples:", int((cv_data["NEXT_1D"] == 0).sum()))

Training samples: 1812
Spatial blocks: 42
Positive samples: 302
Negative samples: 1510


In [98]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

X = cv_data[FEATURES]
y = cv_data["NEXT_1D"]
groups = cv_data["spatial_block"]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_prob = np.full(len(cv_data), np.nan)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X, y, groups), start=1
):
    model = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_prob[val_idx] = model.predict_proba(
        X.iloc[val_idx]
    )[:, 1]

    print(
        f"Fold {fold}: "
        f"train={len(train_idx)}, "
        f"validation={len(val_idx)}, "
        f"validation positives={int(y.iloc[val_idx].sum())}"
    )

valid = ~np.isnan(oof_prob)

print("\nSpatial CV Results")
print("ROC-AUC:", round(roc_auc_score(y[valid], oof_prob[valid]), 4))
print("PR-AUC:", round(average_precision_score(y[valid], oof_prob[valid]), 4))

Fold 1: train=1452, validation=360, validation positives=60
Fold 2: train=1452, validation=360, validation positives=60
Fold 3: train=1443, validation=369, validation positives=59
Fold 4: train=1449, validation=363, validation positives=62
Fold 5: train=1452, validation=360, validation positives=61

Spatial CV Results
ROC-AUC: 0.5427
PR-AUC: 0.2013


In [99]:
import pandas as pd

importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.to_string(index=False))

       Feature  Importance
         SLOPE    0.163669
     ELEVATION    0.156951
      RAIN_15D    0.146015
       RAIN_3D    0.134386
      RAIN_30D    0.131719
       RAIN_7D    0.131503
       RAIN_1D    0.073456
LANDCOVER_CODE    0.062301


In [100]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

# Model A: static terrain features only
static_features = ["SLOPE", "ELEVATION", "LANDCOVER_CODE"]

# Model B: static features + rainfall history
full_features = static_features + [
    "RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"
]

y = cv_data["NEXT_1D"]
groups = cv_data["spatial_block"]

cv = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=42
)

results = []

for name, features in [
    ("Static only", static_features),
    ("Static + rainfall", full_features)
]:
    oof_prob = np.full(len(cv_data), np.nan)

    for train_idx, val_idx in cv.split(
        cv_data[features], y, groups
    ):
        rf = RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

        rf.fit(
            cv_data.iloc[train_idx][features],
            y.iloc[train_idx]
        )

        oof_prob[val_idx] = rf.predict_proba(
            cv_data.iloc[val_idx][features]
        )[:, 1]

    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y, oof_prob),
        "PR-AUC": average_precision_score(y, oof_prob)
    })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))

            Model  ROC-AUC  PR-AUC
      Static only   0.5375  0.2092
Static + rainfall   0.5439  0.2052


In [101]:
import pandas as pd

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

rain_features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

# Compare class counts
print("Class counts:")
print(df["NEXT_1D"].value_counts().sort_index())

# Compare reference-date distributions
print("\nReference-date summary by class:")
print(
    df.groupby("NEXT_1D")["reference_date"]
      .agg(["min", "max", "count"])
)

# Compare rainfall distributions
print("\nRainfall summary by class:")
print(
    df.groupby("NEXT_1D")[rain_features]
      .agg(["mean", "median", "std"])
      .round(2).T
)

Class counts:
NEXT_1D
0    1875
1     375
Name: count, dtype: int64

Reference-date summary by class:
               min        max  count
NEXT_1D                             
0       2007-07-27 2018-10-08   1875
1       2007-07-29 2018-10-08    375

Rainfall summary by class:
NEXT_1D               0       1
RAIN_1D  mean     11.36   12.23
         median    0.00    0.00
         std      22.60   23.71
RAIN_3D  mean     38.13   34.41
         median   22.16   19.76
         std      48.97   44.45
RAIN_7D  mean     86.30   86.61
         median   70.67   68.68
         std      77.61   82.13
RAIN_15D mean    185.03  183.15
         median  167.70  167.70
         std     129.47  126.95
RAIN_30D mean    328.33  330.60
         median  315.08  315.08
         std     225.65  224.93


In [102]:
df["year"] = df["reference_date"].dt.year

year_counts = pd.crosstab(
    df["year"],
    df["NEXT_1D"],
    margins=False
).rename(columns={
    0: "Negative",
    1: "Positive"
})

year_counts["Positive_Rate_%"] = (
    year_counts["Positive"]
    / (year_counts["Positive"] + year_counts["Negative"])
    * 100
)

print(year_counts.round(2).to_string())

NEXT_1D  Negative  Positive  Positive_Rate_%
year                                        
2007           35         7            16.67
2008           45         9            16.67
2009           40         8            16.67
2010          440        88            16.67
2011          245        49            16.67
2012           75        15            16.67
2013          400        80            16.67
2014           35         7            16.67
2015          195        39            16.67
2016          120        24            16.67
2017          115        23            16.67
2018          130        26            16.67


In [103]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")

positive = df[df["NEXT_1D"] == 1][["latitude", "longitude"]]
negative = df[df["NEXT_1D"] == 0][["latitude", "longitude"]]

pos_coords = set(map(tuple, positive.round(6).to_numpy()))
neg_coords = set(map(tuple, negative.round(6).to_numpy()))

print("Unique positive locations:", len(pos_coords))
print("Unique negative locations:", len(neg_coords))
print("Locations shared by both classes:", len(pos_coords & neg_coords))

Unique positive locations: 374
Unique negative locations: 638
Locations shared by both classes: 374


In [104]:
df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

positive = df[df["NEXT_1D"] == 1]
negative = df[df["NEXT_1D"] == 0]

overlap = positive.merge(
    negative,
    on=["latitude", "longitude"],
    suffixes=("_pos", "_neg")
)

print("Positive-negative location matches:", len(overlap))
print("\nExamples:")
print(
    overlap[[
        "latitude", "longitude",
        "reference_date_pos", "reference_date_neg"
    ]].head(10).to_string(index=False)
)

Positive-negative location matches: 754

Examples:
 latitude  longitude reference_date_pos reference_date_neg
28.866500  79.568200         2011-08-11         2011-08-10
28.866500  79.568200         2011-08-11         2011-08-09
29.158500  79.146400         2014-07-22         2014-07-21
29.158500  79.146400         2014-07-22         2014-07-20
29.295586  79.540431         2010-09-17         2010-09-16
29.295586  79.540431         2010-09-17         2010-09-15
29.330000  80.100000         2008-07-17         2008-07-16
29.330000  80.100000         2008-07-17         2008-07-15
29.337636  79.483545         2010-09-19         2010-09-18
29.337636  79.483545         2010-09-19         2010-09-17


In [105]:
print("Positive unique coordinates:", positive[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Negative unique coordinates:", negative[["latitude", "longitude"]].drop_duplicates().shape[0])

print(
    "Exact shared coordinates:",
    positive[["latitude", "longitude"]].drop_duplicates().merge(
        negative[["latitude", "longitude"]].drop_duplicates(),
        on=["latitude", "longitude"]
    ).shape[0]
)

Positive unique coordinates: 374
Negative unique coordinates: 638
Exact shared coordinates: 374


In [106]:
import pandas as pd

files = {
    "Positive definitions": "../data/multi_horizon_positive_definitions.csv",
    "Negative definitions V4": "../data/negative_location_definitions_v4.csv",
    "Negative features V4": "../data/negative_features_v4.csv",
    "Final dataset V4": "../data/himalayan_temporal_final_v4.csv"
}

for name, path in files.items():
    data = pd.read_csv(path)

    print(f"\n--- {name} ---")
    print("Shape:", data.shape)
    print("Columns:", data.columns.tolist())
    print(data.head(2).to_string(index=False))


--- Positive definitions ---
Shape: (1125, 6)
Columns: ['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'future_event_count']
 latitude  longitude reference_date event_date  positive_horizon  future_event_count
  28.8665    79.5682     2011-08-11 2011-08-12                 1                   1
  28.8665    79.5682     2011-08-10 2011-08-12                 2                   1



--- Negative definitions V4 ---
Shape: (1125, 5)
Columns: ['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon']
 latitude  longitude reference_date event_date  positive_horizon
30.755482  78.817021     2011-06-28 2011-06-29                 1
30.755482  78.817021     2011-06-27 2011-06-29                 2

--- Negative features V4 ---
Shape: (1125, 11)
Columns: ['latitude', 'longitude', 'reference_date', 'RAIN_15D', 'RAIN_1D', 'RAIN_30D', 'RAIN_3D', 'RAIN_7D', 'ELEVATION', 'LANDCOVER_CODE', 'SLOPE']
 latitude  longitude reference_date  RAIN_15D  RAIN_1D  RAIN_30D  RAIN_3D   RAIN_7D  ELEVATION  LANDCOVER_CODE     SLOPE
30.755482  78.817021     2011-06-28 351.08252      0.0 351.08252      0.0 351.08252       4212             100 37.086292
30.755482  78.817021     2011-06-27 351.08252      0.0 351.08252      0.0 351.08252       4212             100 37.086292

--- Final dataset V4 ---
Shape: (2250, 15)
Columns: ['latitude', 'longitude', 'reference_date', 'RAIN_1D',

In [107]:
import pandas as pd

files = {
    "Positive definitions": "../data/multi_horizon_positive_definitions.csv",
    "Negative definitions": "../data/negative_location_definitions_v4.csv",
    "Negative features": "../data/negative_features_v4.csv",
}

def coord_set(path):
    d = pd.read_csv(path)
    return set(
        zip(d["latitude"].round(6), d["longitude"].round(6))
    )

positive_coords = coord_set(files["Positive definitions"])

for name, path in files.items():
    coords = coord_set(path)
    overlap = positive_coords & coords

    print(
        f"{name}: unique coordinates={len(coords)}, "
        f"overlap with positives={len(overlap)}"
    )

Positive definitions: unique coordinates=374, overlap with positives=374
Negative definitions: unique coordinates=264, overlap with positives=0
Negative features: unique coordinates=264, overlap with positives=0


In [108]:
import pandas as pd

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")

for label in [0, 1]:
    part = df[df["NEXT_1D"] == label]

    coords = set(zip(
        part["latitude"].round(6),
        part["longitude"].round(6)
    ))

    print(
        f"NEXT_1D={label}: "
        f"rows={len(part)}, "
        f"unique locations={len(coords)}"
    )

print("\nFirst 5 negative coordinates:")
print(
    df.loc[df["NEXT_1D"] == 0, ["latitude", "longitude"]]
      .drop_duplicates()
      .head()
      .to_string(index=False)
)

NEXT_1D=0: rows=1875, unique locations=638
NEXT_1D=1: rows=375, unique locations=374

First 5 negative coordinates:
 latitude  longitude
28.866500  79.568200
29.158500  79.146400
29.295586  79.540431
29.330000  80.100000
29.337636  79.483545


In [109]:
import pandas as pd

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
pos_defs = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

pos_coords = set(zip(
    pos_defs["latitude"].round(6),
    pos_defs["longitude"].round(6)
))

df["coord"] = list(zip(
    df["latitude"].round(6),
    df["longitude"].round(6)
))

overlap_neg = df[
    (df["NEXT_1D"] == 0) &
    (df["coord"].isin(pos_coords))
]

print("Negative rows at positive-event locations:", len(overlap_neg))
print("\nExamples:")
print(
    overlap_neg[
        ["latitude", "longitude", "reference_date",
         "LANDSLIDE", "NEXT_1D", "NEXT_2D", "NEXT_3D"]
    ].head(10).to_string(index=False)
)

Negative rows at positive-event locations: 750

Examples:
 latitude  longitude reference_date  LANDSLIDE  NEXT_1D  NEXT_2D  NEXT_3D
28.866500  79.568200     2011-08-10          1        0        1        1
28.866500  79.568200     2011-08-09          1        0        0        1
29.158500  79.146400     2014-07-21          1        0        1        1
29.158500  79.146400     2014-07-20          1        0        0        1
29.295586  79.540431     2010-09-16          1        0        1        1
29.295586  79.540431     2010-09-15          1        0        0        1
29.330000  80.100000     2008-07-16          1        0        1        1
29.330000  80.100000     2008-07-15          1        0        0        1
29.337636  79.483545     2010-09-18          1        0        1        1
29.337636  79.483545     2010-09-17          1        0        0        1


In [110]:
import pandas as pd

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")

print("Source-type counts:")
print(df["LANDSLIDE"].value_counts().sort_index())

print("\nNEXT_1D labels within each source type:")
print(pd.crosstab(df["LANDSLIDE"], df["NEXT_1D"]))

print("\nAll prediction-target totals:")
for target in ["NEXT_1D", "NEXT_2D", "NEXT_3D"]:
    print(f"{target}: {df[target].sum()} positives out of {len(df)}")

Source-type counts:
LANDSLIDE
0    1125
1    1125
Name: count, dtype: int64

NEXT_1D labels within each source type:


NEXT_1D       0    1
LANDSLIDE           
0          1125    0
1           750  375

All prediction-target totals:
NEXT_1D: 375 positives out of 2250
NEXT_2D: 750 positives out of 2250
NEXT_3D: 1125 positives out of 2250


In [111]:
import pandas as pd

final_df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
neg_source = pd.read_csv("../data/negative_features_v4.csv")

final_neg = final_df[final_df["LANDSLIDE"] == 0]

keys = ["latitude", "longitude", "reference_date"]

final_keys = set(
    map(tuple, final_neg[keys].astype(str).to_numpy())
)
source_keys = set(
    map(tuple, neg_source[keys].astype(str).to_numpy())
)

print("Final control rows:", len(final_neg))
print("Negative source rows:", len(neg_source))
print("Final control rows matching source:", len(final_keys & source_keys))
print("Final control rows absent from source:", len(final_keys - source_keys))

Final control rows: 1125
Negative source rows: 1125
Final control rows matching source: 1125
Final control rows absent from source: 0


In [112]:
import pandas as pd

pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")
neg = pd.read_csv("../data/negative_location_definitions_v4.csv")

for df in [pos, neg]:
    df["reference_date"] = pd.to_datetime(df["reference_date"]).dt.normalize()

print("Positive rows:", len(pos))
print("Negative rows:", len(neg))

print("\nRows by reference date and horizon:")
pos_counts = pos.groupby(
    ["reference_date", "positive_horizon"]
).size()

neg_counts = neg.groupby(
    ["reference_date", "positive_horizon"]
).size()

comparison = pd.concat(
    [pos_counts.rename("positive"), neg_counts.rename("negative")],
    axis=1
).fillna(0)

print(comparison.value_counts().head())
print(
    "\nDate-horizon combinations with unequal counts:",
    int((comparison["positive"] != comparison["negative"]).sum())
)

print("\nUnique negative locations:", len(
    neg[["latitude", "longitude"]].drop_duplicates()
))
print("Negative rows with repeated coordinates:", len(neg) - len(
    neg[["latitude", "longitude"]].drop_duplicates()
))

Positive rows: 1125
Negative rows: 1125

Rows by reference date and horizon:
positive  negative
1         1           426
2         2           105
3         3            48
4         4            24
6         6            12
Name: count, dtype: int64

Date-horizon combinations with unequal counts: 0

Unique negative locations: 264
Negative rows with repeated coordinates: 861


In [113]:
import pandas as pd

neg = pd.read_csv("../data/negative_location_definitions_v4.csv")

location_counts = (
    neg.groupby(["latitude", "longitude"])
       .size()
       .rename("sample_count")
)

print("Unique negative locations:", len(location_counts))
print("Locations used once:", int((location_counts == 1).sum()))
print("Locations used 2–3 times:", int(location_counts.between(2, 3).sum()))
print("Locations used 4–5 times:", int(location_counts.between(4, 5).sum()))
print("Locations used 6+ times:", int((location_counts >= 6).sum()))

print("\nMost reused locations:")
print(location_counts.sort_values(ascending=False).head(10))

Unique negative locations: 264
Locations used once: 0
Locations used 2–3 times: 207
Locations used 4–5 times: 0
Locations used 6+ times: 57

Most reused locations:
latitude   longitude
31.081884  77.184508    33
31.691819  76.917657    24
30.732828  78.460793    21
32.212393  76.342256    18
29.566824  80.229463    15
30.260281  78.903310    15
32.547894  76.144510    15
30.887627  77.103774    15
30.368549  79.331198    12
30.660784  79.521137    12
Name: sample_count, dtype: int64


In [114]:
import pandas as pd

pool = pd.read_csv("../data/negative_safe_candidate_pool_v2.csv")

print("Candidate pool shape:", pool.shape)
print("Columns:", pool.columns.tolist())
print("First 5 rows:")
print(pool.head().to_string(index=False))

Candidate pool shape: (11901, 3)
Columns: ['latitude', 'longitude', 'nearest_event_km']
First 5 rows:
 latitude  longitude  nearest_event_km
33.160126  76.456053         66.844491
33.158295  76.463849         66.273663
33.154722  76.465480         65.858360
33.148490  76.496609         63.716602
33.123340  76.457237         63.420684


In [115]:
import pandas as pd

pool = pd.read_csv("../data/negative_safe_candidate_pool_v2.csv")
neg = pd.read_csv("../data/negative_location_definitions_v4.csv")

pool_coords = set(zip(
    pool["latitude"].round(6),
    pool["longitude"].round(6)
))

neg_coords = set(zip(
    neg["latitude"].round(6),
    neg["longitude"].round(6)
))

print("Candidate rows:", len(pool))
print("Unique candidate locations:", len(pool_coords))
print("Existing control locations:", len(neg_coords))
print("Existing controls found in candidate pool:", len(pool_coords & neg_coords))
print("Candidates at least 5 km from nearest known event:",
      int((pool["nearest_event_km"] >= 5).sum()))

Candidate rows: 11901
Unique candidate locations: 11901
Existing control locations: 264
Existing controls found in candidate pool: 264
Candidates at least 5 km from nearest known event: 10344


In [116]:
import pandas as pd
import numpy as np

pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")
pool = pd.read_csv("../data/negative_safe_candidate_pool_v2.csv")

# Prefer candidates at least 5 km from known event locations
pool = pool[pool["nearest_event_km"] >= 5].copy()

# Ensure unique coordinates
pool = pool.drop_duplicates(["latitude", "longitude"])
pool = pool.sample(frac=1, random_state=42).reset_index(drop=True)

if len(pool) < len(pos):
    raise ValueError("Not enough unique control candidates.")

# Assign a distinct control location to every positive definition row
neg = pos[["reference_date", "event_date", "positive_horizon"]].copy()
neg["latitude"] = pool.loc[:len(neg)-1, "latitude"].to_numpy()
neg["longitude"] = pool.loc[:len(neg)-1, "longitude"].to_numpy()

neg = neg[
    ["latitude", "longitude", "reference_date",
     "event_date", "positive_horizon"]
]

neg.to_csv("../data/negative_location_definitions_v5.csv", index=False)

print("Saved: negative_location_definitions_v5.csv")
print("Rows:", len(neg))
print("Unique locations:", neg[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Date-horizon rows matched:", len(neg) == len(pos))

Saved: negative_location_definitions_v5.csv
Rows: 1125
Unique locations: 1125
Date-horizon rows matched: True


In [117]:
import pandas as pd
import numpy as np

pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")
neg = pd.read_csv("../data/negative_location_definitions_v5.csv")

# Haversine distance in kilometres
def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2)**2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    )
    return 6371 * 2 * np.arcsin(np.sqrt(a))

distances = haversine(
    pos["latitude"].to_numpy(),
    pos["longitude"].to_numpy(),
    neg["latitude"].to_numpy(),
    neg["longitude"].to_numpy()
)

print("Matched pairs:", len(distances))
print("Median distance (km):", round(float(np.median(distances)), 2))
print("Mean distance (km):", round(float(np.mean(distances)), 2))
print("90th percentile (km):", round(float(np.percentile(distances, 90)), 2))
print("Pairs within 25 km:", int((distances <= 25).sum()))
print("Pairs within 50 km:", int((distances <= 50).sum()))

Matched pairs: 1125
Median distance (km): 173.5
Mean distance (km): 192.33
90th percentile (km): 344.37
Pairs within 25 km: 19
Pairs within 50 km: 71


In [118]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")
pool = pd.read_csv("../data/negative_safe_candidate_pool_v2.csv")

# Keep candidates at least 5 km from known event locations
pool = pool[pool["nearest_event_km"] >= 5].copy()
pool = pool.drop_duplicates(["latitude", "longitude"]).reset_index(drop=True)

# Build a spatial index using radians
candidate_rad = np.radians(pool[["latitude", "longitude"]].to_numpy())
tree = BallTree(candidate_rad, metric="haversine")

positive_rad = np.radians(pos[["latitude", "longitude"]].to_numpy())
radius_rad = 25 / 6371.0

# Process samples with the fewest nearby candidates first
nearby = tree.query_radius(positive_rad, r=radius_rad)
order = np.argsort([len(x) for x in nearby])

used = set()
matches = {}

for i in order:
    candidates = nearby[i]
    if len(candidates) == 0:
        continue

    distances = np.linalg.norm(candidate_rad[candidates] - positive_rad[i], axis=1)
    candidates = candidates[np.argsort(distances)]

    for j in candidates:
        j = int(j)
        if j not in used:
            used.add(j)
            matches[i] = j
            break

matched = pos.iloc[list(matches.keys())][
    ["reference_date", "event_date", "positive_horizon"]
].reset_index(drop=True)

chosen = pool.iloc[list(matches.values())][
    ["latitude", "longitude", "nearest_event_km"]
].reset_index(drop=True)

v5_matched = pd.concat([chosen, matched], axis=1)
v5_matched = v5_matched[
    ["latitude", "longitude", "reference_date",
     "event_date", "positive_horizon", "nearest_event_km"]
]

v5_matched.to_csv(
    "../data/negative_location_definitions_v5_matched.csv",
    index=False
)

print("Positive samples:", len(pos))
print("Successfully matched:", len(v5_matched))
print("Unmatched:", len(pos) - len(v5_matched))
print("Unique control locations:",
      v5_matched[["latitude", "longitude"]].drop_duplicates().shape[0])

if len(v5_matched):
    distances, _ = tree.query(
        np.radians(v5_matched[["latitude", "longitude"]].to_numpy()),
        k=1
    )
    print("Minimum event separation (km):",
          round(v5_matched["nearest_event_km"].min(), 2))

Positive samples: 1125
Successfully matched: 1125
Unmatched: 0
Unique control locations: 1125
Minimum event separation (km): 5.0


In [119]:
import pandas as pd
import numpy as np

pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")
neg = pd.read_csv("../data/negative_location_definitions_v5_matched.csv")

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = (
        np.sin(dlat / 2)**2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    )
    return 6371 * 2 * np.arcsin(np.sqrt(a))

dist = haversine(
    pos["latitude"].to_numpy(),
    pos["longitude"].to_numpy(),
    neg["latitude"].to_numpy(),
    neg["longitude"].to_numpy()
)

print("Maximum pair distance (km):", round(float(dist.max()), 2))
print("Median pair distance (km):", round(float(np.median(dist)), 2))
print("Pairs within 25 km:", int((dist <= 25).sum()))
print("Unique reference-date/horizon pairs preserved:",
      len(neg[["reference_date", "positive_horizon"]].drop_duplicates()))

Maximum pair distance (km): 511.83
Median pair distance (km): 132.56
Pairs within 25 km: 51
Unique reference-date/horizon pairs preserved: 636


In [120]:
import pandas as pd

pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")
neg = pd.read_csv("../data/negative_location_definitions_v5_matched.csv")

keys = ["reference_date", "event_date", "positive_horizon"]

print("Positive rows:", len(pos))
print("Matched control rows:", len(neg))
print("Positive key duplicates:", int(pos.duplicated(keys).sum()))
print("Control key duplicates:", int(neg.duplicated(keys).sum()))
print("Exact matching keys:", len(pos.merge(neg, on=keys)))

Positive rows: 1125
Matched control rows: 1125
Positive key duplicates: 489
Control key duplicates: 489
Exact matching keys: 4041


In [121]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nChecking possible file locations:")

paths = [
    Path("data/himalayan_temporal_final_v4.csv"),
    Path("../data/himalayan_temporal_final_v4.csv")
]

for path in paths:
    print(path.resolve(), "->", path.exists())

Current working directory:
c:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\notebooks

Checking possible file locations:
C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\notebooks\data\himalayan_temporal_final_v4.csv -> False
C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\himalayan_temporal_final_v4.csv -> True


In [122]:
file_path = Path("../data/himalayan_temporal_final_v4.csv")
df = pd.read_csv(file_path)

print(df.shape)

(2250, 15)


In [123]:
print("All columns:")
print(df.columns.tolist())

print("\nDate and target columns:")
for col in df.columns:
    if any(word in col.lower() for word in ["date", "year", "next_", "landslide", "horizon"]):
        print(f"\n{col}:")
        print(df[col].head(5).tolist())

print("\nFirst 5 rows:")
display(df.head())

All columns:
['latitude', 'longitude', 'reference_date', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D']

Date and target columns:

reference_date:
['2011-08-11', '2011-08-10', '2011-08-09', '2014-07-22', '2014-07-21']

LANDSLIDE:
[1, 1, 1, 1, 1]

NEXT_1D:
[1, 0, 0, 1, 0]

NEXT_2D:
[1, 1, 0, 1, 1]

NEXT_3D:
[1, 1, 1, 1, 1]

First 5 rows:


,latitude,longitude,reference_date,RAIN_1D,RAIN_3D,RAIN_7D,RAIN_15D,RAIN_30D,ELEVATION,SLOPE,LANDCOVER_CODE,LANDSLIDE,NEXT_1D,NEXT_2D,NEXT_3D
0,28.8665,79.5682,2011-08-11,0.000000,0.000000,70.687263,277.301850,576.698295,206,3.492825,40,1,1,1,1
1,28.8665,79.5682,2011-08-10,0.000000,12.873711,93.157185,277.301850,602.931840,206,3.492825,40,1,0,1,1
2,28.8665,79.5682,2011-08-09,0.000000,25.747421,138.097027,305.314025,620.043511,206,3.492825,40,1,0,0,1
3,29.1585,79.1464,2014-07-22,0.000000,47.428223,153.994546,238.254884,301.103128,230,3.681808,40,1,1,1,1
4,29.1585,79.1464,2014-07-21,47.428223,106.713505,177.414294,247.610094,301.103128,230,3.681808,40,1,0,1,1


In [124]:
from pathlib import Path
import pandas as pd

candidates = [
    Path("data/multi_horizon_positive_definitions.csv"),
    Path("../data/multi_horizon_positive_definitions.csv")
]

file_path = next((p for p in candidates if p.exists()), None)

if file_path is None:
    print("Definitions file not found in the expected locations.")
else:
    events = pd.read_csv(file_path)

    print("File:", file_path.resolve())
    print("Shape:", events.shape)
    print("\nColumns:")
    print(events.columns.tolist())

    print("\nFirst 10 rows:")
    display(events.head(10))

File: C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\multi_horizon_positive_definitions.csv
Shape: (1125, 6)

Columns:
['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'future_event_count']

First 10 rows:


,latitude,longitude,reference_date,event_date,positive_horizon,future_event_count
0,28.866500,79.568200,2011-08-11,2011-08-12,1,1
1,28.866500,79.568200,2011-08-10,2011-08-12,2,1
2,28.866500,79.568200,2011-08-09,2011-08-12,3,1
3,29.158500,79.146400,2014-07-22,2014-07-23,1,1
4,29.158500,79.146400,2014-07-21,2014-07-23,2,1
5,29.158500,79.146400,2014-07-20,2014-07-23,3,1
6,29.295586,79.540431,2010-09-17,2010-09-18,1,1
7,29.295586,79.540431,2010-09-16,2010-09-18,2,1
8,29.295586,79.540431,2010-09-15,2010-09-18,3,1
9,29.330000,80.100000,2008-07-17,2008-07-18,1,1


In [125]:
events_check = events.copy()

events_check["reference_date"] = pd.to_datetime(
    events_check["reference_date"]
)
events_check["event_date"] = pd.to_datetime(
    events_check["event_date"]
)

events_check["actual_gap_days"] = (
    events_check["event_date"]
    - events_check["reference_date"]
).dt.days

print("Reference-to-event day gaps:")
print(events_check["actual_gap_days"].value_counts().sort_index())

print("\nDeclared horizon vs actual day gap:")
print(pd.crosstab(
    events_check["positive_horizon"],
    events_check["actual_gap_days"]
))

print("\nRows with inconsistent horizons:")
print(
    events_check[
        events_check["positive_horizon"]
        != events_check["actual_gap_days"]
    ].shape[0]
)

Reference-to-event day gaps:
actual_gap_days
1    375
2    375
3    375
Name: count, dtype: int64

Declared horizon vs actual day gap:
actual_gap_days     1    2    3
positive_horizon               
1                 375    0    0
2                   0  375    0
3                   0    0  375

Rows with inconsistent horizons:
0


In [126]:
print("V4 rainfall feature columns:")
print([
    c for c in df.columns
    if c.startswith("RAIN_")
])

print("\nExample event-reference rows:")
display(
    events_check[
        ["reference_date", "event_date", "positive_horizon"]
    ].head(6)
)

V4 rainfall feature columns:
['RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D']

Example event-reference rows:


,reference_date,event_date,positive_horizon
0,2011-08-11,2011-08-12,1
1,2011-08-10,2011-08-12,2
2,2011-08-09,2011-08-12,3
3,2014-07-22,2014-07-23,1
4,2014-07-21,2014-07-23,2
5,2014-07-20,2014-07-23,3


In [127]:
from pathlib import Path

project = Path.cwd()

# Move to the project root if currently inside notebooks/
if project.name == "notebooks":
    project = project.parent

print("Project folder:", project)

for file in project.rglob("*"):
    if (
        file.is_file()
        and file.suffix in [".py", ".ipynb"]
        and ".venv" not in file.parts
        and ".git" not in file.parts
    ):
        try:
            content = file.read_text(encoding="utf-8")
            if "RAIN_1D" in content:
                print("\nFound in:", file.relative_to(project))
        except (UnicodeDecodeError, OSError):
            pass

Project folder: c:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction

Found in: checkpoints\checkpoint_20261004_170111\notebooks\02_event_data.ipynb

Found in: notebooks\02_event_data.ipynb

Found in: notebooks\03_event_data.ipynb

Found in: notebooks\04_model_training.ipynb


In [128]:
import json
from pathlib import Path

paths = [
    Path("notebooks/02_event_data.ipynb"),
    Path("02_event_data.ipynb")
]

notebook_path = next((p for p in paths if p.exists()), None)

if notebook_path is None:
    print("Notebook file not found.")
else:
    notebook = json.loads(
        notebook_path.read_text(encoding="utf-8")
    )

    for i, cell in enumerate(notebook["cells"]):
        source = "".join(cell.get("source", []))

        if "RAIN_1D" in source:
            print(f"\n--- Cell {i} ---")
            print(source)


--- Cell 22 ---
def get_antecedent_rainfall(lat, lon, event_date):
    point = ee.Geometry.Point([lon, lat])

    end_date = ee.Date(event_date)

    rain_1d = chirps.filterDate(
        end_date.advance(-1, "day"), end_date
    ).sum()

    rain_3d = chirps.filterDate(
        end_date.advance(-3, "day"), end_date
    ).sum()

    rain_7d = chirps.filterDate(
        end_date.advance(-7, "day"), end_date
    ).sum()

    rain_15d = chirps.filterDate(
        end_date.advance(-15, "day"), end_date
    ).sum()

    rain_30d = chirps.filterDate(
        end_date.advance(-30, "day"), end_date
    ).sum()

    combined = ee.Image.cat([
        rain_1d.rename("RAIN_1D"),
        rain_3d.rename("RAIN_3D"),
        rain_7d.rename("RAIN_7D"),
        rain_15d.rename("RAIN_15D"),
        rain_30d.rename("RAIN_30D")
    ])

    values = combined.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=5000
    )

    return values

--- Cell 24 ---
rainfall_results

In [129]:
import json
from pathlib import Path

p = Path("notebooks/02_event_data.ipynb")
if not p.exists():
    p = Path("../notebooks/02_event_data.ipynb")

nb = json.loads(p.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    src = "".join(cell.get("source", []))
    if "get_antecedent_rainfall" in src:
        print(f"\n--- Cell {i} ---\n{src}")


--- Cell 22 ---
def get_antecedent_rainfall(lat, lon, event_date):
    point = ee.Geometry.Point([lon, lat])

    end_date = ee.Date(event_date)

    rain_1d = chirps.filterDate(
        end_date.advance(-1, "day"), end_date
    ).sum()

    rain_3d = chirps.filterDate(
        end_date.advance(-3, "day"), end_date
    ).sum()

    rain_7d = chirps.filterDate(
        end_date.advance(-7, "day"), end_date
    ).sum()

    rain_15d = chirps.filterDate(
        end_date.advance(-15, "day"), end_date
    ).sum()

    rain_30d = chirps.filterDate(
        end_date.advance(-30, "day"), end_date
    ).sum()

    combined = ee.Image.cat([
        rain_1d.rename("RAIN_1D"),
        rain_3d.rename("RAIN_3D"),
        rain_7d.rename("RAIN_7D"),
        rain_15d.rename("RAIN_15D"),
        rain_30d.rename("RAIN_30D")
    ])

    values = combined.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=5000
    )

    return values

--- Cell 23 ---
test_event = eve

In [130]:
import json
from pathlib import Path

p = Path("notebooks/02_event_data.ipynb")
if not p.exists():
    p = Path("../notebooks/02_event_data.ipynb")

nb = json.loads(p.read_text(encoding="utf-8"))

for i, cell in enumerate(nb["cells"]):
    src = "".join(cell.get("source", []))
    lines = src.splitlines()

    matches = [
        j for j, line in enumerate(lines)
        if "get_antecedent_rainfall(" in line
        and not line.lstrip().startswith("def ")
    ]

    for j in matches:
        print(f"\n--- Cell {i}, line {j + 1} ---")
        for k in range(max(0, j - 3), min(len(lines), j + 5)):
            print(f"{k + 1}: {lines[k]}")


--- Cell 23, line 3 ---
1: test_event = event_records[0]
2: 
3: test_values = get_antecedent_rainfall(
4:     test_event["latitude"],
5:     test_event["longitude"],
6:     test_event["event_date"]
7: ).getInfo()

--- Cell 24, line 5 ---
2: 
3: for i, event in enumerate(event_records):
4: 
5:     values = get_antecedent_rainfall(
6:         event["latitude"],
7:         event["longitude"],
8:         event["event_date"]
9:     ).getInfo()

--- Cell 68, line 3 ---
1: test_negative = safe_negative_df.iloc[0]
2: 
3: test_values = get_antecedent_rainfall(
4:     float(test_negative["latitude"]),
5:     float(test_negative["longitude"]),
6:     test_negative["reference_date"].strftime("%Y-%m-%d")
7: ).getInfo()

--- Cell 69, line 5 ---
2: 
3: for i, row in safe_negative_df.iterrows():
4: 
5:     values = get_antecedent_rainfall(
6:         float(row["latitude"]),
7:         float(row["longitude"]),
8:         row["reference_date"].strftime("%Y-%m-%d")
9:     ).getInfo()

--- Cell 73, line 

In [131]:
from pathlib import Path
import pandas as pd

p = Path("data/multi_horizon_positive_features_v2.csv")
if not p.exists():
    p = Path("../data/multi_horizon_positive_features_v2.csv")

positive = pd.read_csv(p)

print("Shape:", positive.shape)
print("Columns:", positive.columns.tolist())
display(positive.head(3))

Shape: (1125, 17)
Columns: ['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE']


,latitude,longitude,reference_date,event_date,positive_horizon,RAIN_1D,RAIN_3D,RAIN_7D,RAIN_15D,RAIN_30D,LANDSLIDE,NEXT_1D,NEXT_2D,NEXT_3D,ELEVATION,SLOPE,LANDCOVER_CODE
0,28.8665,79.5682,2011-08-11,2011-08-12,1,0.0,0.000000,70.687263,277.301850,576.698295,1,1,1,1,206,3.492825,40
1,28.8665,79.5682,2011-08-10,2011-08-12,2,0.0,12.873711,93.157185,277.301850,602.931840,1,0,1,1,206,3.492825,40
2,28.8665,79.5682,2011-08-09,2011-08-12,3,0.0,25.747421,138.097027,305.314025,620.043511,1,0,0,1,206,3.492825,40


In [132]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

positive = pd.read_csv(
    root / "data/multi_horizon_positive_features_v2.csv"
)
negative = pd.read_csv(
    root / "data/negative_features_v4.csv"
)

print("Positive rows:", len(positive))
print("Negative rows:", len(negative))

print("\nPositive sample:")
print(positive[
    ["reference_date", "event_date", "RAIN_1D", "RAIN_3D"]
].head(3).to_string(index=False))

print("\nNegative sample:")
print(negative[
    ["reference_date", "RAIN_1D", "RAIN_3D"]
].head(3).to_string(index=False))

Positive rows: 1125
Negative rows: 1125

Positive sample:
reference_date event_date  RAIN_1D   RAIN_3D
    2011-08-11 2011-08-12      0.0  0.000000
    2011-08-10 2011-08-12      0.0 12.873711
    2011-08-09 2011-08-12      0.0 25.747421

Negative sample:
reference_date  RAIN_1D   RAIN_3D
    2011-06-28      0.0   0.00000
    2011-06-27      0.0   0.00000
    2011-06-26      0.0 175.54126


In [133]:
import json
from pathlib import Path

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

for notebook_file in [
    root / "notebooks/02_event_data.ipynb",
    root / "notebooks/03_event_data.ipynb",
]:
    if not notebook_file.exists():
        continue

    nb = json.loads(notebook_file.read_text(encoding="utf-8"))

    for i, cell in enumerate(nb["cells"]):
        source = "".join(cell.get("source", []))
        if "multi_positive_temporal" in source or "positive_features" in source:
            print(f"\n--- {notebook_file.name}, cell {i} ---")
            print(source[:1800])


--- 02_event_data.ipynb, cell 93 ---
multi_positive_records = []

for i, row in multi_horizon_positive.iterrows():

    values = get_antecedent_rainfall(
        float(row["latitude"]),
        float(row["longitude"]),
        row["reference_date"].strftime("%Y-%m-%d")
    ).getInfo()

    multi_positive_records.append({
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "reference_date": row["reference_date"],
        "event_date": row["event_date"],
        "positive_horizon": row["positive_horizon"],
        "RAIN_1D": values.get("RAIN_1D"),
        "RAIN_3D": values.get("RAIN_3D"),
        "RAIN_7D": values.get("RAIN_7D"),
        "RAIN_15D": values.get("RAIN_15D"),
        "RAIN_30D": values.get("RAIN_30D"),
        "LANDSLIDE": 1
    })

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1} / {len(multi_horizon_positive)}")

multi_positive_temporal = pd.DataFrame(
    multi_positive_records
)

multi_positive_temporal.to_csv(
    "../data/mu

In [134]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

files = {
    "positive": root / "data/multi_horizon_positive_features_v2.csv",
    "negative": root / "data/negative_features_v4.csv",
    "final": root / "data/himalayan_temporal_final_v4.csv",
}

for name, path in files.items():
    print(f"\n{name.upper()}: {path.exists()}")

    if path.exists():
        data = pd.read_csv(path)
        print("Shape:", data.shape)

        if "reference_date" in data.columns:
            print(
                "Date range:",
                data["reference_date"].min(),
                "to",
                data["reference_date"].max()
            )

        if "LANDSLIDE" in data.columns:
            print("LANDSLIDE counts:")
            print(data["LANDSLIDE"].value_counts().to_dict())


POSITIVE: True
Shape: (1125, 17)
Date range: 2007-07-27 to 2018-10-08
LANDSLIDE counts:
{1: 1125}

NEGATIVE: True
Shape: (1125, 11)
Date range: 2007-07-27 to 2018-10-08

FINAL: True
Shape: (2250, 15)
Date range: 2007-07-27 to 2018-10-08
LANDSLIDE counts:
{1: 1125, 0: 1125}


In [135]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

pos = pd.read_csv(root / "data/multi_horizon_positive_features_v2.csv")
neg = pd.read_csv(root / "data/negative_features_v4.csv")
final = pd.read_csv(root / "data/himalayan_temporal_final_v4.csv")

features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

print("Positive rainfall features match final LANDSLIDE=1 rows:")
print(
    final[final["LANDSLIDE"] == 1][features]
    .reset_index(drop=True)
    .equals(pos[features].reset_index(drop=True))
)

print("\nNegative rainfall features match final LANDSLIDE=0 rows:")
print(
    final[final["LANDSLIDE"] == 0][features]
    .reset_index(drop=True)
    .equals(neg[features].reset_index(drop=True))
)

Positive rainfall features match final LANDSLIDE=1 rows:
True

Negative rainfall features match final LANDSLIDE=0 rows:
False


In [136]:
keys = ["latitude", "longitude", "reference_date"]
features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

final_neg = final[final["LANDSLIDE"] == 0].copy()

neg_sorted = neg.sort_values(keys).reset_index(drop=True)
final_sorted = final_neg.sort_values(keys).reset_index(drop=True)

print("Same negative sample keys:")
print(neg_sorted[keys].equals(final_sorted[keys]))

print("\nSame rainfall features after sorting:")
print(
    neg_sorted[features].reset_index(drop=True).equals(
        final_sorted[features].reset_index(drop=True)
    )
)

print("\nNegative source sample:")
print(neg_sorted[keys + features].head(3).to_string(index=False))

print("\nFinal dataset negative sample:")
print(final_sorted[keys + features].head(3).to_string(index=False))

Same negative sample keys:
True

Same rainfall features after sorting:
False

Negative source sample:
 latitude  longitude reference_date  RAIN_1D   RAIN_3D    RAIN_7D   RAIN_15D   RAIN_30D
28.879174  79.586699     2011-08-09      0.0 25.747421 138.097027 305.314025 620.043511
28.879174  79.586699     2011-08-10      0.0 12.873711  93.157185 277.301850 602.931840
28.879174  79.586699     2011-08-11      0.0  0.000000  70.687263 277.301850 576.698295

Final dataset negative sample:
 latitude  longitude reference_date  RAIN_1D   RAIN_3D    RAIN_7D   RAIN_15D   RAIN_30D
28.879174  79.586699     2011-08-09      0.0 25.747421 138.097027 305.314025 620.043511
28.879174  79.586699     2011-08-10      0.0 12.873711  93.157185 277.301850 602.931840
28.879174  79.586699     2011-08-11      0.0  0.000000  70.687263 277.301850 576.698295


In [137]:
import numpy as np

a = neg_sorted[features].to_numpy(dtype=float)
b = final_sorted[features].to_numpy(dtype=float)

diff = np.abs(a - b)

print("Maximum absolute difference:", diff.max())
print("Number of unequal feature values:", np.count_nonzero(a != b))

print("\nDifferences by rainfall feature:")
for i, col in enumerate(features):
    print(
        f"{col}:",
        np.count_nonzero(a[:, i] != b[:, i]),
        "unequal values; max difference =",
        diff[:, i].max()
    )

print("\nEqual within tolerance (1e-8):", np.allclose(a, b, atol=1e-8, rtol=0))

Maximum absolute difference: 5.684341886080802e-14
Number of unequal feature values: 67

Differences by rainfall feature:
RAIN_1D: 7 unequal values; max difference = 3.552713678800501e-15
RAIN_3D: 11 unequal values; max difference = 2.842170943040401e-14
RAIN_7D: 7 unequal values; max difference = 2.842170943040401e-14
RAIN_15D: 23 unequal values; max difference = 5.684341886080802e-14
RAIN_30D: 19 unequal values; max difference = 2.842170943040401e-14

Equal within tolerance (1e-8): True


In [138]:
from pathlib import Path
import geopandas as gpd

root = Path.cwd()

if root.name == "notebooks":
    root = root.parent

shapefile = (
    root / "data" /
    "HMA_LS_Cat_point_v02.0" /
    "HMA_LS_Cat_point_v02.0.shp"
)

print("File exists:", shapefile.exists())

if shapefile.exists():
    catalog = gpd.read_file(shapefile)

    print("\nCatalog shape:", catalog.shape)
    print("\nColumns:")
    print(catalog.columns.tolist())

    print("\nSample records:")
    display(catalog.head(5))
else:
    print("Shapefile not found at the expected path.")

File exists: True

Catalog shape: (2801, 30)

Columns:
['src_name', 'src_link', 'ev_id', 'ev_date', 'ev_time', 'ev_title', 'ev_desc', 'loc_desc', 'loc_accu', 'ls_cat', 'ls_trig', 'ls_size', 'ls_setting', 'fatalities', 'injuries', 'storm_name', 'photo_link', 'comments', 'ev_imp_src', 'ev_imp_id', 'latitude', 'longitude', 'ctry_name', 'ctry_code', 'div_name', 'gaz_point', 'gaz_dist', 'sub_date', 'edit_date', 'geometry']

Sample records:


,src_name,src_link,ev_id,ev_date,ev_time,ev_title,ev_desc,loc_desc,loc_accu,ls_cat,...,latitude,longitude,ctry_name,ctry_code,div_name,gaz_point,gaz_dist,sub_date,edit_date,geometry
0,AGU,https://blogs.agu.org/landslideblog/2008/10/14...,684,2008-08-01,None,"Sigou Village, Loufan County, Shanxi Province","occurred early in morning, 11 villagers buried...","Sigou Village, Loufan County, Shanxi Province",unknown,landslide,...,32.5625,107.4500,China,CN,Shaanxi,Micang Shan,19.18,2014-04-01,2023-04-11,POINT (107.45 32.5625)
1,Reuters,https://in.reuters.com/article/idINIndia-41450...,1067,2009-07-31,None,Dailekh district,"One person was killed in Dailekh district, pol...",Dailekh district,unknown,landslide,...,28.8378,81.7080,Nepal,NP,Bheri,Katti,11.78,2014-04-01,2023-04-11,POINT (81.708 28.8378)
2,UPI,https://www.upi.com/Top_News/2008/11/05/China-...,873,2008-11-01,None,"Lincang City, Yunnan, Yunnan-Tibet No. 214 hig...",The report said heavy rainfall since Oct. 24 h...,"Lincang City, Yunnan, Yunnan-Tibet No. 214 hig...",10km,complex,...,23.8900,100.0845,China,CN,Yunnan,Lin-Tsang,0.27,2014-04-01,2023-04-11,POINT (100.0845 23.89)
3,BBC News,http://news.bbc.co.uk/2/hi/asia-pacific/771010...,874,2008-11-01,None,"Kunming, Yunnan",Yunnan has so far reported 40 deaths and 43 pe...,"Kunming, Yunnan",25km,complex,...,25.0967,102.6950,China,CN,Yunnan,Kunming/Wuchia Pa,14.97,2014-04-01,2023-04-11,POINT (102.695 25.0967)
4,One India,https://www.oneindia.com/2009/07/23/darjeeling...,1054,2009-07-22,None,"Gayabari, West Bengal",The disruption of popular toy train services i...,"Gayabari, West Bengal",10km,landslide,...,26.8572,88.3143,India,IN,West Bengal,Kurseong,4.60,2014-04-01,2023-04-11,POINT (88.3143 26.8572)


In [139]:
# Filter the original catalog to the two study states
study = catalog[
    catalog["ctry_name"].eq("India")
    & catalog["div_name"].isin([
        "Himachal Pradesh",
        "Uttarakhand"
    ])
].copy()

study["ev_date"] = pd.to_datetime(
    study["ev_date"], errors="coerce"
)

print("Study-region rows:", len(study))
print("Valid event dates:", study["ev_date"].notna().sum())
print("Unique event dates:", study["ev_date"].nunique())
print("Unique coordinates:", study[
    ["latitude", "longitude"]
].drop_duplicates().shape[0])

print("\nEvents by year:")
print(study["ev_date"].dt.year.value_counts().sort_index())

print("\nEvents by trigger category:")
print(study["ls_trig"].value_counts(dropna=False).head(10))

Study-region rows: 208
Valid event dates: 208
Unique event dates: 141
Unique coordinates: 207

Events by year:
ev_date
2007     2
2008    10
2009     5
2010    44
2011    19
2012     9
2013    30
2014     6
2015    22
2016    11
2017    22
2018    28
Name: count, dtype: int64

Events by trigger category:
ls_trig
downpour             104
rain                  45
unknown               18
continuous_rain       17
construction           9
monsoon                7
snowfall_snowmelt      5
mining                 3
Name: count, dtype: int64


In [140]:
print("Country values:")
print(catalog["ctry_name"].value_counts().head(10))

print("\nDivision values containing Himachal or Uttarakhand:")
mask = catalog["div_name"].astype(str).str.contains(
    "Himachal|Uttarakhand|Uttaranchal",
    case=False,
    na=False
)
print(catalog.loc[mask, "div_name"].value_counts())

print("\nIndia division values:")
print(
    catalog.loc[
        catalog["ctry_name"].eq("India"), "div_name"
    ].value_counts().head(40)
)

Country values:
ctry_name
India         1349
Nepal          586
China          339
Pakistan       181
Bangladesh      91
Vietnam         65
Kyrgyzstan      64
Myanmar         43
Tajikistan      30
Bhutan          26
Name: count, dtype: int64

Division values containing Himachal or Uttarakhand:
div_name
Uttaranchal         221
Himachal Pradesh    208
Name: count, dtype: int64

India division values:
div_name
Jammu and Kashmir    295
Uttaranchal          221
Himachal Pradesh     208
Manipur              102
Assam                 94
Nagaland              91
West Bengal           88
Arunachal Pradesh     74
Sikkim                56
Mizoram               38
Meghalaya             35
Tripura                8
Orissa                 7
Rajasthan              6
Uttar Pradesh          6
Haryana                5
Delhi                  4
Bihar                  3
Madhya Pradesh         2
Gujarat                2
Maharashtra            2
Jharkhand              2
Name: count, dtype: int64


In [141]:
# Include the historical name Uttaranchal
study_corrected = catalog[
    catalog["ctry_name"].eq("India")
    & catalog["div_name"].isin([
        "Himachal Pradesh",
        "Uttaranchal",
        "Uttarakhand"
    ])
].copy()

study_corrected["ev_date"] = pd.to_datetime(
    study_corrected["ev_date"], errors="coerce"
)

print("Corrected study-region records:", len(study_corrected))
print("Valid event dates:", study_corrected["ev_date"].notna().sum())
print("Unique event dates:", study_corrected["ev_date"].nunique())
print(
    "Unique coordinates:",
    study_corrected[["latitude", "longitude"]]
    .drop_duplicates().shape[0]
)

print("\nRecords by division:")
print(study_corrected["div_name"].value_counts())

Corrected study-region records: 429
Valid event dates: 429
Unique event dates: 257
Unique coordinates: 425

Records by division:
div_name
Uttaranchal         221
Himachal Pradesh    208
Name: count, dtype: int64


In [142]:
study_corrected["event_key"] = (
    study_corrected["latitude"].round(4).astype(str)
    + "_"
    + study_corrected["longitude"].round(4).astype(str)
    + "_"
    + study_corrected["ev_date"].astype(str)
)

print("Total event records:", len(study_corrected))
print("Unique location-date combinations:",
      study_corrected["event_key"].nunique())

print("\nRepeated location-date combinations:")
duplicates = study_corrected[
    study_corrected.duplicated("event_key", keep=False)
].sort_values("event_key")

display(duplicates[
    ["ev_id", "ev_date", "latitude", "longitude",
     "ev_title", "ls_trig"]
].head(20))

Total event records: 429
Unique location-date combinations: 427

Repeated location-date combinations:


,ev_id,ev_date,latitude,longitude,ev_title,ls_trig
1075,5088,2013-07-12,29.3802,79.4633,"Lohardobha Billage, Dhari, Nainital",downpour
1096,5086,2013-07-12,29.3802,79.4633,"Nainital District, Uttarakhand",rain
1085,5196,2013-07-25,30.8928,78.4800,"Uttarkashi, Uttarakhand",continuous_rain
1918,5198,2013-07-25,30.8928,78.4800,"Quaritok, Uttarkashi, Uttarakhand",continuous_rain


In [143]:
duplicates = study_corrected[
    study_corrected.duplicated(
        ["latitude", "longitude", "ev_date"],
        keep=False
    )
].sort_values(["ev_date", "latitude", "longitude"])

cols = [
    "ev_id", "ev_date", "ev_title", "ev_desc",
    "loc_desc", "ls_trig", "src_name", "src_link"
]

display(duplicates[cols].T)

,1075,1096,1085,1918
ev_id,5088,5086,5196,5198
ev_date,2013-07-12 00:00:00,2013-07-12 00:00:00,2013-07-25 00:00:00,2013-07-25 00:00:00
ev_title,"Lohardobha Billage, Dhari, Nainital","Nainital District, Uttarakhand","Uttarkashi, Uttarakhand","Quaritok, Uttarkashi, Uttarakhand"
ev_desc,Six people were killed in a landslide in Uttar...,Six people were killed in a landslide in Uttar...,Incessant rains continued to buffet Uttarakhan...,Landslides struck Quaritok and Malla villages ...
loc_desc,"Lohardobha Billage, Dhari, Nainital","Nainital District, Uttarakhand","Uttarkashi, Uttarakhand","Quaritok, Uttarkashi, Uttarakhand"
ls_trig,downpour,rain,continuous_rain,continuous_rain
src_name,www.newsworms.com,www.newsworms.com,www.newindianexpress.com,www.newindianexpress.com
src_link,http://www.newsworms.com/news_detail.php?ncatI...,http://www.newsworms.com/news_detail.php?ncatI...,http://www.newindianexpress.com/nation/Fresh-l...,http://www.newindianexpress.com/nation/Fresh-l...


In [144]:
rain_triggers = [
    "downpour",
    "rain",
    "continuous_rain",
    "monsoon"
]

rain_events = study_corrected[
    study_corrected["ls_trig"]
    .astype(str)
    .str.lower()
    .isin(rain_triggers)
].copy()

rain_events["latitude_key"] = rain_events["latitude"].round(4)
rain_events["longitude_key"] = rain_events["longitude"].round(4)

print("Rainfall-related records:", len(rain_events))

print(
    "Unique rainfall-related location-date pairs:",
    rain_events.drop_duplicates(
        ["latitude_key", "longitude_key", "ev_date"]
    ).shape[0]
)

print("\nTrigger counts:")
print(rain_events["ls_trig"].value_counts())

Rainfall-related records: 377
Unique rainfall-related location-date pairs: 375

Trigger counts:
ls_trig
downpour           221
rain                94
continuous_rain     52
monsoon             10
Name: count, dtype: int64


In [145]:
print("First event date:", rain_events["ev_date"].min())
print("Last event date:", rain_events["ev_date"].max())

print("\nEvents by year:")
print(
    rain_events["ev_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

print("\nMissing coordinates:")
print(
    rain_events[["latitude", "longitude"]]
    .isna()
    .sum()
)

First event date: 2007-07-30 00:00:00
Last event date: 2018-10-09 00:00:00

Events by year:
ev_date
2007     7
2008     9
2009     8
2010    88
2011    49
2012    15
2013    82
2014     7
2015    39
2016    24
2017    23
2018    26
Name: count, dtype: int64

Missing coordinates:
latitude     0
longitude    0
dtype: int64


In [148]:
import ee

ee.Authenticate()
ee.Initialize(project="integral-server-484809-p2")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [149]:
import ee

# Check the CHIRPS collection's date coverage
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

print("First rainfall image:")
print(chirps.aggregate_min("system:time_start").getInfo())

print("\nLast rainfall image:")
print(chirps.aggregate_max("system:time_start").getInfo())

print("\nNumber of rainfall images:")
print(chirps.size().getInfo())

First rainfall image:
347155200000

Last rainfall image:
1788134400000

Number of rainfall images:
16679


In [150]:
import ee

chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")

# Sample landslide location in Uttarakhand
point = ee.Geometry.Point([78.45, 30.75])

sample_date = ee.Date("2013-07-12")

rain = chirps.filterDate(
    sample_date.advance(-7, "day"),
    sample_date
)

print("Images in 7-day window:", rain.size().getInfo())

print(
    "Rainfall image dates:",
    rain.aggregate_array("system:time_start")
        .map(lambda t: ee.Date(t).format("YYYY-MM-dd"))
        .getInfo()
)

Images in 7-day window: 7
Rainfall image dates: ['2013-07-05', '2013-07-06', '2013-07-07', '2013-07-08', '2013-07-09', '2013-07-10', '2013-07-11']


In [152]:
import os

print("Current folder:", os.getcwd())
print("\nFiles and folders here:")
print(os.listdir())

Current folder: c:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\notebooks

Files and folders here:
['01_data_analysis.ipynb', '02_event_data.ipynb', '03_event_data.ipynb', '04_model_training.ipynb']


In [153]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

train = df[df["reference_date"].dt.year < 2016]
test = df[df["reference_date"].dt.year >= 2016]

X_train = train.drop(columns=[
    "reference_date", "LANDSLIDE", "NEXT_1D", "NEXT_2D", "NEXT_3D"
], errors="ignore")

X_test = test[X_train.columns]
y_train = train["NEXT_1D"]
y_test = test["NEXT_1D"]

baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)

scores = baseline.predict_proba(X_test)[:, 1]

print("Training rows:", len(train))
print("Test rows:", len(test))
print("Test positive rate:", round(y_test.mean(), 4))
print("Baseline ROC-AUC:", round(roc_auc_score(y_test, scores), 4))
print("Baseline PR-AUC:", round(average_precision_score(y_test, scores), 4))

Training rows: 1812
Test rows: 438
Test positive rate: 0.1667
Baseline ROC-AUC: 0.5
Baseline PR-AUC: 0.1667


In [154]:
print("Training features:")
print(X_train.columns.tolist())

print("\nData types:")
print(X_train.dtypes)

print("\nMissing values:")
print(X_train.isnull().sum())

Training features:
['latitude', 'longitude', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE']

Data types:
latitude          float64
longitude         float64
RAIN_1D           float64
RAIN_3D           float64
RAIN_7D           float64
RAIN_15D          float64
RAIN_30D          float64
ELEVATION           int64
SLOPE             float64
LANDCOVER_CODE      int64
dtype: object

Missing values:
latitude          0
longitude         0
RAIN_1D           0
RAIN_3D           0
RAIN_7D           0
RAIN_15D          0
RAIN_30D          0
ELEVATION         0
SLOPE             0
LANDCOVER_CODE    0
dtype: int64


In [155]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

static_features = [
    "latitude", "longitude", "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]

full_features = static_features + [
    "RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"
]

def evaluate_model(features):
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    model.fit(train[features], y_train)
    probabilities = model.predict_proba(test[features])[:, 1]

    return (
        round(roc_auc_score(y_test, probabilities), 4),
        round(average_precision_score(y_test, probabilities), 4)
    )

print("Static features only:", evaluate_model(static_features))
print("Static + rainfall:", evaluate_model(full_features))


Static features only: (0.7052, 0.2856)
Static + rainfall: (0.6326, 0.2486)


In [156]:

rain_features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

rain_check = test[["NEXT_1D"] + rain_features].copy()

print(
    rain_check.groupby("NEXT_1D")[rain_features]
    .median()
    .T
    .rename(columns={0: "No event", 1: "Event"})
)


NEXT_1D     No event       Event
RAIN_1D     0.000000    0.000000
RAIN_3D    11.220812   19.880651
RAIN_7D    49.202517   54.533562
RAIN_15D  133.635026  135.364208
RAIN_30D  281.120983  284.150730


In [157]:

from sklearn.metrics import roc_auc_score

for feature in rain_features:
    auc = roc_auc_score(y_test, test[feature])
    print(f"{feature}: ROC-AUC = {auc:.3f}")


RAIN_1D: ROC-AUC = 0.550
RAIN_3D: ROC-AUC = 0.528
RAIN_7D: ROC-AUC = 0.509
RAIN_15D: ROC-AUC = 0.507
RAIN_30D: ROC-AUC = 0.507


In [158]:

# Compare locations of event and non-event samples
print("Latitude and longitude by target class:\n")

print(
    test.groupby("NEXT_1D")[["latitude", "longitude"]]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

print("\nNumber of unique locations by target class:")

for label, group in test.groupby("NEXT_1D"):
    locations = group[["latitude", "longitude"]].round(3).drop_duplicates()
    print(f"Class {label}: {len(locations)} unique rounded locations")


Latitude and longitude by target class:

        latitude                        longitude                       
            mean    std     min     max      mean    std     min     max
NEXT_1D                                                                 
0         31.062  0.772  29.335  32.770    78.010  1.146  76.123  80.569
1         31.064  0.778  29.351  32.764    78.007  1.153  76.123  80.565

Number of unique locations by target class:
Class 0: 138 unique rounded locations
Class 1: 73 unique rounded locations


In [159]:

location_counts = (
    test.assign(
        lat_round=test["latitude"].round(3),
        lon_round=test["longitude"].round(3)
    )
    .groupby(["lat_round", "lon_round", "NEXT_1D"])
    .size()
    .unstack(fill_value=0)
)

location_counts = location_counts.reindex(columns=[0, 1], fill_value=0)

print("Locations appearing in both classes:",
      ((location_counts[0] > 0) & (location_counts[1] > 0)).sum())

print("Locations appearing only in class 0:",
      ((location_counts[0] > 0) & (location_counts[1] == 0)).sum())

print("Locations appearing only in class 1:",
      ((location_counts[1] > 0) & (location_counts[0] == 0)).sum())

print("\nMost repeated locations:")
print(location_counts.assign(total=location_counts.sum(axis=1))
      .sort_values("total", ascending=False)
      .head(10))


Locations appearing in both classes: 73
Locations appearing only in class 0: 65
Locations appearing only in class 1: 0

Most repeated locations:
NEXT_1D              0  1  total
lat_round lon_round             
32.212    76.342     6  0      6
32.548    76.145     6  0      6
31.484    76.244     6  0      6
31.082    77.185     6  0      6
30.727    79.520     6  0      6
30.915    78.698     6  0      6
31.692    76.918     6  0      6
30.349    79.326     6  0      6
29.335    79.447     3  0      3
31.441    78.407     3  0      3


In [160]:

print("Target counts:")
print(test["NEXT_1D"].value_counts())

print("\nDuplicate location-date rows:")
print(
    test.duplicated(
        subset=["latitude", "longitude", "reference_date"]
    ).sum()
)

print("\nRows per reference date:")
print(
    test.groupby("reference_date").size()
    .describe()
)


Target counts:
NEXT_1D
0    365
1     73
Name: count, dtype: int64

Duplicate location-date rows:
0

Rows per reference date:
count    125.000000
mean       3.504000
std        2.104619
min        2.000000
25%        2.000000
50%        2.000000
75%        4.000000
max       12.000000
dtype: float64


In [161]:

print("Rows by source class:")
print(df["LANDSLIDE"].value_counts())

print("\nNEXT_1D distribution by source class:")
print(pd.crosstab(df["LANDSLIDE"], df["NEXT_1D"]))

print("\nReference year range by source class:")
print(
    df.groupby("LANDSLIDE")["reference_date"]
    .agg(["min", "max", "nunique"])
)


Rows by source class:
LANDSLIDE
1    1125
0    1125
Name: count, dtype: int64

NEXT_1D distribution by source class:
NEXT_1D       0    1
LANDSLIDE           
0          1125    0
1           750  375

Reference year range by source class:
                 min        max  nunique
LANDSLIDE                               
0         2007-07-27 2018-10-08      477
1         2007-07-27 2018-10-08      477


In [162]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

features_no_coords = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]

model_no_coords = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model_no_coords.fit(train[features_no_coords], y_train)

probs = model_no_coords.predict_proba(
    test[features_no_coords]
)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, probs), 4))
print("PR-AUC:", round(average_precision_score(y_test, probs), 4))


ROC-AUC: 0.6112
PR-AUC: 0.2357


In [163]:
import pandas as pd

controls = pd.read_csv("../data/negative_location_definitions_v4.csv")

print("Columns:", controls.columns.tolist())
print("\nFirst 10 rows:")
print(controls.head(10).to_string(index=False))

Columns: ['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon']

First 10 rows:
 latitude  longitude reference_date event_date  positive_horizon
30.755482  78.817021     2011-06-28 2011-06-29                 1
30.755482  78.817021     2011-06-27 2011-06-29                 2
30.755482  78.817021     2011-06-26 2011-06-29                 3
30.755361  78.590701     2011-07-08 2011-07-09                 1
30.755361  78.590701     2011-07-07 2011-07-09                 2
30.755361  78.590701     2011-07-06 2011-07-09                 3
30.696075  79.089299     2011-07-08 2011-07-09                 1
30.696075  79.089299     2011-07-07 2011-07-09                 2
30.696075  79.089299     2011-07-06 2011-07-09                 3
31.029412  78.399596     2011-07-08 2011-07-09                 1


In [164]:
import pandas as pd

controls = pd.read_csv("../data/negative_location_definitions_v4.csv")

print("Rows:", len(controls))
print("\nHorizon counts:")
print(controls["positive_horizon"].value_counts(dropna=False))

print("\nEvent date equals reference date?")
print(
    (pd.to_datetime(controls["event_date"]) ==
     pd.to_datetime(controls["reference_date"])).sum()
)

print("\nUnique location-date pairs:",
      controls[["latitude", "longitude", "reference_date"]]
      .drop_duplicates().shape[0])

Rows: 1125

Horizon counts:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64

Event date equals reference date?
0

Unique location-date pairs: 1125


In [166]:
import json

for notebook in [
    "02_event_data.ipynb",
    "03_event_data.ipynb",
    "04_model_training.ipynb"
]:
    with open(notebook, encoding="utf-8") as f:
        nb = json.load(f)

    for i, cell in enumerate(nb["cells"]):
        source = "".join(cell.get("source", []))
        if "negative_location_definitions_v4.csv" in source:
            print(f"\n{notebook} — cell {i}")
            print(source[:1500])


03_event_data.ipynb — cell 28
import ee
import pandas as pd
import os

ee.Authenticate()
ee.Initialize(project="integral-server-484809-p2")



controls = pd.read_csv(
    "../data/negative_location_definitions_v4.csv"
)

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

output_path = "../data/negative_features_v4.csv"
progress_path = "../data/negative_features_v4_progress.csv"

# --------------------------------------------------
# Earth Engine datasets
# --------------------------------------------------

dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

chirps = ee.ImageCollection(
    "UCSB-CHG/CHIRPS/DAILY"
)

# --------------------------------------------------
# Extract features for one control
# --------------------------------------------------

def extract_features(lat, lon, ref_date):

    point = ee.Geometry.Point([lon, 

In [167]:

import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if any(term in source.lower() for term in [
        "negative_location_definitions",
        "random.choice",
        "np.random",
        "negative locations",
        "control locations",
        "reference_date"
    ]):
        print(f"\n{'='*15} CELL {i} {'='*15}")
        print(source[:3500])



=============== CELL 1 ===============
import pandas as pd

# Load the completed rainfall extraction
df = pd.read_csv("../data/multi_horizon_positive_temporal.csv")

df["reference_date"] = pd.to_datetime(df["reference_date"]).dt.normalize()
df["event_date"] = pd.to_datetime(df["event_date"]).dt.normalize()

# Build the unique event history from the extracted records
events = df[["latitude", "longitude", "event_date"]].drop_duplicates()

location_events = (
    events.groupby(["latitude", "longitude"])["event_date"]
    .apply(list)
    .to_dict()
)

# Calculate cumulative targets from each reference date
for horizon in [1, 2, 3]:
    labels = []

    for _, row in df.iterrows():
        key = (row["latitude"], row["longitude"])
        reference_date = row["reference_date"]

        future_events = [
            event_date
            for event_date in location_events[key]
            if reference_date < event_date
            <= reference_date + pd.Timedelta(days=horizon)
        ]



In [168]:

import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    if "negative_location_definitions_v4.csv" in source and ".to_csv" in source:
        print(f"\n--- CELL {i} ---\n")
        print(source)



--- CELL 28 ---

import ee
import pandas as pd
import os

ee.Authenticate()
ee.Initialize(project="integral-server-484809-p2")



controls = pd.read_csv(
    "../data/negative_location_definitions_v4.csv"
)

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

output_path = "../data/negative_features_v4.csv"
progress_path = "../data/negative_features_v4_progress.csv"

# --------------------------------------------------
# Earth Engine datasets
# --------------------------------------------------

dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

chirps = ee.ImageCollection(
    "UCSB-CHG/CHIRPS/DAILY"
)

# --------------------------------------------------
# Extract features for one control
# --------------------------------------------------

def extract_features(lat, lon, ref_date):

    point = ee.Geometry.Point([lon, lat])

    en

In [169]:

import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell.get("source", []))
        print(f"\n--- CELL {i} ---")
        print("\n".join(source.splitlines()[:8]))



--- CELL 0 ---
import os
import pandas as pd

file_path = "../data/multi_horizon_positive_temporal.csv"

if os.path.exists(file_path):
    saved_data = pd.read_csv(file_path)


--- CELL 1 ---
import pandas as pd

# Load the completed rainfall extraction
df = pd.read_csv("../data/multi_horizon_positive_temporal.csv")

df["reference_date"] = pd.to_datetime(df["reference_date"]).dt.normalize()
df["event_date"] = pd.to_datetime(df["event_date"]).dt.normalize()


--- CELL 2 ---
import os

print("CSV files in data folder:")
print(sorted(f for f in os.listdir("../data") if f.endswith(".csv")))

if "background_locations" in globals():
    print("\nbackground_locations is still in memory:", background_locations.shape)
    background_locations.to_csv(

--- CELL 3 ---
import pandas as pd

files = [
    "../data/matched_negative_temporal.csv",
    "../data/negative_temporal_samples.csv",
]

for path in files:

--- CELL 4 ---
from pathlib import Path
from datetime import datetime
import pandas as 

In [170]:

import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

keywords = [
    "negative_location_definitions",
    "negative samples",
    "control locations",
    "np.random",
    "random.choice",
    "random.sample",
    "uniform(",
    "to_csv("
]

for i, cell in enumerate(nb["cells"]):
    source = "".join(cell.get("source", []))
    matches = [
        line.strip()
        for line in source.splitlines()
        if any(k in line.lower() for k in keywords)
    ]
    if matches:
        print(f"\nCELL {i}")
        print("\n".join(matches))



CELL 1
df.to_csv("../data/multi_horizon_positive_labeled.csv", index=False)

CELL 2
background_locations.to_csv(

CELL 4
background_locations.to_csv(background_path, index=False)

CELL 7
rng = np.random.default_rng(42)
controls.to_csv(

CELL 8
controls.to_csv(

CELL 12
background_v2.to_csv(
controls_v2.to_csv(

CELL 15
completed.to_csv(progress_path, index=False)
completed.to_csv(output_path, index=False)

CELL 18
positive_fixed.to_csv(output_path, index=False)

CELL 21
model_data.to_csv(

CELL 24
candidates_df.to_csv(

CELL 25
safe_candidates.to_csv(

CELL 26
negative_locations.to_csv(
"../data/negative_location_definitions_v3.csv",
print("../data/negative_location_definitions_v3.csv")

CELL 27
"../data/negative_location_definitions_v3.csv"
controls.to_csv(
"../data/negative_location_definitions_v3_audited.csv",
print("../data/negative_location_definitions_v3_audited.csv")

CELL 28
"../data/negative_location_definitions_v4.csv"
completed.to_csv(
completed.to_csv(

CELL 29
controls = 

In [171]:

import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

for i in [2, 4, 7, 8, 12]:
    cell = nb["cells"][i]
    print(f"\n{'='*12} CELL {i} {'='*12}\n")
    print("".join(cell.get("source", [])))



============ CELL 2 ============

import os

print("CSV files in data folder:")
print(sorted(f for f in os.listdir("../data") if f.endswith(".csv")))

if "background_locations" in globals():
    print("\nbackground_locations is still in memory:", background_locations.shape)
    background_locations.to_csv(
        "../data/background_locations.csv", index=False
    )
    print("Saved background_locations.csv")
elif "safe_negative_df" in globals():
    print("\nsafe_negative_df is still in memory:", safe_negative_df.shape)
    print("This may help recover the earlier background coordinates.")
else:
    print("\nBackground-location DataFrame not found in memory.")

============ CELL 4 ============

from pathlib import Path
from datetime import datetime
import pandas as pd
import shutil
import hashlib
import json
import zipfile

data_dir = Path("../data")
notebook_dir = Path("../notebooks")
backup_root = Path("../checkpoints")

# 1. Recover the missing background coordinate file
negative

In [172]:
import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

print("".join(nb["cells"][7].get("source", [])))

import pandas as pd
import numpy as np

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
existing = pd.read_csv("../data/matched_negative_temporal.csv")
background = pd.read_csv("../data/background_locations.csv")

for df in [positive, existing]:
    df["reference_date"] = pd.to_datetime(
        df["reference_date"]
    ).dt.normalize()

rng = np.random.default_rng(42)
rows = []
reused_dates = 0
new_dates = 0

# One control for each unique positive reference date
for ref_date in sorted(positive["reference_date"].unique()):

    available = existing[
        existing["reference_date"] == ref_date
    ]

    if not available.empty:
        # Reuse a saved control for this date
        chosen = available.iloc[0]
        lat, lon = chosen["latitude"], chosen["longitude"]
        reused_dates += 1
    else:
        # Select a previously screened background location
        chosen_idx = rng.integers(len(background))
        chosen = background.iloc[chosen_idx]
        la

In [173]:
import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

print("".join(nb["cells"][7].get("source", [])))

import pandas as pd
import numpy as np

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
existing = pd.read_csv("../data/matched_negative_temporal.csv")
background = pd.read_csv("../data/background_locations.csv")

for df in [positive, existing]:
    df["reference_date"] = pd.to_datetime(
        df["reference_date"]
    ).dt.normalize()

rng = np.random.default_rng(42)
rows = []
reused_dates = 0
new_dates = 0

# One control for each unique positive reference date
for ref_date in sorted(positive["reference_date"].unique()):

    available = existing[
        existing["reference_date"] == ref_date
    ]

    if not available.empty:
        # Reuse a saved control for this date
        chosen = available.iloc[0]
        lat, lon = chosen["latitude"], chosen["longitude"]
        reused_dates += 1
    else:
        # Select a previously screened background location
        chosen_idx = rng.integers(len(background))
        chosen = background.iloc[chosen_idx]
        la

In [174]:
import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

source = "".join(nb["cells"][7].get("source", []))

print(source[:3500])
print("\n--- REMAINDER ---\n")
print(source[3500:])

import pandas as pd
import numpy as np

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
existing = pd.read_csv("../data/matched_negative_temporal.csv")
background = pd.read_csv("../data/background_locations.csv")

for df in [positive, existing]:
    df["reference_date"] = pd.to_datetime(
        df["reference_date"]
    ).dt.normalize()

rng = np.random.default_rng(42)
rows = []
reused_dates = 0
new_dates = 0

# One control for each unique positive reference date
for ref_date in sorted(positive["reference_date"].unique()):

    available = existing[
        existing["reference_date"] == ref_date
    ]

    if not available.empty:
        # Reuse a saved control for this date
        chosen = available.iloc[0]
        lat, lon = chosen["latitude"], chosen["longitude"]
        reused_dates += 1
    else:
        # Select a previously screened background location
        chosen_idx = rng.integers(len(background))
        chosen = background.iloc[chosen_idx]
        la

In [175]:
import pandas as pd

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
negative = pd.read_csv("../data/matched_negative_temporal.csv")

positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
).dt.normalize()

negative["reference_date"] = pd.to_datetime(
    negative["reference_date"]
).dt.normalize()

positive_dates = set(positive["reference_date"])
negative_dates = set(negative["reference_date"])

print("Positive rows:", len(positive))
print("Negative rows:", len(negative))
print("Unique positive dates:", len(positive_dates))
print("Unique negative dates:", len(negative_dates))
print("Dates shared:", len(positive_dates & negative_dates))
print("Positive dates without controls:", len(positive_dates - negative_dates))

Positive rows: 1125
Negative rows: 377
Unique positive dates: 477
Unique negative dates: 212
Dates shared: 212
Positive dates without controls: 265


In [176]:
import pandas as pd

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
controls = pd.read_csv("../data/negative_location_definitions_v4.csv")

positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
).dt.normalize()

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

p_dates = set(positive["reference_date"])
c_dates = set(controls["reference_date"])

print("Positive unique dates:", len(p_dates))
print("Control unique dates:", len(c_dates))
print("Shared dates:", len(p_dates & c_dates))
print("Positive dates without controls:", len(p_dates - c_dates))
print("Control dates outside positive dates:", len(c_dates - p_dates))

Positive unique dates: 477
Control unique dates: 477
Shared dates: 477
Positive dates without controls: 0
Control dates outside positive dates: 0


In [177]:
import pandas as pd
import numpy as np

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
controls = pd.read_csv("../data/negative_location_definitions_v4.csv")

# Compare coordinates rounded to 3 decimal places (~100 m scale)
positive_locations = set(zip(
    positive["latitude"].round(3),
    positive["longitude"].round(3)
))

control_locations = set(zip(
    controls["latitude"].round(3),
    controls["longitude"].round(3)
))

overlap = positive_locations & control_locations

print("Unique positive locations:", len(positive_locations))
print("Unique control locations:", len(control_locations))
print("Locations overlapping:", len(overlap))

Unique positive locations: 369
Unique control locations: 264
Locations overlapping: 0


In [178]:

import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
controls = pd.read_csv("../data/negative_location_definitions_v4.csv")

# Use radians for haversine distance calculations
event_coords = np.radians(
    positive[["latitude", "longitude"]]
    .drop_duplicates()
    .to_numpy()
)

control_coords = np.radians(
    controls[["latitude", "longitude"]]
    .drop_duplicates()
    .to_numpy()
)

tree = BallTree(event_coords, metric="haversine")
distances, _ = tree.query(control_coords, k=1)

# Earth mean radius in kilometres
nearest_km = distances[:, 0] * 6371.0088

print("Nearest-event distance (km):")
print("Minimum:", round(nearest_km.min(), 2))
print("Median:", round(np.median(nearest_km), 2))
print("25th percentile:", round(np.percentile(nearest_km, 25), 2))
print("Controls within 1 km:", int((nearest_km < 1).sum()))
print("Controls within 5 km:", int((nearest_km < 5).sum()))
print("Controls within 10 km:", int((nearest_km < 10).sum()))


Nearest-event distance (km):
Minimum: 1.01
Median: 1.83
25th percentile: 1.43
Controls within 1 km: 0
Controls within 5 km: 264
Controls within 10 km: 264


In [179]:
import pandas as pd

controls = pd.read_csv(
    "../data/negative_location_definitions_v4.csv"
)

location_counts = (
    controls.groupby(
        [controls["latitude"].round(5),
         controls["longitude"].round(5)]
    )
    .size()
)

print("Unique control locations:", len(location_counts))
print("Locations used more than once:",
      int((location_counts > 1).sum()))
print("Maximum uses of one location:", int(location_counts.max()))
print("\nUses per location:")
print(location_counts.describe().round(2))

Unique control locations: 264
Locations used more than once: 264
Maximum uses of one location: 33

Uses per location:
count    264.00
mean       4.26
std        3.40
min        3.00
25%        3.00
50%        3.00
75%        3.00
max       33.00
dtype: float64


In [180]:

import pandas as pd

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

train = df[df["reference_date"].dt.year < 2016]
test = df[df["reference_date"].dt.year >= 2016]

def location_set(frame):
    return set(zip(
        frame["latitude"].round(5),
        frame["longitude"].round(5)
    ))

train_locations = location_set(train)
test_locations = location_set(test)

print("Unique training locations:", len(train_locations))
print("Unique test locations:", len(test_locations))
print("Locations appearing in both:", len(train_locations & test_locations))
print("Test locations unseen in training:", len(test_locations - train_locations))


Unique training locations: 523
Unique test locations: 138
Locations appearing in both: 23
Test locations unseen in training: 115


In [181]:

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

# Use only the existing training period
train = df[df["reference_date"].dt.year < 2016].copy()

static_features = [
    "latitude", "longitude",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]

full_features = static_features + [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

# Treat coordinates rounded to 3 decimals as a location group
groups = (
    train["latitude"].round(3).astype(str) + "_" +
    train["longitude"].round(3).astype(str)
)

cv = GroupKFold(n_splits=5)

for name, features in [
    ("Static only", static_features),
    ("Static + rainfall", full_features)
]:
    aucs, aps = [], []

    for tr_idx, val_idx in cv.split(
        train[features], train["NEXT_1D"], groups
    ):
        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
        model.fit(
            train.iloc[tr_idx][features],
            train.iloc[tr_idx]["NEXT_1D"]
        )

        probabilities = model.predict_proba(
            train.iloc[val_idx][features]
        )[:, 1]

        y_val = train.iloc[val_idx]["NEXT_1D"]
        aucs.append(roc_auc_score(y_val, probabilities))
        aps.append(average_precision_score(y_val, probabilities))

    print(
        f"{name}: "
        f"ROC-AUC={np.mean(aucs):.4f}, "
        f"PR-AUC={np.mean(aps):.4f}"
    )


Static only: ROC-AUC=0.5557, PR-AUC=0.2299
Static + rainfall: ROC-AUC=0.4796, PR-AUC=0.1618


In [182]:

import pandas as pd

train = df[df["reference_date"].dt.year < 2016].copy()

rain_features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

print(
    train.groupby("NEXT_1D")[rain_features]
    .agg(["median", "mean", "std"])
    .round(2)
    .T
)


NEXT_1D               0       1
RAIN_1D  median    0.00    0.00
         mean     11.68   12.25
         std      23.26   24.58
RAIN_3D  median   23.68   19.74
         mean     40.06   35.13
         std      50.94   46.22
RAIN_7D  median   74.87   71.42
         mean     92.21   92.00
         std      80.22   85.34
RAIN_15D median  179.29  179.46
         mean    193.44  190.35
         std     130.26  127.09
RAIN_30D median  321.73  316.58
         mean    337.13  339.21
         std     227.02  226.48


In [1]:

import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

source = "".join(nb["cells"][7].get("source", []))

marker = "# One control for each unique positive reference date"
start = source.find(marker)

print("Cell length:", len(source))
print("Sampling section starts at:", start)

if start >= 0:
    print(source[start:start + 2000])
else:
    print("Marker not found. Search for 'for ref_date in'.")


Cell length: 1669
Sampling section starts at: 435
# One control for each unique positive reference date
for ref_date in sorted(positive["reference_date"].unique()):

    available = existing[
        existing["reference_date"] == ref_date
    ]

    if not available.empty:
        # Reuse a saved control for this date
        chosen = available.iloc[0]
        lat, lon = chosen["latitude"], chosen["longitude"]
        reused_dates += 1
    else:
        # Select a previously screened background location
        chosen_idx = rng.integers(len(background))
        chosen = background.iloc[chosen_idx]
        lat, lon = chosen["latitude"], chosen["longitude"]
        new_dates += 1

    rows.append({
        "latitude": lat,
        "longitude": lon,
        "reference_date": ref_date,
        "LANDSLIDE": 0
    })

controls = pd.DataFrame(rows)
controls.to_csv(
    "../data/multi_horizon_negative_definitions.csv",
    index=False
)

print("Control definitions saved.")
print("Rows:", len(c

In [2]:

import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

source = "".join(nb["cells"][7].get("source", []))

print(source[400:1100])
print("\n--- NEXT PART ---\n")
print(source[1100:])


[]
reused_dates = 0
new_dates = 0

# One control for each unique positive reference date
for ref_date in sorted(positive["reference_date"].unique()):

    available = existing[
        existing["reference_date"] == ref_date
    ]

    if not available.empty:
        # Reuse a saved control for this date
        chosen = available.iloc[0]
        lat, lon = chosen["latitude"], chosen["longitude"]
        reused_dates += 1
    else:
        # Select a previously screened background location
        chosen_idx = rng.integers(len(background))
        chosen = background.iloc[chosen_idx]
        lat, lon = chosen["latitude"], chosen["longitude"]
        new_dates += 1

    rows.append({
        "

--- NEXT PART ---

latitude": lat,
        "longitude": lon,
        "reference_date": ref_date,
        "LANDSLIDE": 0
    })

controls = pd.DataFrame(rows)
controls.to_csv(
    "../data/multi_horizon_negative_definitions.csv",
    index=False
)

print("Control definitions saved.")
print("Rows:",

In [3]:
import json

with open("03_event_data.ipynb", encoding="utf-8") as f:
    nb = json.load(f)

source = "".join(nb["cells"][7].get("source", []))

print(source[-900:])

titude"], chosen["longitude"]
        reused_dates += 1
    else:
        # Select a previously screened background location
        chosen_idx = rng.integers(len(background))
        chosen = background.iloc[chosen_idx]
        lat, lon = chosen["latitude"], chosen["longitude"]
        new_dates += 1

    rows.append({
        "latitude": lat,
        "longitude": lon,
        "reference_date": ref_date,
        "LANDSLIDE": 0
    })

controls = pd.DataFrame(rows)
controls.to_csv(
    "../data/multi_horizon_negative_definitions.csv",
    index=False
)

print("Control definitions saved.")
print("Rows:", len(controls))
print("Unique dates:", controls["reference_date"].nunique())
print("Existing dates reused:", reused_dates)
print("New dates needing feature extraction:", new_dates)
print("Missing values:", controls.isnull().sum().sum())
print("File: multi_horizon_negative_definitions.csv")


In [4]:
import pandas as pd

controls = pd.read_csv(
    "../data/multi_horizon_negative_definitions.csv"
)

location_counts = (
    controls.groupby([
        controls["latitude"].round(5),
        controls["longitude"].round(5)
    ]).size()
)

print("Control rows:", len(controls))
print("Unique locations:", len(location_counts))
print("Locations used more than once:", int((location_counts > 1).sum()))
print("Maximum uses of one location:", int(location_counts.max()))
print("\nUses per location:")
print(location_counts.describe().round(2))

Control rows: 477
Unique locations: 346
Locations used more than once: 106
Maximum uses of one location: 4

Uses per location:
count    346.00
mean       1.38
std        0.62
min        1.00
25%        1.00
50%        1.00
75%        2.00
max        4.00
dtype: float64


In [5]:

import pandas as pd

positive = pd.read_csv(
    "../data/multi_horizon_positive_labeled.csv"
)
controls = pd.read_csv(
    "../data/multi_horizon_negative_definitions.csv"
)

print("Positive rows:", len(positive))
print("Control rows:", len(controls))

print("\nPositive columns:")
print(positive.columns.tolist())

print("\nControl columns:")
print(controls.columns.tolist())

print("\nControl sample:")
print(controls.head())


Positive rows: 1125
Control rows: 477

Positive columns:
['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D']

Control columns:
['latitude', 'longitude', 'reference_date', 'LANDSLIDE']

Control sample:
    latitude  longitude reference_date  LANDSLIDE
0  32.301650  77.934624     2007-07-27          0
1  29.841000  79.775800     2007-07-28          0
2  31.598600  78.438400     2007-07-29          0
3  30.653877  79.597045     2007-08-08          0
4  31.050818  77.185099     2007-08-09          0


In [6]:

import pandas as pd

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
controls = pd.read_csv("../data/multi_horizon_negative_definitions.csv")

positive["reference_date"] = pd.to_datetime(positive["reference_date"])
controls["reference_date"] = pd.to_datetime(controls["reference_date"])

# Check whether controls share reference dates with positive examples
positive_dates = set(positive["reference_date"].dt.normalize())
control_dates = set(controls["reference_date"].dt.normalize())

# Check exact coordinate overlap
positive_locations = set(zip(
    positive["latitude"].round(5),
    positive["longitude"].round(5)
))
control_locations = set(zip(
    controls["latitude"].round(5),
    controls["longitude"].round(5)
))

print("Positive dates:", len(positive_dates))
print("Control dates:", len(control_dates))
print("Dates shared:", len(positive_dates & control_dates))
print("Control dates without positives:", len(control_dates - positive_dates))
print("Exact rounded location overlap:", len(positive_locations & control_locations))


Positive dates: 477
Control dates: 477
Dates shared: 477
Control dates without positives: 0
Exact rounded location overlap: 187


In [7]:

import pandas as pd

positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")
controls = pd.read_csv("../data/multi_horizon_negative_definitions.csv")

for frame in [positive, controls]:
    frame["reference_date"] = pd.to_datetime(
        frame["reference_date"]
    ).dt.normalize()

positive["event_date"] = pd.to_datetime(
    positive["event_date"]
).dt.normalize()

# Match locations at five-decimal coordinate precision
for frame in [positive, controls]:
    frame["lat_key"] = frame["latitude"].round(5)
    frame["lon_key"] = frame["longitude"].round(5)

events = positive[
    ["lat_key", "lon_key", "event_date"]
].drop_duplicates()

matches = controls.merge(
    events,
    on=["lat_key", "lon_key"],
    how="inner"
)

matches["days_until_event"] = (
    matches["event_date"] - matches["reference_date"]
).dt.days

contaminated = matches[
    matches["days_until_event"].between(1, 3)
]

print("Controls:", len(controls))
print("Controls matching known event locations:", 
      controls[["lat_key", "lon_key"]].drop_duplicates().merge(
          events[["lat_key", "lon_key"]].drop_duplicates(),
          on=["lat_key", "lon_key"]
      ).shape[0])
print("Controls with known event in next 1–3 days:",
      contaminated[["latitude", "longitude", "reference_date"]]
      .drop_duplicates().shape[0])


Controls: 477
Controls matching known event locations: 187
Controls with known event in next 1–3 days: 4


In [8]:
print(
    contaminated[
        [
            "latitude",
            "longitude",
            "reference_date",
            "event_date",
            "days_until_event",
        ]
    ].drop_duplicates().to_string(index=False)
)

 latitude  longitude reference_date event_date  days_until_event
30.719800  77.169500     2010-09-18 2010-09-19                 1
30.430500  78.080300     2014-07-18 2014-07-19                 1
31.103127  77.203555     2017-09-01 2017-09-02                 1
30.722031  78.445173     2018-05-31 2018-06-01                 1


In [9]:

# Create a cleaned experimental copy; preserve the original file.
controls = pd.read_csv("../data/multi_horizon_negative_definitions.csv")
controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

controls["lat_key"] = controls["latitude"].round(5)
controls["lon_key"] = controls["longitude"].round(5)

event_keys = events.copy()
event_keys["lat_key"] = event_keys["lat_key"].round(5)
event_keys["lon_key"] = event_keys["lon_key"].round(5)

check = controls.merge(
    event_keys,
    on=["lat_key", "lon_key"],
    how="left"
)

days = (check["event_date"] - check["reference_date"]).dt.days

bad_keys = set(
    zip(
        check.loc[days.between(1, 3), "lat_key"],
        check.loc[days.between(1, 3), "lon_key"],
        check.loc[days.between(1, 3), "reference_date"]
    )
)

control_keys = list(zip(
    controls["lat_key"],
    controls["lon_key"],
    controls["reference_date"]
))

keep = [key not in bad_keys for key in control_keys]
clean_controls = controls.loc[keep].drop(
    columns=["lat_key", "lon_key"]
)

clean_controls.to_csv(
    "../data/multi_horizon_negative_definitions_cleaned.csv",
    index=False
)

print("Original controls:", len(controls))
print("Cleaned controls:", len(clean_controls))
print("Saved cleaned experimental copy.")


Original controls: 477
Cleaned controls: 473
Saved cleaned experimental copy.


In [11]:
import pandas as pd

files = [
    "../data/multi_horizon_negative_definitions.csv",
    "../data/negative_location_definitions_v4.csv",
    "../data/negative_features_v4.csv",
    "../data/himalayan_temporal_final_v4.csv",
]

for file in files:
    df = pd.read_csv(file)
    print(f"\n{file}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    


../data/multi_horizon_negative_definitions.csv
Shape: (477, 4)
Columns: ['latitude', 'longitude', 'reference_date', 'LANDSLIDE']

../data/negative_location_definitions_v4.csv
Shape: (1125, 5)
Columns: ['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon']

../data/negative_features_v4.csv
Shape: (1125, 11)
Columns: ['latitude', 'longitude', 'reference_date', 'RAIN_15D', 'RAIN_1D', 'RAIN_30D', 'RAIN_3D', 'RAIN_7D', 'ELEVATION', 'LANDCOVER_CODE', 'SLOPE']

../data/himalayan_temporal_final_v4.csv
Shape: (2250, 15)
Columns: ['latitude', 'longitude', 'reference_date', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D']


In [12]:
import pandas as pd

neg = pd.read_csv("../data/negative_location_definitions_v4.csv")
pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

for df in [neg, pos]:
    df["reference_date"] = pd.to_datetime(df["reference_date"])
    df["event_date"] = pd.to_datetime(df["event_date"])

print("Negative definitions — horizons:")
print(neg["positive_horizon"].value_counts(dropna=False).sort_index())

print("\nNegative definitions — event date offsets:")
print((neg["event_date"] - neg["reference_date"]).dt.days.value_counts().sort_index())

print("\nFirst 10 negative definitions:")
print(neg.head(10).to_string(index=False))

Negative definitions — horizons:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64

Negative definitions — event date offsets:
1    375
2    375
3    375
Name: count, dtype: int64

First 10 negative definitions:
 latitude  longitude reference_date event_date  positive_horizon
30.755482  78.817021     2011-06-28 2011-06-29                 1
30.755482  78.817021     2011-06-27 2011-06-29                 2
30.755482  78.817021     2011-06-26 2011-06-29                 3
30.755361  78.590701     2011-07-08 2011-07-09                 1
30.755361  78.590701     2011-07-07 2011-07-09                 2
30.755361  78.590701     2011-07-06 2011-07-09                 3
30.696075  79.089299     2011-07-08 2011-07-09                 1
30.696075  79.089299     2011-07-07 2011-07-09                 2
30.696075  79.089299     2011-07-06 2011-07-09                 3
31.029412  78.399596     2011-07-08 2011-07-09                 1


In [13]:
import pandas as pd

neg = pd.read_csv("../data/negative_location_definitions_v4.csv")
pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

for df in [neg, pos]:
    df["lat_key"] = df["latitude"].round(5)
    df["lon_key"] = df["longitude"].round(5)

pos_locations = set(zip(pos["lat_key"], pos["lon_key"]))
neg_locations = set(zip(neg["lat_key"], neg["lon_key"]))

print("Unique positive locations:", len(pos_locations))
print("Unique negative locations:", len(neg_locations))
print("Overlapping locations:", len(pos_locations & neg_locations))
print("Negative locations also in positives:", 
      sorted(neg_locations & pos_locations)[:10])

Unique positive locations: 374
Unique negative locations: 264
Overlapping locations: 0
Negative locations also in positives: []


In [14]:
import pandas as pd

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")

print("LANDSLIDE vs NEXT_1D:")
print(pd.crosstab(df["LANDSLIDE"], df["NEXT_1D"]))

print("\nLANDSLIDE vs NEXT_2D:")
print(pd.crosstab(df["LANDSLIDE"], df["NEXT_2D"]))

print("\nLANDSLIDE vs NEXT_3D:")
print(pd.crosstab(df["LANDSLIDE"], df["NEXT_3D"]))

LANDSLIDE vs NEXT_1D:
NEXT_1D       0    1
LANDSLIDE           
0          1125    0
1           750  375

LANDSLIDE vs NEXT_2D:
NEXT_2D       0    1
LANDSLIDE           
0          1125    0
1           375  750

LANDSLIDE vs NEXT_3D:
NEXT_3D       0     1
LANDSLIDE            
0          1125     0
1             0  1125


In [15]:
import pandas as pd

df = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

print("Reference-date range by source class:")
print(
    df.groupby("LANDSLIDE")["reference_date"]
      .agg(["min", "max", "nunique"])
)

print("\nSamples per year by source class:")
df["year"] = df["reference_date"].dt.year
print(pd.crosstab(df["year"], df["LANDSLIDE"]))

Reference-date range by source class:
                 min        max  nunique
LANDSLIDE                               
0         2007-07-27 2018-10-08      477
1         2007-07-27 2018-10-08      477

Samples per year by source class:
LANDSLIDE    0    1
year               
2007        21   21
2008        27   27
2009        24   24
2010       264  264
2011       147  147
2012        45   45
2013       240  240
2014        21   21
2015       117  117
2016        72   72
2017        69   69
2018        78   78


In [16]:
rain_cols = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

print(
    df.groupby("LANDSLIDE")[rain_cols]
      .median()
      .T
      .rename(columns={0: "Control", 1: "Positive"})
)

print("\nMean rainfall:")
print(
    df.groupby("LANDSLIDE")[rain_cols]
      .mean()
      .T
      .rename(columns={0: "Control", 1: "Positive"})
)

LANDSLIDE     Control    Positive
RAIN_1D      0.000000    0.000000
RAIN_3D     22.155958   20.674885
RAIN_7D     70.778240   69.502933
RAIN_15D   168.162197  167.700060
RAIN_30D   317.565555  313.130581

Mean rainfall:
LANDSLIDE     Control    Positive
RAIN_1D     11.543861   11.469132
RAIN_3D     37.654364   37.370567
RAIN_7D     86.657346   86.036882
RAIN_15D   185.215966  184.210917
RAIN_30D   329.073974  328.346757


In [17]:

import pandas as pd
import numpy as np

pos = pd.read_csv("../data/multi_horizon_positive_definitions.csv")
neg = pd.read_csv("../data/negative_location_definitions_v4.csv")

# Approximate CHIRPS 0.05-degree grid-cell indices
for df in [pos, neg]:
    df["grid_lon"] = np.floor((df["longitude"] + 180) / 0.05).astype(int)
    df["grid_lat"] = np.floor((50 - df["latitude"]) / 0.05).astype(int)

keys = ["reference_date", "positive_horizon", "grid_lat", "grid_lon"]

pos_cells = pos[keys].drop_duplicates()
neg_cells = neg[keys].drop_duplicates()

same_cell = neg_cells.merge(pos_cells, on=keys, how="inner")

print("Negative samples:", len(neg))
print("Unique negative date-horizon-cell combinations:", len(neg_cells))
print("Matching positive grid cells on the same date and horizon:", len(same_cell))
print("Percentage sharing a cell:", round(100 * len(same_cell) / len(neg_cells), 2), "%")


Negative samples: 1125
Unique negative date-horizon-cell combinations: 1113
Matching positive grid cells on the same date and horizon: 552
Percentage sharing a cell: 49.6 %


In [18]:

import pandas as pd

pos = pd.read_csv("../data/multi_horizon_positive_features_v2.csv")
neg = pd.read_csv("../data/negative_features_v4.csv")

for df in [pos, neg]:
    df["reference_date"] = pd.to_datetime(df["reference_date"])
    df["grid_lon"] = ((df["longitude"] + 180) / 0.05).apply(int)
    df["grid_lat"] = ((50 - df["latitude"]) / 0.05).apply(int)

keys = ["reference_date", "grid_lat", "grid_lon"]
rain = ["RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"]

p = pos[keys + rain].drop_duplicates(keys)
n = neg[keys + rain].drop_duplicates(keys)

matched = n.merge(p, on=keys, suffixes=("_neg", "_pos"))

print("Matched grid-date combinations:", len(matched))

for col in rain:
    diff = (matched[f"{col}_neg"] - matched[f"{col}_pos"]).abs()
    print(f"{col}: median absolute difference = {diff.median():.4f}")


Matched grid-date combinations: 554
RAIN_1D: median absolute difference = 0.0000
RAIN_3D: median absolute difference = 0.0000
RAIN_7D: median absolute difference = 0.0000
RAIN_15D: median absolute difference = 0.0000
RAIN_30D: median absolute difference = 0.0000


In [19]:

import pandas as pd

background = pd.read_csv("../data/background_locations.csv")
positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")

print("Background locations")
print("Shape:", background.shape)
print("Columns:", background.columns.tolist())
print(background.head())

print("\nPositive samples")
print("Shape:", positive.shape)
print("Columns:", positive.columns.tolist())

print("\nBackground coordinate duplicates:")
print(
    background.duplicated(
        subset=["latitude", "longitude"]
    ).sum()
    if {"latitude", "longitude"}.issubset(background.columns)
    else "Coordinate columns not found"
)


Background locations
Shape: (374, 2)
Columns: ['latitude', 'longitude']
    latitude  longitude
0  28.866500  79.568200
1  29.158500  79.146400
2  29.295586  79.540431
3  29.330000  80.100000
4  29.337636  79.483545

Positive samples
Shape: (1125, 14)
Columns: ['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D']

Background coordinate duplicates:
0


In [21]:

import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

background = pd.read_csv("../data/background_locations.csv")
positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")

# Coordinates in radians for distance calculation
bg_rad = np.radians(background[["latitude", "longitude"]].to_numpy())
pos_rad = np.radians(
    positive[["latitude", "longitude"]].drop_duplicates().to_numpy()
)

tree = BallTree(pos_rad, metric="haversine")
distances, _ = tree.query(bg_rad, k=1)

# Earth radius in kilometres
nearest_km = distances[:, 0] * 6371

background["nearest_known_landslide_km"] = nearest_km

print("Nearest known landslide distance (km):")
print(background["nearest_known_landslide_km"].describe().round(2))

for radius in [1, 3, 5, 10]:
    count = (nearest_km >= radius).sum()
    print(f"At least {radius} km away: {count} / {len(background)}")

print("\nFarthest-separated candidates:")
print(
    background.nlargest(10, "nearest_known_landslide_km")
    .round(3)
    .to_string(index=False)
)


Nearest known landslide distance (km):
count    374.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: nearest_known_landslide_km, dtype: float64
At least 1 km away: 0 / 374
At least 3 km away: 0 / 374
At least 5 km away: 0 / 374
At least 10 km away: 0 / 374

Farthest-separated candidates:
 latitude  longitude  nearest_known_landslide_km
   28.866     79.568                         0.0
   29.158     79.146                         0.0
   29.296     79.540                         0.0
   29.330     80.100                         0.0
   29.338     79.484                         0.0
   29.351     79.451                         0.0
   29.358     79.552                         0.0
   29.360     80.126                         0.0
   29.380     79.463                         0.0
   29.391     79.455                         0.0


In [22]:

import pandas as pd

background = pd.read_csv("../data/background_locations.csv")
positive = pd.read_csv("../data/multi_horizon_positive_labeled.csv")

bg_coords = set(zip(
    background["latitude"].round(6),
    background["longitude"].round(6)
))

pos_coords = set(zip(
    positive["latitude"].round(6),
    positive["longitude"].round(6)
))

overlap = bg_coords & pos_coords

print("Unique background locations:", len(bg_coords))
print("Unique positive locations:", len(pos_coords))
print("Exact rounded coordinate overlaps:", len(overlap))
print("Example overlapping coordinates:", list(overlap)[:5])
print("\nBackground sample:")
print(background.head().to_string(index=False))


Unique background locations: 374
Unique positive locations: 374
Exact rounded coordinate overlaps: 374
Example overlapping coordinates: [(32.1797, 77.1947), (30.0667, 79.0193), (31.4898, 76.7043), (31.1581, 77.7155), (32.5606, 76.1206)]

Background sample:
 latitude  longitude
28.866500  79.568200
29.158500  79.146400
29.295586  79.540431
29.330000  80.100000
29.337636  79.483545


In [23]:

import pandas as pd

files = [
    "../data/himalayan_landslide_v1_dataset.csv",
    "../data/multi_horizon_positive_definitions.csv",
    "../data/multi_horizon_negative_definitions.csv",
    "../data/negative_location_definitions_v4.csv",
]

for path in files:
    df = pd.read_csv(path)
    print(f"\n{path}")
    print("Rows:", len(df))
    if {"latitude", "longitude"}.issubset(df.columns):
        print(
            "Unique locations:",
            df[["latitude", "longitude"]].round(5).drop_duplicates().shape[0]
        )
        print(df[["latitude", "longitude"]].head(3).to_string(index=False))



../data/himalayan_landslide_v1_dataset.csv
Rows: 23018

../data/multi_horizon_positive_definitions.csv
Rows: 1125
Unique locations: 374
 latitude  longitude
  28.8665    79.5682
  28.8665    79.5682
  28.8665    79.5682

../data/multi_horizon_negative_definitions.csv
Rows: 477
Unique locations: 346
 latitude  longitude
 32.30165  77.934624
 29.84100  79.775800
 31.59860  78.438400

../data/negative_location_definitions_v4.csv
Rows: 1125
Unique locations: 264
 latitude  longitude
30.755482  78.817021
30.755482  78.817021
30.755482  78.817021


In [25]:

import pandas as pd

v1 = pd.read_csv("../data/himalayan_landslide_v1_dataset.csv")

print("Columns:", v1.columns.tolist())
print("\nClass distribution:")
print(v1["LANDSLIDE"].value_counts())

background_v1 = v1[v1["LANDSLIDE"] == 0][
    ["LATITUDE", "LONGITUDE"]
].drop_duplicates()

positive = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

v1_keys = set(zip(
    background_v1["LATITUDE"].round(5),
    background_v1["LONGITUDE"].round(5)
))
positive_keys = set(zip(
    positive["latitude"].round(5),
    positive["longitude"].round(5)
))

print("\nUnique V1 background locations:", len(background_v1))
print("Overlaps with positive locations:", len(v1_keys & positive_keys))
print("Independent candidate locations:", len(v1_keys - positive_keys))


Columns: ['LATITUDE', 'LONGITUDE', 'ELEVATION', 'SLOPE', 'RAINFALL', 'LANDCOVER_CODE', 'LANDSLIDE']

Class distribution:
LANDSLIDE
1    11509
0    11509
Name: count, dtype: int64

Unique V1 background locations: 11509
Overlaps with positive locations: 0
Independent candidate locations: 11509


In [26]:

import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

v1 = pd.read_csv("../data/himalayan_landslide_v1_dataset.csv")
positive = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

candidates = v1[v1["LANDSLIDE"] == 0][
    ["LATITUDE", "LONGITUDE"]
].drop_duplicates().copy()

events = positive[["latitude", "longitude"]].drop_duplicates()

tree = BallTree(
    np.radians(events[["latitude", "longitude"]].to_numpy()),
    metric="haversine"
)

distances, _ = tree.query(
    np.radians(candidates[["LATITUDE", "LONGITUDE"]].to_numpy()),
    k=1
)

candidates["nearest_event_km"] = distances[:, 0] * 6371

print("Candidate locations:", len(candidates))
print("Nearest-event distance (km):")
print(candidates["nearest_event_km"].describe().round(2))

for radius in [1, 3, 5, 10]:
    print(
        f"At least {radius} km from known events:",
        int((candidates["nearest_event_km"] >= radius).sum())
    )


Candidate locations: 11509
Nearest-event distance (km):
count    11509.00
mean        15.91
std         10.94
min          0.06
25%          7.67
50%         13.28
75%         21.73
max         66.03
Name: nearest_event_km, dtype: float64
At least 1 km from known events: 11448
At least 3 km from known events: 10933
At least 5 km from known events: 10035
At least 10 km from known events: 7361


In [27]:

# Save candidate locations for the V5 experiment
# Requires the `candidates` DataFrame from the previous cell.

v5_candidates = candidates.loc[
    candidates["nearest_event_km"] >= 5
].copy()

v5_candidates = v5_candidates.rename(columns={
    "LATITUDE": "latitude",
    "LONGITUDE": "longitude"
})

v5_candidates.to_csv(
    "../data/v5_candidate_controls.csv",
    index=False
)

print("V5 candidate controls:", len(v5_candidates))
print("Saved: ../data/v5_candidate_controls.csv")


V5 candidate controls: 10035
Saved: ../data/v5_candidate_controls.csv


In [28]:

import pandas as pd
import numpy as np

candidates = pd.read_csv("../data/v5_candidate_controls.csv")
positive = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
).dt.normalize()
positive["event_date"] = pd.to_datetime(
    positive["event_date"]
).dt.normalize()

print("Candidate locations:", len(candidates))
print("Positive reference dates:", positive["reference_date"].nunique())
print("Positive event dates:", positive["event_date"].nunique())
print("Positive horizon counts:")
print(positive["positive_horizon"].value_counts().sort_index())


Candidate locations: 10035
Positive reference dates: 477
Positive event dates: 212
Positive horizon counts:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64


In [29]:

import pandas as pd
import numpy as np

candidates = pd.read_csv("../data/v5_candidate_controls.csv")
positive = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

assert len(candidates) >= len(positive), "Not enough candidate locations"
assert positive["positive_horizon"].isin([1, 2, 3]).all()
assert candidates["nearest_event_km"].min() >= 5

# Reproducible selection without reusing candidate locations
rng = np.random.default_rng(42)
chosen_idx = rng.choice(len(candidates), size=len(positive), replace=False)
chosen = candidates.iloc[chosen_idx].reset_index(drop=True)

# Match every control to a positive sample's reference date and horizon
controls = pd.DataFrame({
    "control_id": np.arange(len(positive)),
    "latitude": chosen["latitude"].to_numpy(),
    "longitude": chosen["longitude"].to_numpy(),
    "reference_date": positive["reference_date"].to_numpy(),
    "positive_horizon": positive["positive_horizon"].to_numpy(),
    "nearest_event_km": chosen["nearest_event_km"].to_numpy(),
    "LANDSLIDE": 0
})

controls.to_csv(
    "../data/v5_negative_definitions.csv",
    index=False
)

print("V5 controls created:", len(controls))
print("Unique control locations:",
      controls[["latitude", "longitude"]].drop_duplicates().shape[0])
print("\nHorizon counts:")
print(controls["positive_horizon"].value_counts().sort_index())
print("\nSaved: ../data/v5_negative_definitions.csv")


V5 controls created: 1125
Unique control locations: 1125

Horizon counts:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64

Saved: ../data/v5_negative_definitions.csv


In [30]:

import pandas as pd

controls = pd.read_csv("../data/v5_negative_definitions.csv")
positive = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

for df in [controls, positive]:
    df["reference_date"] = pd.to_datetime(
        df["reference_date"]
    ).dt.normalize()

print("Control rows:", len(controls))
print("Unique control locations:",
      controls[["latitude", "longitude"]].drop_duplicates().shape[0])

print("\nReference dates match:",
      set(controls["reference_date"]) ==
      set(positive["reference_date"]))

print("\nHorizon counts:")
print(controls["positive_horizon"].value_counts().sort_index())

print("\nMinimum distance from known event locations (km):",
      round(controls["nearest_event_km"].min(), 2))

print("\nControls within 5 km of known event locations:",
      (controls["nearest_event_km"] < 5).sum())

print("\nMissing values:")
print(controls.isnull().sum())


Control rows: 1125
Unique control locations: 1125

Reference dates match: True

Horizon counts:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64

Minimum distance from known event locations (km): 5.03

Controls within 5 km of known event locations: 0

Missing values:
control_id          0
latitude            0
longitude           0
reference_date      0
positive_horizon    0
nearest_event_km    0
LANDSLIDE           0
dtype: int64


In [31]:

import pandas as pd

controls = pd.read_csv("../data/v5_negative_definitions.csv")
positive = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()
positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
).dt.normalize()
positive["event_date"] = pd.to_datetime(
    positive["event_date"]
).dt.normalize()

# Match coordinates at five-decimal precision
for df in [controls, positive]:
    df["lat_key"] = df["latitude"].round(5)
    df["lon_key"] = df["longitude"].round(5)

events = positive[
    ["lat_key", "lon_key", "event_date"]
].drop_duplicates()

matches = controls.merge(
    events,
    on=["lat_key", "lon_key"],
    how="inner"
)

matches["days_until_event"] = (
    matches["event_date"] - matches["reference_date"]
).dt.days

contaminated = matches[
    matches["days_until_event"].between(1, 3)
]

print("Controls checked:", len(controls))
print("Controls matching known event locations:",
      controls[["lat_key", "lon_key"]].drop_duplicates()
      .merge(events[["lat_key", "lon_key"]].drop_duplicates(),
             on=["lat_key", "lon_key"]).shape[0])
print("Controls with known events in next 1–3 days:",
      contaminated["control_id"].nunique())


Controls checked: 1125
Controls matching known event locations: 0
Controls with known events in next 1–3 days: 0


In [32]:

import ee
import pandas as pd
import os

# Reuse the existing authenticated Earth Engine session if available.
try:
    ee.Initialize(project="integral-server-484809-p2")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="integral-server-484809-p2")

controls = pd.read_csv("../data/v5_negative_definitions.csv")
controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()

output_path = "../data/v5_negative_features.csv"
progress_path = "../data/v5_negative_features_progress.csv"

dem = ee.Image("USGS/SRTMGL1_003")
slope = ee.Terrain.slope(dem)

worldcover = (
    ee.ImageCollection("ESA/WorldCover/v200")
    .first()
    .select("Map")
)

chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")


def extract_features(lat, lon, ref_date):
    point = ee.Geometry.Point([lon, lat])
    end = ee.Date(ref_date.strftime("%Y-%m-%d"))

    rainfall_images = []
    for days in [1, 3, 7, 15, 30]:
        rainfall = (
            chirps
            .filterDate(end.advance(-days, "day"), end)
            .sum()
            .rename(f"RAIN_{days}D")
        )
        rainfall_images.append(rainfall)

    rainfall_values = ee.Image.cat(
        rainfall_images
    ).reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=5000,
        maxPixels=1_000_000
    ).getInfo()

    static_image = ee.Image.cat([
        dem.rename("ELEVATION"),
        slope.rename("SLOPE"),
        worldcover.rename("LANDCOVER_CODE")
    ])

    static_values = static_image.reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=point,
        scale=30,
        maxPixels=1_000_000
    ).getInfo()

    return {
        "latitude": lat,
        "longitude": lon,
        "reference_date": ref_date.strftime("%Y-%m-%d"),
        **rainfall_values,
        **static_values
    }


# Resume from the V5 progress file only.
if os.path.exists(progress_path):
    completed = pd.read_csv(progress_path)
else:
    completed = pd.DataFrame()

key_cols = ["latitude", "longitude", "reference_date"]

if not completed.empty:
    completed["reference_date"] = pd.to_datetime(
        completed["reference_date"]
    ).dt.strftime("%Y-%m-%d")

    completed_keys = set(
        zip(
            completed["latitude"].round(6),
            completed["longitude"].round(6),
            completed["reference_date"]
        )
    )
else:
    completed_keys = set()

pending = []
for _, row in controls.iterrows():
    key = (
        round(row["latitude"], 6),
        round(row["longitude"], 6),
        row["reference_date"].strftime("%Y-%m-%d")
    )
    if key not in completed_keys:
        pending.append(row)

print("Total controls:", len(controls))
print("Already extracted:", len(completed))
print("Remaining:", len(pending))

for start in range(0, len(pending), 25):
    batch = pending[start:start + 25]
    results = []

    for row in batch:
        result = extract_features(
            float(row["latitude"]),
            float(row["longitude"]),
            row["reference_date"]
        )
        results.append(result)

    batch_df = pd.DataFrame(results)
    completed = pd.concat(
        [completed, batch_df],
        ignore_index=True
    ).drop_duplicates(subset=key_cols, keep="last")

    completed.to_csv(progress_path, index=False)
    print(f"Saved {len(completed)} / {len(controls)}")

completed.to_csv(output_path, index=False)

print("\nV5 extraction complete")
print("Rows:", len(completed))
print("Missing values:", int(completed.isna().sum().sum()))
print("Saved:", output_path)


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Total controls: 1125
Already extracted: 0
Remaining: 1125
Saved 25 / 1125
Saved 50 / 1125
Saved 75 / 1125
Saved 100 / 1125
Saved 125 / 1125
Saved 150 / 1125
Saved 175 / 1125
Saved 200 / 1125
Saved 225 / 1125
Saved 250 / 1125
Saved 275 / 1125
Saved 300 / 1125
Saved 325 / 1125
Saved 350 / 1125
Saved 375 / 1125
Saved 400 / 1125
Saved 425 / 1125
Saved 450 / 1125
Saved 475 / 1125
Saved 500 / 1125
Saved 525 / 1125
Saved 550 / 1125
Saved 575 / 1125
Saved 600 / 1125
Saved 625 / 1125
Saved 650 / 1125
Saved 675 / 1125
Saved 700 / 1125
Saved 725 / 1125
Saved 750 / 1125
Saved 775 / 1125
Saved 800 / 1125
Saved 825 / 1125
Saved 850 / 1125
Saved 875 / 1125
Saved 900 / 1125
Saved 925 / 1125
Saved 950 / 1125
Saved 975 / 1125
Saved 1000 / 1125
Saved 1025 / 1125
Saved 1050 / 1125
Saved 1075 / 1125
Saved 1100 / 1125
Saved 1125 / 1125

V5 extraction complete
Rows: 1125
Missing values: 0
Saved: ../data/v5_negative_features.csv


In [33]:

import pandas as pd

controls = pd.read_csv("../data/v5_negative_definitions.csv")
features = pd.read_csv("../data/v5_negative_features.csv")

print("Control definitions:", len(controls))
print("Extracted feature rows:", len(features))

keys = ["latitude", "longitude", "reference_date"]

for df in [controls, features]:
    df["reference_date"] = pd.to_datetime(
        df["reference_date"]
    ).dt.strftime("%Y-%m-%d")

definition_keys = set(map(tuple, controls[keys].round(
    {"latitude": 6, "longitude": 6}
).astype({"reference_date": str}).to_numpy()))

feature_keys = set(map(tuple, features[keys].round(
    {"latitude": 6, "longitude": 6}
).astype({"reference_date": str}).to_numpy()))

print("Missing feature records:", len(definition_keys - feature_keys))
print("Unexpected feature records:", len(feature_keys - definition_keys))
print("Duplicate feature rows:", features.duplicated(keys).sum())

print("\nFeature columns:")
print(features.columns.tolist())

print("\nMissing values:")
print(features.isna().sum())



Control definitions: 1125
Extracted feature rows: 1125
Missing feature records: 0
Unexpected feature records: 0
Duplicate feature rows: 0

Feature columns:
['latitude', 'longitude', 'reference_date', 'RAIN_15D', 'RAIN_1D', 'RAIN_30D', 'RAIN_3D', 'RAIN_7D', 'ELEVATION', 'LANDCOVER_CODE', 'SLOPE']

Missing values:
latitude          0
longitude         0
reference_date    0
RAIN_15D          0
RAIN_1D           0
RAIN_30D          0
RAIN_3D           0
RAIN_7D           0
ELEVATION         0
LANDCOVER_CODE    0
SLOPE             0
dtype: int64


In [35]:

files = {
    "positive_features": "../data/multi_horizon_positive_features_v2.csv",
    "positive_definitions": "../data/multi_horizon_positive_definitions.csv",
    "negative_features": "../data/v5_negative_features.csv",
    "negative_definitions": "../data/v5_negative_definitions.csv",
}

for name, path in files.items():
    temp = pd.read_csv(path)
    print(f"\n{name}:")
    print(temp.columns.tolist())



positive_features:
['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE']

positive_definitions:
['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'future_event_count']

negative_features:
['latitude', 'longitude', 'reference_date', 'RAIN_15D', 'RAIN_1D', 'RAIN_30D', 'RAIN_3D', 'RAIN_7D', 'ELEVATION', 'LANDCOVER_CODE', 'SLOPE']

negative_definitions:
['control_id', 'latitude', 'longitude', 'reference_date', 'positive_horizon', 'nearest_event_km', 'LANDSLIDE']


In [36]:

import pandas as pd

# Load existing files
positive = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)
negative_features = pd.read_csv(
    "../data/v5_negative_features.csv"
)
negative_definitions = pd.read_csv(
    "../data/v5_negative_definitions.csv"
)

keys = ["latitude", "longitude", "reference_date"]

# Normalize dates for reliable merging
for df in [positive, negative_features, negative_definitions]:
    df["reference_date"] = pd.to_datetime(
        df["reference_date"]
    ).dt.strftime("%Y-%m-%d")

# Positive features already contain their labels and horizons.
# Do not merge positive definitions again.
positive = positive.copy()

# Attach horizon metadata to the new negative features
negative = negative_features.merge(
    negative_definitions[
        keys + ["positive_horizon", "LANDSLIDE"]
    ],
    on=keys,
    how="inner",
    validate="one_to_one"
)

# Negative controls have no recorded event in their assigned horizon
for col in ["NEXT_1D", "NEXT_2D", "NEXT_3D"]:
    negative[col] = 0

common_cols = [
    "latitude", "longitude", "reference_date",
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE", "LANDSLIDE",
    "NEXT_1D", "NEXT_2D", "NEXT_3D"
]

# Ensure both datasets have the required columns
assert set(common_cols).issubset(positive.columns)
assert set(common_cols).issubset(negative.columns)

v5 = pd.concat(
    [positive[common_cols], negative[common_cols]],
    ignore_index=True
)

# Save as a new experimental dataset
v5.to_csv(
    "../data/himalayan_temporal_final_v5.csv",
    index=False
)

print("V5 shape:", v5.shape)
print("\nSource-class distribution:")
print(v5["LANDSLIDE"].value_counts().sort_index())

print("\nTarget distributions:")
for col in ["NEXT_1D", "NEXT_2D", "NEXT_3D"]:
    print(col, v5[col].value_counts().sort_index().to_dict())

print("\nMissing values:", int(v5.isna().sum().sum()))
print("Saved: ../data/himalayan_temporal_final_v5.csv")


V5 shape: (2250, 15)

Source-class distribution:
LANDSLIDE
0    1125
1    1125
Name: count, dtype: int64

Target distributions:
NEXT_1D {0: 1875, 1: 375}
NEXT_2D {0: 1500, 1: 750}
NEXT_3D {0: 1125, 1: 1125}

Missing values: 0
Saved: ../data/himalayan_temporal_final_v5.csv


In [37]:
import pandas as pd

v4 = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
v5 = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

# Compare control locations
v4_neg = set(zip(
    v4.loc[v4["LANDSLIDE"] == 0, "latitude"].round(5),
    v4.loc[v4["LANDSLIDE"] == 0, "longitude"].round(5)
))

v5_neg = set(zip(
    v5.loc[v5["LANDSLIDE"] == 0, "latitude"].round(5),
    v5.loc[v5["LANDSLIDE"] == 0, "longitude"].round(5)
))

print("V4 unique control locations:", len(v4_neg))
print("V5 unique control locations:", len(v5_neg))
print("V5 controls overlapping V4:", len(v4_neg & v5_neg))

# Verify target-label consistency
print("\nTarget labels by source class:")
print(v5.groupby("LANDSLIDE")[["NEXT_1D", "NEXT_2D", "NEXT_3D"]].sum())

# Check duplicate rows
print("\nDuplicate rows:", v5.duplicated().sum())

V4 unique control locations: 264
V5 unique control locations: 1125
V5 controls overlapping V4: 0

Target labels by source class:
           NEXT_1D  NEXT_2D  NEXT_3D
LANDSLIDE                           
0                0        0        0
1              375      750     1125

Duplicate rows: 0


In [38]:

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

features = [
    "latitude", "longitude",
    "RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]
target = "NEXT_1D"

def evaluate_dataset(path, name):
    df = pd.read_csv(path)
    df["reference_date"] = pd.to_datetime(df["reference_date"])

    train = df[df["reference_date"] < "2016-01-01"]
    test = df[df["reference_date"] >= "2016-01-01"]

    X_train, y_train = train[features], train[target]
    X_test, y_test = test[features], test[target]

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = model.predict(X_test)

    print(f"\n{'=' * 15} {name} {'=' * 15}")
    print("Train rows:", len(train))
    print("Test rows:", len(test))
    print("ROC-AUC:", round(roc_auc_score(y_test, probabilities), 4))
    print("PR-AUC:", round(average_precision_score(y_test, probabilities), 4))
    print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
    print("Precision:", round(precision_score(y_test, predictions, zero_division=0), 4))
    print("Recall:", round(recall_score(y_test, predictions, zero_division=0), 4))
    print("F1:", round(f1_score(y_test, predictions, zero_division=0), 4))

evaluate_dataset("../data/himalayan_temporal_final_v4.csv", "V4")
evaluate_dataset("../data/himalayan_temporal_final_v5.csv", "V5")



=============== V4 ===============
Train rows: 1812
Test rows: 438
ROC-AUC: 0.6305
PR-AUC: 0.2456
Accuracy: 0.7785
Precision: 0.26
Recall: 0.1781
F1: 0.2114

=============== V5 ===============
Train rows: 1812
Test rows: 438
ROC-AUC: 0.7085
PR-AUC: 0.2951
Accuracy: 0.7991
Precision: 0.3529
Recall: 0.2466
F1: 0.2903


In [39]:

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

train = df[df["reference_date"] < "2016-01-01"]
test = df[df["reference_date"] >= "2016-01-01"]

static_features = [
    "latitude", "longitude",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]

rainfall_features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"
]

target = "NEXT_1D"

def evaluate(features, name):
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    model.fit(train[features], train[target])
    probabilities = model.predict_proba(test[features])[:, 1]

    print(f"\n{name}")
    print("ROC-AUC:", round(roc_auc_score(test[target], probabilities), 4))
    print("PR-AUC:", round(average_precision_score(test[target], probabilities), 4))

evaluate(static_features, "V5 — Static features only")
evaluate(static_features + rainfall_features, "V5 — Static + rainfall")



V5 — Static features only
ROC-AUC: 0.7193
PR-AUC: 0.2978

V5 — Static + rainfall
ROC-AUC: 0.714
PR-AUC: 0.3061


In [40]:

import pandas as pd
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

train = df[df["reference_date"] < "2016-01-01"]

features = [
    "latitude", "longitude",
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(train[features], train["NEXT_1D"])

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(importance.to_string(index=False))


       Feature  Importance
     ELEVATION    0.157845
      RAIN_15D    0.134701
       RAIN_7D    0.115792
      RAIN_30D    0.114792
       RAIN_3D    0.105971
      latitude    0.098973
     longitude    0.094914
         SLOPE    0.072516
       RAIN_1D    0.068490
LANDCOVER_CODE    0.036006


In [41]:

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

features = [
    "latitude", "longitude",
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]

target = "NEXT_1D"

train = df[df["reference_date"] < "2016-01-01"]
test = df[df["reference_date"] >= "2016-01-01"]

X_train, y_train = train[features], train[target]
X_test, y_test = test[features], test[target]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)
model.fit(X_train, y_train)

result = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    "Feature": features,
    "Mean_AP_decrease": result.importances_mean,
    "Std": result.importances_std
}).sort_values("Mean_AP_decrease", ascending=False)

print("Test ROC-AUC:", round(
    roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 4
))
print("Test PR-AUC:", round(
    average_precision_score(y_test, model.predict_proba(X_test)[:, 1]), 4
))
print("\nPermutation importance (average precision):")
print(importance.to_string(index=False))


Test ROC-AUC: 0.7085
Test PR-AUC: 0.2951

Permutation importance (average precision):
       Feature  Mean_AP_decrease      Std
     ELEVATION          0.089554 0.024294
      latitude          0.031538 0.015312
      RAIN_15D          0.008598 0.015237
LANDCOVER_CODE          0.006887 0.025705
     longitude          0.005044 0.016150
       RAIN_1D          0.000897 0.010312
         SLOPE          0.000619 0.009700
       RAIN_3D         -0.007068 0.013935
       RAIN_7D         -0.009751 0.020927
      RAIN_30D         -0.014449 0.021849


In [42]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

features = [
    "latitude", "longitude",
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]
target = "NEXT_1D"

# Approximate 5 km grid cells; nearby samples share a group.
# This is a diagnostic grid, not an exact equal-area projection.
lat_km = df["latitude"] * 111.0
lon_km = (
    df["longitude"] * 111.0
    * np.cos(np.radians(df["latitude"]))
)
df["spatial_group"] = (
    np.floor(lat_km / 5).astype(int).astype(str)
    + "_"
    + np.floor(lon_km / 5).astype(int).astype(str)
)

X = df[features]
y = df[target]
groups = df["spatial_group"]

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
)
train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train spatial groups:", groups.iloc[train_idx].nunique())
print("Test spatial groups:", groups.iloc[test_idx].nunique())
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

if y_train.nunique() < 2 or y_test.nunique() < 2:
    print("Cannot calculate AUC: one split contains only one target class.")
else:
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]

    print("Spatial ROC-AUC:", round(
        roc_auc_score(y_test, probabilities), 4
    ))
    print("Spatial PR-AUC:", round(
        average_precision_score(y_test, probabilities), 4
    ))


Train rows: 1697
Test rows: 553
Train spatial groups: 915
Test spatial groups: 305
Train positive rate: 0.1679
Test positive rate: 0.1627
Spatial ROC-AUC: 0.6729
Spatial PR-AUC: 0.2625


In [43]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

static_features = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE"
]
rain_features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]
target = "NEXT_1D"

# Group nearby points into approximate 5 km grid cells
lat_km = df["latitude"] * 111.0
lon_km = (
    df["longitude"] * 111.0
    * np.cos(np.radians(df["latitude"]))
)
groups = (
    np.floor(lat_km / 5).astype(int).astype(str)
    + "_"
    + np.floor(lon_km / 5).astype(int).astype(str)
)

# Hold out complete spatial groups
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
)
train_idx, test_idx = next(
    splitter.split(df, df[target], groups=groups)
)

test = df.iloc[test_idx].copy()
train_candidates = df.iloc[train_idx].copy()

# Remove training points within 5 km of any test point
coords_test = np.radians(
    test[["latitude", "longitude"]].to_numpy()
)
coords_train = np.radians(
    train_candidates[["latitude", "longitude"]].to_numpy()
)

neighbors = NearestNeighbors(
    n_neighbors=1, algorithm="ball_tree", metric="haversine"
)
neighbors.fit(coords_test)

nearest_radians, _ = neighbors.kneighbors(coords_train)
nearest_km = nearest_radians[:, 0] * 6371.0

train = train_candidates.loc[nearest_km >= 5].copy()

print("Test rows:", len(test))
print("Training rows before buffer:", len(train_candidates))
print("Training rows after buffer:", len(train))
print("Training rows excluded:", len(train_candidates) - len(train))
print("Train positive rate:", round(train[target].mean(), 4))
print("Test positive rate:", round(test[target].mean(), 4))

def evaluate(features, name):
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    model.fit(train[features], train[target])
    probabilities = model.predict_proba(test[features])[:, 1]

    print(f"\n{name}")
    print("ROC-AUC:", round(
        roc_auc_score(test[target], probabilities), 4
    ))
    print("PR-AUC:", round(
        average_precision_score(test[target], probabilities), 4
    ))

if train[target].nunique() < 2 or test[target].nunique() < 2:
    print("Cannot evaluate: training or test data has only one target class.")
else:
    evaluate(static_features, "Spatial + 5 km buffer: Static only")
    evaluate(
        static_features + rain_features,
        "Spatial + 5 km buffer: Static + rainfall"
    )


Test rows: 553
Training rows before buffer: 1697
Training rows after buffer: 1416
Training rows excluded: 281
Train positive rate: 0.1667
Test positive rate: 0.1627

Spatial + 5 km buffer: Static only
ROC-AUC: 0.6508
PR-AUC: 0.2434

Spatial + 5 km buffer: Static + rainfall
ROC-AUC: 0.6578
PR-AUC: 0.2612


In [44]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

static_features = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE"
]
rain_features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]
target = "NEXT_1D"

# Approximate spatial grid groups
lat_km = df["latitude"] * 111.0
lon_km = (
    df["longitude"] * 111.0
    * np.cos(np.radians(df["latitude"]))
)
groups = (
    np.floor(lat_km / 5).astype(int).astype(str)
    + "_"
    + np.floor(lon_km / 5).astype(int).astype(str)
)

results = []

splitter = GroupShuffleSplit(
    n_splits=5, test_size=0.25, random_state=42
)

for split_no, (train_idx, test_idx) in enumerate(
    splitter.split(df, df[target], groups=groups), start=1
):
    test = df.iloc[test_idx].copy()
    train_candidates = df.iloc[train_idx].copy()

    test_coords = np.radians(
        test[["latitude", "longitude"]].to_numpy()
    )
    train_coords = np.radians(
        train_candidates[["latitude", "longitude"]].to_numpy()
    )

    nn = NearestNeighbors(
        n_neighbors=1, algorithm="ball_tree", metric="haversine"
    ).fit(test_coords)

    distances, _ = nn.kneighbors(train_coords)
    distances_km = distances[:, 0] * 6371.0

    train = train_candidates.loc[distances_km >= 5].copy()

    if train[target].nunique() < 2 or test[target].nunique() < 2:
        print(f"Split {split_no}: skipped; only one class present.")
        continue

    for name, features in [
        ("Static", static_features),
        ("Static + rainfall", static_features + rain_features)
    ]:
        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        )
        model.fit(train[features], train[target])
        probabilities = model.predict_proba(test[features])[:, 1]

        results.append({
            "Split": split_no,
            "Model": name,
            "Train_rows": len(train),
            "Test_rows": len(test),
            "ROC_AUC": roc_auc_score(test[target], probabilities),
            "PR_AUC": average_precision_score(test[target], probabilities)
        })

results_df = pd.DataFrame(results)

print("RESULTS BY SPLIT")
print(results_df.round(4).to_string(index=False))

print("\nMEAN AND STANDARD DEVIATION")
print(
    results_df.groupby("Model")[["ROC_AUC", "PR_AUC"]]
    .agg(["mean", "std"])
    .round(4)
)


RESULTS BY SPLIT
 Split             Model  Train_rows  Test_rows  ROC_AUC  PR_AUC
     1            Static        1416        553   0.6508  0.2434
     1 Static + rainfall        1416        553   0.6578  0.2612
     2            Static        1338        574   0.6939  0.2767
     2 Static + rainfall        1338        574   0.6798  0.2841
     3            Static        1452        533   0.6783  0.2594
     3 Static + rainfall        1452        533   0.6709  0.2508
     4            Static        1393        559   0.6849  0.2735
     4 Static + rainfall        1393        559   0.6797  0.3006
     5            Static        1366        577   0.6737  0.2769
     5 Static + rainfall        1366        577   0.6729  0.2723

MEAN AND STANDARD DEVIATION
                  ROC_AUC          PR_AUC        
                     mean     std    mean     std
Model                                            
Static             0.6763  0.0161  0.2660  0.0145
Static + rainfall  0.6722  0.0090  0.27

In [45]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

static = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE"
]
rainfall = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]
target = "NEXT_1D"

# Approximate 5 km grid groups
lat_km = df["latitude"] * 111.0
lon_km = (
    df["longitude"] * 111.0
    * np.cos(np.radians(df["latitude"]))
)
groups = (
    np.floor(lat_km / 5).astype(int).astype(str)
    + "_"
    + np.floor(lon_km / 5).astype(int).astype(str)
)

# Keep the same five spatial splits for every feature set
splits = list(GroupShuffleSplit(
    n_splits=5, test_size=0.25, random_state=42
).split(df, df[target], groups=groups))

feature_sets = {"Static only": static}

for rain in rainfall:
    feature_sets[f"Static + {rain}"] = static + [rain]

feature_sets["Static + all rainfall"] = static + rainfall

results = []

for split_no, (train_idx, test_idx) in enumerate(splits, start=1):
    test = df.iloc[test_idx].copy()
    candidates = df.iloc[train_idx].copy()

    test_coords = np.radians(
        test[["latitude", "longitude"]].to_numpy()
    )
    train_coords = np.radians(
        candidates[["latitude", "longitude"]].to_numpy()
    )

    nn = NearestNeighbors(
        n_neighbors=1, algorithm="ball_tree", metric="haversine"
    ).fit(test_coords)

    distances, _ = nn.kneighbors(train_coords)
    train = candidates.loc[
        distances[:, 0] * 6371.0 >= 5
    ].copy()

    if train[target].nunique() < 2 or test[target].nunique() < 2:
        continue

    for name, features in feature_sets.items():
        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        )
        model.fit(train[features], train[target])
        probabilities = model.predict_proba(test[features])[:, 1]

        results.append({
            "Split": split_no,
            "Feature_Set": name,
            "ROC_AUC": roc_auc_score(test[target], probabilities),
            "PR_AUC": average_precision_score(test[target], probabilities)
        })

results_df = pd.DataFrame(results)

summary = results_df.groupby("Feature_Set").agg(
    ROC_AUC_mean=("ROC_AUC", "mean"),
    ROC_AUC_std=("ROC_AUC", "std"),
    PR_AUC_mean=("PR_AUC", "mean"),
    PR_AUC_std=("PR_AUC", "std")
).sort_values("PR_AUC_mean", ascending=False)

print(summary.round(4).to_string())


                       ROC_AUC_mean  ROC_AUC_std  PR_AUC_mean  PR_AUC_std
Feature_Set                                                              
Static + RAIN_3D             0.6731       0.0176       0.2787      0.0236
Static + all rainfall        0.6722       0.0090       0.2738      0.0195
Static + RAIN_7D             0.6718       0.0162       0.2709      0.0165
Static + RAIN_1D             0.6687       0.0139       0.2667      0.0148
Static only                  0.6763       0.0161       0.2660      0.0145
Static + RAIN_15D            0.6695       0.0210       0.2641      0.0204
Static + RAIN_30D            0.6706       0.0192       0.2573      0.0209


In [46]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

static = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE"
]
target = "NEXT_1D"

feature_sets = {
    "Static only": static,
    "Static + RAIN_3D": static + ["RAIN_3D"]
}

lat_km = df["latitude"] * 111.0
lon_km = (
    df["longitude"] * 111.0
    * np.cos(np.radians(df["latitude"]))
)
groups = (
    np.floor(lat_km / 5).astype(int).astype(str)
    + "_"
    + np.floor(lon_km / 5).astype(int).astype(str)
)

splits = list(GroupShuffleSplit(
    n_splits=5, test_size=0.25, random_state=42
).split(df, df[target], groups=groups))

results = []

for split_no, (train_idx, test_idx) in enumerate(splits, start=1):
    test = df.iloc[test_idx].copy()
    candidates = df.iloc[train_idx].copy()

    test_coords = np.radians(
        test[["latitude", "longitude"]].to_numpy()
    )
    train_coords = np.radians(
        candidates[["latitude", "longitude"]].to_numpy()
    )

    nn = NearestNeighbors(
        n_neighbors=1, algorithm="ball_tree", metric="haversine"
    ).fit(test_coords)

    distances, _ = nn.kneighbors(train_coords)
    train = candidates.loc[
        distances[:, 0] * 6371.0 >= 5
    ].copy()

    if train[target].nunique() < 2 or test[target].nunique() < 2:
        print(f"Split {split_no}: skipped")
        continue

    for name, features in feature_sets.items():
        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        )
        model.fit(train[features], train[target])
        probabilities = model.predict_proba(test[features])[:, 1]

        results.append({
            "Split": split_no,
            "Model": name,
            "ROC_AUC": roc_auc_score(test[target], probabilities),
            "PR_AUC": average_precision_score(test[target], probabilities)
        })

results_df = pd.DataFrame(results)

print("RESULTS BY SPLIT")
print(results_df.round(4).to_string(index=False))

print("\nSUMMARY")
print(
    results_df.groupby("Model")[["ROC_AUC", "PR_AUC"]]
    .agg(["mean", "std"])
    .round(4)
)


RESULTS BY SPLIT
 Split            Model  ROC_AUC  PR_AUC
     1      Static only   0.6508  0.2434
     1 Static + RAIN_3D   0.6487  0.2498
     2      Static only   0.6939  0.2767
     2 Static + RAIN_3D   0.6973  0.3136
     3      Static only   0.6783  0.2594
     3 Static + RAIN_3D   0.6666  0.2697
     4      Static only   0.6849  0.2735
     4 Static + RAIN_3D   0.6772  0.2737
     5      Static only   0.6737  0.2769
     5 Static + RAIN_3D   0.6756  0.2869

SUMMARY
                 ROC_AUC          PR_AUC        
                    mean     std    mean     std
Model                                           
Static + RAIN_3D  0.6731  0.0176  0.2787  0.0236
Static only       0.6763  0.0161  0.2660  0.0145


In [47]:

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
    brier_score_loss
)

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

static = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE"
]

feature_sets = {
    "Static only": static,
    "Static + RAIN_3D": static + ["RAIN_3D"]
}

target = "NEXT_1D"

# Same spatial grouping and five splits as our previous experiment
lat_km = df["latitude"] * 111.0
lon_km = (
    df["longitude"] * 111.0
    * np.cos(np.radians(df["latitude"]))
)

groups = (
    np.floor(lat_km / 5).astype(int).astype(str)
    + "_"
    + np.floor(lon_km / 5).astype(int).astype(str)
)

splits = list(GroupShuffleSplit(
    n_splits=5, test_size=0.25, random_state=42
).split(df, df[target], groups=groups))

results = []

for split_no, (train_idx, test_idx) in enumerate(splits, start=1):
    test = df.iloc[test_idx].copy()
    candidates = df.iloc[train_idx].copy()

    test_coords = np.radians(
        test[["latitude", "longitude"]].to_numpy()
    )
    train_coords = np.radians(
        candidates[["latitude", "longitude"]].to_numpy()
    )

    nn = NearestNeighbors(
        n_neighbors=1,
        algorithm="ball_tree",
        metric="haversine"
    ).fit(test_coords)

    distances, _ = nn.kneighbors(train_coords)
    train = candidates.loc[
        distances[:, 0] * 6371.0 >= 5
    ].copy()

    y_test = test[target]

    for name, features in feature_sets.items():
        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        )

        model.fit(train[features], train[target])

        probabilities = model.predict_proba(
            test[features]
        )[:, 1]

        # Standard classification threshold for this diagnostic
        predictions = (probabilities >= 0.5).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_test, predictions, labels=[0, 1]
        ).ravel()

        results.append({
            "Split": split_no,
            "Model": name,
            "ROC_AUC": roc_auc_score(y_test, probabilities),
            "PR_AUC": average_precision_score(y_test, probabilities),
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1": f1_score(y_test, predictions, zero_division=0),
            "Specificity": tn / (tn + fp) if tn + fp else 0,
            "Balanced_Accuracy": balanced_accuracy_score(
                y_test, predictions
            ),
            "MCC": matthews_corrcoef(y_test, predictions),
            "Brier_Score": brier_score_loss(y_test, probabilities)
        })

results_df = pd.DataFrame(results)

metrics = [
    "ROC_AUC", "PR_AUC", "Accuracy", "Precision",
    "Recall", "F1", "Specificity", "Balanced_Accuracy",
    "MCC", "Brier_Score"
]

summary = results_df.groupby("Model")[metrics].agg(
    ["mean", "std"]
)

print(summary.round(4).to_string())


                 ROC_AUC          PR_AUC         Accuracy         Precision         Recall              F1         Specificity         Balanced_Accuracy             MCC         Brier_Score        
                    mean     std    mean     std     mean     std      mean    std    mean     std    mean     std        mean     std              mean     std    mean     std        mean     std
Model                                                                                                                                                                                               
Static + RAIN_3D  0.6731  0.0176  0.2787  0.0236   0.7913  0.0262    0.3219  0.013  0.2112  0.0849  0.2478  0.0497      0.9080  0.0445            0.5596  0.0206  0.1421  0.0238      0.1490  0.0074
Static only       0.6763  0.0161  0.2660  0.0145   0.7601  0.0199    0.2888  0.016  0.2939  0.0684  0.2892  0.0395      0.8538  0.0335            0.5739  0.0181  0.1465  0.0262      0.1561  0.0082


In [48]:

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_recall_curve

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

static = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE"
]
feature_sets = {
    "Static only": static,
    "Static + RAIN_3D": static + ["RAIN_3D"]
}
target = "NEXT_1D"

# Approximate 5 km spatial groups
lat_km = df["latitude"] * 111.0
lon_km = (
    df["longitude"] * 111.0
    * np.cos(np.radians(df["latitude"]))
)
groups = (
    np.floor(lat_km / 5).astype(int).astype(str)
    + "_"
    + np.floor(lon_km / 5).astype(int).astype(str)
)

splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=123
)
train_idx, val_idx = next(
    splitter.split(df, df[target], groups=groups)
)

val = df.iloc[val_idx].copy()
candidates = df.iloc[train_idx].copy()

# Exclude training points within 5 km of validation points
val_coords = np.radians(val[["latitude", "longitude"]].to_numpy())
train_coords = np.radians(
    candidates[["latitude", "longitude"]].to_numpy()
)

nn = NearestNeighbors(
    n_neighbors=1, algorithm="ball_tree", metric="haversine"
).fit(val_coords)

distances, _ = nn.kneighbors(train_coords)
train = candidates.loc[distances[:, 0] * 6371.0 >= 5].copy()

print("Training rows:", len(train))
print("Validation rows:", len(val))

for name, features in feature_sets.items():
    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    )
    model.fit(train[features], train[target])

    probabilities = model.predict_proba(val[features])[:, 1]
    precision, recall, thresholds = precision_recall_curve(
        val[target], probabilities
    )

    # Best F1 threshold
    f1 = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )
    best_idx = np.argmax(f1)

    print(f"\n{name}")
    print("Best F1 threshold:", round(thresholds[best_idx], 4))
    print("Precision at best F1:", round(precision[best_idx], 4))
    print("Recall at best F1:", round(recall[best_idx], 4))
    print("Best validation F1:", round(f1[best_idx], 4))

    # Highest recall among thresholds with precision >= 30%
    eligible = np.where(precision[:-1] >= 0.30)[0]
    if len(eligible):
        recall_idx = eligible[np.argmax(recall[eligible])]
        print("Threshold for max recall at precision >= 0.30:",
              round(thresholds[recall_idx], 4))
        print("Precision:", round(precision[recall_idx], 4))
        print("Recall:", round(recall[recall_idx], 4))
    else:
        print("No threshold reached 30% precision.")


Training rows: 1395
Validation rows: 534

Static only
Best F1 threshold: 0.4372
Precision at best F1: 0.2876
Recall at best F1: 0.5301
Best validation F1: 0.3729
Threshold for max recall at precision >= 0.30: 0.4836
Precision: 0.3009
Recall: 0.4096

Static + RAIN_3D
Best F1 threshold: 0.3593
Precision at best F1: 0.2599
Recall at best F1: 0.5542
Best validation F1: 0.3538
Threshold for max recall at precision >= 0.30: 0.4535
Precision: 0.3059
Recall: 0.3133


In [1]:

import pandas as pd

positive = pd.read_csv(
    "../data/multi_horizon_positive_definitions.csv"
)
positive_features = pd.read_csv(
    "../data/multi_horizon_positive_features_v2.csv"
)
negative = pd.read_csv(
    "../data/v5_negative_definitions.csv"
)
v5 = pd.read_csv(
    "../data/himalayan_temporal_final_v5.csv"
)

# 1. Check event dates against reference dates
positive["reference_date"] = pd.to_datetime(
    positive["reference_date"]
)
positive["event_date"] = pd.to_datetime(
    positive["event_date"]
)

positive["days_until_event"] = (
    positive["event_date"] - positive["reference_date"]
).dt.days

print("EVENT DATE OFFSETS")
print(positive["days_until_event"].value_counts().sort_index())

print("\nOFFSET BY POSITIVE HORIZON")
print(pd.crosstab(
    positive["positive_horizon"],
    positive["days_until_event"]
))

# 2. Check that the positive labels match the event dates
for col, horizon in [
    ("NEXT_1D", 1),
    ("NEXT_2D", 2),
    ("NEXT_3D", 3)
]:
    expected = (
        positive["days_until_event"] <= horizon
    ).astype(int)

    actual = positive_features[col].astype(int)

    print(
        f"{col}: mismatches =",
        int((expected.reset_index(drop=True) !=
             actual.reset_index(drop=True)).sum())
    )

# 3. Check control definitions
print("\nCONTROL DEFINITIONS")
print("Rows:", len(negative))
print("Unique control locations:", len(
    negative[["latitude", "longitude"]].drop_duplicates()
))
print("Missing values:\n", negative.isna().sum())

# 4. Confirm final V5 target distributions
print("\nV5 TARGET COUNTS")
for col in ["NEXT_1D", "NEXT_2D", "NEXT_3D"]:
    print(col, v5[col].value_counts().sort_index().to_dict())


EVENT DATE OFFSETS
days_until_event
1    375
2    375
3    375
Name: count, dtype: int64

OFFSET BY POSITIVE HORIZON
days_until_event    1    2    3
positive_horizon               
1                 375    0    0
2                   0  375    0
3                   0    0  375
NEXT_1D: mismatches = 0
NEXT_2D: mismatches = 0
NEXT_3D: mismatches = 0

CONTROL DEFINITIONS
Rows: 1125
Unique control locations: 1125
Missing values:
 control_id          0
latitude            0
longitude           0
reference_date      0
positive_horizon    0
nearest_event_km    0
LANDSLIDE           0
dtype: int64

V5 TARGET COUNTS
NEXT_1D {0: 1875, 1: 375}
NEXT_2D {0: 1500, 1: 750}
NEXT_3D {0: 1125, 1: 1125}


In [2]:

import pandas as pd

controls = pd.read_csv("../data/v5_negative_definitions.csv")
events = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

controls["reference_date"] = pd.to_datetime(
    controls["reference_date"]
).dt.normalize()
events["reference_date"] = pd.to_datetime(
    events["reference_date"]
).dt.normalize()

# Compare coordinates rounded to 5 decimal places
for df in [controls, events]:
    df["lat_key"] = df["latitude"].round(5)
    df["lon_key"] = df["longitude"].round(5)

event_keys = events[
    ["lat_key", "lon_key", "reference_date"]
].drop_duplicates()

matched = controls.merge(
    event_keys,
    on=["lat_key", "lon_key", "reference_date"],
    how="inner"
)

print("Total controls:", len(controls))
print("Controls matching known event locations on the same date:",
      len(matched))
print("Unique matched controls:",
      matched["control_id"].nunique() if len(matched) else 0)

# Check spatial distance to known event locations on the same date
from sklearn.neighbors import NearestNeighbors
import numpy as np

distances_km = []

for date, group in controls.groupby("reference_date"):
    same_date_events = events[events["reference_date"] == date]
    if same_date_events.empty:
        continue

    control_coords = np.radians(
        group[["latitude", "longitude"]].to_numpy()
    )
    event_coords = np.radians(
        same_date_events[["latitude", "longitude"]].to_numpy()
    )

    nn = NearestNeighbors(
        n_neighbors=1,
        algorithm="ball_tree",
        metric="haversine"
    ).fit(event_coords)

    distances, _ = nn.kneighbors(control_coords)
    distances_km.extend(distances[:, 0] * 6371.0)

if distances_km:
    print("Control-date comparisons:", len(distances_km))
    print("Nearest known event, median km:",
          round(float(np.median(distances_km)), 2))
    print("Nearest known event, minimum km:",
          round(float(np.min(distances_km)), 2))
else:
    print("No same-date event comparisons available.")


Total controls: 1125
Controls matching known event locations on the same date: 0
Unique matched controls: 0
Control-date comparisons: 1125
Nearest known event, median km: 108.15
Nearest known event, minimum km: 5.51


In [3]:

from pathlib import Path

data_dir = Path("../data")

for path in sorted(data_dir.iterdir()):
    if path.is_file():
        print(f"{path.name:50} {path.stat().st_size / 1024:.1f} KB")


background_locations.csv                           6.6 KB
background_locations_v2.csv                        17.4 KB
himalayan_event_rainfall.csv                       47.7 KB
himalayan_landslide_v1_dataset.csv                 1659.3 KB
himalayan_temporal_dataset.csv                     91.6 KB
himalayan_temporal_dataset_v2.csv                  103.0 KB
himalayan_temporal_final_v3.csv                    320.2 KB
himalayan_temporal_final_v4.csv                    321.7 KB
himalayan_temporal_final_v5.csv                    318.9 KB
himalayan_temporal_model_dataset_v1.csv            320.4 KB
HMA_LS_Cat_point_v02.0.zip                         820.3 KB
matched_negative_temporal.csv                      54.8 KB
multi_horizon_negative_definitions.csv             18.5 KB
multi_horizon_negative_definitions_audited.csv     26.2 KB
multi_horizon_negative_definitions_cleaned.csv     18.4 KB
multi_horizon_negative_definitions_v2.csv          23.5 KB
multi_horizon_negative_features_progress.csv     

In [4]:

import pandas as pd
from pathlib import Path
import zipfile

data_dir = Path("../data")

# Search CSV schemas for geology-related columns
print("GEOLOGY-RELATED COLUMNS\n")

for path in sorted(data_dir.glob("*.csv")):
    try:
        columns = pd.read_csv(path, nrows=0).columns.tolist()
        matches = [
            col for col in columns
            if any(word in col.lower()
                   for word in ["geology", "lithology", "rock", "soil"])
        ]
        if matches:
            print(path.name, "->", matches)
    except Exception as e:
        print("Could not inspect:", path.name, str(e))

# Inspect the contents of the existing landslide inventory ZIP
zip_path = data_dir / "HMA_LS_Cat_point_v02.0.zip"

print("\nINVENTORY ZIP CONTENTS\n")

if zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist()[:50]:
            print(name)
else:
    print("Inventory ZIP not found.")


GEOLOGY-RELATED COLUMNS


INVENTORY ZIP CONTENTS

HMA_LS_Cat_point_v02.0.cpg
HMA_LS_Cat_point_v02.0.dbf
HMA_LS_Cat_point_v02.0.prj
HMA_LS_Cat_point_v02.0.sbn
HMA_LS_Cat_point_v02.0.sbx
HMA_LS_Cat_point_v02.0.shp
HMA_LS_Cat_point_v02.0.shp.xml
HMA_LS_Cat_point_v02.0.shx
HMA_LS_Cat_point_v02.0.xml


In [5]:

import geopandas as gpd

path = "../data/HMA_LS_Cat_point_v02.0.zip"

inventory = gpd.read_file(f"zip://{path}")

print("Shape:", inventory.shape)
print("\nColumns:")
print(inventory.columns.tolist())

print("\nSample records:")
print(inventory.drop(columns="geometry", errors="ignore").head(5).to_string())

print("\nCRS:", inventory.crs)


Shape: (2801, 30)

Columns:
['src_name', 'src_link', 'ev_id', 'ev_date', 'ev_time', 'ev_title', 'ev_desc', 'loc_desc', 'loc_accu', 'ls_cat', 'ls_trig', 'ls_size', 'ls_setting', 'fatalities', 'injuries', 'storm_name', 'photo_link', 'comments', 'ev_imp_src', 'ev_imp_id', 'latitude', 'longitude', 'ctry_name', 'ctry_code', 'div_name', 'gaz_point', 'gaz_dist', 'sub_date', 'edit_date', 'geometry']

Sample records:
    src_name                                                                                               src_link  ev_id    ev_date ev_time                                             ev_title                                                                                                                                                                                                                                                         ev_desc                                             loc_desc loc_accu     ls_cat   ls_trig ls_size ls_setting  fatalities  injuries storm_name ph

In [6]:
import requests
from pathlib import Path

url = (
    "https://services8.arcgis.com/4KhTMTZ1x0f76DSg/"
    "ArcGIS/rest/services/GLiM_Niveau_I/FeatureServer/1/query"
)

params = {
    "where": "1=1",
    "geometry": "75.7,29.0,81.2,33.0",
    "geometryType": "esriGeometryEnvelope",
    "inSR": 4326,
    "spatialRel": "esriSpatialRelIntersects",
    "outFields": "*",
    "returnGeometry": "true",
    "outSR": 4326,
    "f": "geojson"
}

response = requests.get(url, params=params, timeout=120)
response.raise_for_status()
data = response.json()

if "features" not in data:
    raise ValueError(data)

output = Path("../data/himalayan_lithology.geojson")
output.parent.mkdir(parents=True, exist_ok=True)

output.write_text(
    response.text,
    encoding="utf-8"
)

print("Features downloaded:", len(data["features"]))
print("Saved to:", output.resolve())
print("File size (MB):", round(output.stat().st_size / 1_000_000, 2))

Features downloaded: 753
Saved to: C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\himalayan_lithology.geojson
File size (MB): 3.68


In [7]:
import geopandas as gpd

path = "../data/himalayan_lithology.geojson"
lithology = gpd.read_file(path)

print("Shape:", lithology.shape)
print("\nCRS:", lithology.crs)
print("\nColumns:")
print(lithology.columns.tolist())

print("\nSample attributes:")
print(
    lithology.drop(columns="geometry", errors="ignore")
    .head(5)
    .to_string(index=False)
)

print("\nGeometry types:")
print(lithology.geom_type.value_counts())

print("\nBounds:")
print(lithology.total_bounds)

Shape: (753, 12)

CRS: EPSG:4326

Columns:
['OBJECTID', 'IDENTITY_', 'Litho', 'xx', 'yy', 'zz', 'xx_Description', 'yy_Description', 'zz_Description', 'Shape__Area', 'Shape__Length', 'geometry']

Sample attributes:
 OBJECTID IDENTITY_  Litho xx yy zz                  xx_Description   yy_Description zz_Description  Shape__Area  Shape__Length
   994671  HIMAL450 pa____ pa __ __             Acid plutonic rocks             None           None 6.376024e+07   35196.195847
   994814  HIMAL594 vb____ vb __ __            Basic volcanic rocks             None           None 2.280923e+07   27354.153135
   994920  HIMAL941 pa____ pa __ __             Acid plutonic rocks             None           None 1.926458e+07   20290.556656
   994780  HIMAL560 scpu__ sc pu __     Carbonate sedimentary rocks (Pure) carbonate           None 1.250851e+07   20566.260754
   994734  HIMAL514 ssmx__ ss mx __ Siliciclastic sedimentary rocks Mixed grain size           None 7.026297e+07   52921.031709

Geometry types:
P

In [8]:
print("Lithology categories:")
print(lithology["xx_Description"].value_counts(dropna=False))

print("\nMissing values:")
print(lithology[["Litho", "xx_Description"]].isna().sum())

print("\nGeometry validity:")
print("Invalid geometries:", (~lithology.geometry.is_valid).sum())

print("\nTotal polygons:", len(lithology))

Lithology categories:
xx_Description
Mixed sedimentary rocks            197
Siliciclastic sedimentary rocks    171
Carbonate sedimentary rocks        134
Acid plutonic rocks                 94
Metamorphics                        60
Basic volcanic rocks                32
Basic plutonic rocks                27
Unconsolidated sediments            23
Intermediate volcanic rocks          5
Acid volcanic rocks                  3
Ice and Glaciers                     3
Intermediate plutonic rocks          2
Water Bodies                         1
No Data                              1
Name: count, dtype: int64

Missing values:
Litho             0
xx_Description    0
dtype: int64

Geometry validity:
Invalid geometries: 6

Total polygons: 753


In [10]:
from pathlib import Path

print("Current directory:", Path.cwd())

path = Path("../data/himalyan_lithology.geojson")
print("Expected path:", path.resolve())
print("File exists:", path.exists())

Current directory: c:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\notebooks
Expected path: C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\himalyan_lithology.geojson
File exists: False


In [11]:
import geopandas as gpd

path = r"C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\himalayan_lithology.geojson"

lithology = gpd.read_file(path)
print("Loaded successfully:", lithology.shape)

Loaded successfully: (753, 12)


In [12]:
import geopandas as gpd
import pandas as pd

lithology = gpd.read_file(
    r"C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\himalayan_lithology.geojson"
)

lithology["geometry"] = lithology.geometry.make_valid()

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

result = gpd.sjoin(
    points,
    lithology[["Litho", "xx_Description", "geometry"]],
    how="left",
    predicate="intersects"
)

# A point can intersect overlapping polygons, so count unique samples
matched = result.drop_duplicates(
    subset=["latitude", "longitude", "reference_date"]
)

print("Total samples:", len(df))
print("Matched:", matched["Litho"].notna().sum())
print("Unmatched:", matched["Litho"].isna().sum())

print("\nCoverage by class (LANDSLIDE: 0=control, 1=positive):")
print(matched.groupby("LANDSLIDE")["Litho"].agg(
    matched=lambda s: s.notna().sum(),
    total="size"
))

print("\nLithology categories assigned:")
print(matched["xx_Description"].value_counts(dropna=False))

Total samples: 2250
Matched: 2250
Unmatched: 0

Coverage by class (LANDSLIDE: 0=control, 1=positive):
           matched  total
LANDSLIDE                
0             1125   1125
1             1125   1125

Lithology categories assigned:
xx_Description
Metamorphics                       658
Siliciclastic sedimentary rocks    531
Mixed sedimentary rocks            402
Carbonate sedimentary rocks        272
Acid plutonic rocks                161
Unconsolidated sediments           154
Basic volcanic rocks                49
Intermediate volcanic rocks         22
Water Bodies                         1
Name: count, dtype: int64


In [13]:
import geopandas as gpd
import pandas as pd

lithology = gpd.read_file(
    r"C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\himalayan_lithology.geojson"
)
lithology["geometry"] = lithology.geometry.make_valid()

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

points = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
)

joined = gpd.sjoin(
    points,
    lithology[["Litho", "xx_Description", "geometry"]],
    how="left",
    predicate="intersects"
)

matches_per_sample = joined.groupby(level=0).size()

print("Samples:", len(df))
print("Samples with exactly one match:", (matches_per_sample == 1).sum())
print("Samples with multiple matches:", (matches_per_sample > 1).sum())
print("Maximum matches for one sample:", matches_per_sample.max())

Samples: 2250
Samples with exactly one match: 2250
Samples with multiple matches: 0
Maximum matches for one sample: 1


In [14]:
import geopandas as gpd
import pandas as pd

# Load original V5 dataset
df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

# Load lithology polygons
lithology = gpd.read_file(
    r"C:\Users\imman\OneDrive\Desktop\Project\ML Project\himalyan-landslide-prediction\data\himalayan_lithology.geojson"
)
lithology["geometry"] = lithology.geometry.make_valid()

# Convert sample coordinates to points
points = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
)

# Assign lithology
joined = gpd.sjoin(
    points,
    lithology[["Litho", "xx_Description", "geometry"]],
    how="left",
    predicate="intersects"
)

# Retain original columns and add geology features
df["LITHO_CODE"] = joined["Litho"].to_numpy()
df["LITHOLOGY"] = joined["xx_Description"].to_numpy()

# Save as a new experimental dataset
output = "../data/himalayan_temporal_final_v5_lithology.csv"
df.to_csv(output, index=False)

print("Saved:", output)
print("Shape:", df.shape)
print("\nMissing values:")
print(df[["LITHO_CODE", "LITHOLOGY"]].isna().sum())
print("\nLithology distribution:")
print(df["LITHOLOGY"].value_counts())
print("\nClass balance:")
print(df["LANDSLIDE"].value_counts())

Saved: ../data/himalayan_temporal_final_v5_lithology.csv
Shape: (2250, 17)

Missing values:
LITHO_CODE    0
LITHOLOGY     0
dtype: int64

Lithology distribution:
LITHOLOGY
Metamorphics                       658
Siliciclastic sedimentary rocks    531
Mixed sedimentary rocks            402
Carbonate sedimentary rocks        272
Acid plutonic rocks                161
Unconsolidated sediments           154
Basic volcanic rocks                49
Intermediate volcanic rocks         22
Water Bodies                         1
Name: count, dtype: int64

Class balance:
LANDSLIDE
1    1125
0    1125
Name: count, dtype: int64


In [15]:
import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# Load the separate lithology experiment
df = pd.read_csv("../data/himalayan_temporal_final_v5_lithology.csv")

# Target: landslide event within the next day
y = df["NEXT_1D"].astype(int)

# Compare the same numeric features, with and without lithology
numeric = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE", "RAIN_3D"
]
with_lithology = numeric + ["LITHOLOGY"]

# Create approximate 5 km spatial groups
lat_km = df["latitude"] * 111.32
lon_km = (
    df["longitude"] * 111.32
    * np.cos(np.radians(df["latitude"]))
)
groups = (
    np.floor(lat_km / 5).astype(str) + "_"
    + np.floor(lon_km / 5).astype(str)
)

# Five identical spatial splits for both models
splitter = GroupShuffleSplit(
    n_splits=5, test_size=0.25, random_state=42
)

results = []

for split_no, (train_idx, test_idx) in enumerate(
    splitter.split(df, y, groups), start=1
):
    # Exclude training samples within 5 km of test samples
    coords = np.radians(
        df[["latitude", "longitude"]].to_numpy()
    )

    nn = NearestNeighbors(
        n_neighbors=1, metric="haversine"
    )
    nn.fit(coords[test_idx])
    distances, _ = nn.kneighbors(coords[train_idx])

    train_idx = train_idx[
        distances[:, 0] * 6371 >= 5
    ]

    print(
        f"Split {split_no}: "
        f"train={len(train_idx)}, test={len(test_idx)}"
    )

    for name, features in [
        ("Baseline", numeric),
        ("With lithology", with_lithology)
    ]:
        preprocessor = ColumnTransformer([
            ("numeric", "passthrough", numeric),
            ("lithology", OneHotEncoder(
                handle_unknown="ignore"
            ), ["LITHOLOGY"])
        ]) if name == "With lithology" else "passthrough"

        model = make_pipeline(
            preprocessor,
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight="balanced"
            )
        )

        model.fit(df.iloc[train_idx][features], y.iloc[train_idx])

        probabilities = model.predict_proba(
            df.iloc[test_idx][features]
        )[:, 1]

        results.append({
            "Split": split_no,
            "Model": name,
            "ROC_AUC": roc_auc_score(
                y.iloc[test_idx], probabilities
            ),
            "PR_AUC": average_precision_score(
                y.iloc[test_idx], probabilities
            )
        })

results_df = pd.DataFrame(results)

print("\nResults by split:")
print(results_df.round(4).to_string(index=False))

print("\nMean and standard deviation:")
print(
    results_df.groupby("Model")[["ROC_AUC", "PR_AUC"]]
    .agg(["mean", "std"]).round(4)
)

Split 1: train=1410, test=600
Split 2: train=1395, test=555
Split 3: train=1362, test=596
Split 4: train=1437, test=528
Split 5: train=1431, test=561

Results by split:
 Split          Model  ROC_AUC  PR_AUC
     1       Baseline   0.6748  0.2961
     1 With lithology   0.6794  0.3022
     2       Baseline   0.6965  0.2746
     2 With lithology   0.6969  0.2677
     3       Baseline   0.6999  0.3159
     3 With lithology   0.6851  0.2926
     4       Baseline   0.6927  0.2660
     4 With lithology   0.7053  0.2807
     5       Baseline   0.6829  0.2619
     5 With lithology   0.6889  0.2615

Mean and standard deviation:
               ROC_AUC          PR_AUC        
                  mean     std    mean     std
Model                                         
Baseline        0.6893  0.0103  0.2829  0.0227
With lithology  0.6911  0.0101  0.2809  0.0169


In [16]:
import pandas as pd

v4 = pd.read_csv("../data/himalayan_temporal_final_v4.csv")
v5 = pd.read_csv("../data/himalayan_temporal_final_v5.csv")

print("V4 shape:", v4.shape)
print("V5 shape:", v5.shape)

# Check whether the positive samples are unchanged
positive_v4 = v4[v4["LANDSLIDE"] == 1].sort_values(
    ["latitude", "longitude", "reference_date"]
).reset_index(drop=True)

positive_v5 = v5[v5["LANDSLIDE"] == 1].sort_values(
    ["latitude", "longitude", "reference_date"]
).reset_index(drop=True)

print("\nPositive samples:")
print("V4:", len(positive_v4))
print("V5:", len(positive_v5))
print(
    "Identical positive samples:",
    positive_v4.equals(positive_v5)
)

# Compare negative-control locations
controls_v4 = set(zip(
    v4.loc[v4["LANDSLIDE"] == 0, "latitude"].round(6),
    v4.loc[v4["LANDSLIDE"] == 0, "longitude"].round(6),
    v4.loc[v4["LANDSLIDE"] == 0, "reference_date"].astype(str)
))

controls_v5 = set(zip(
    v5.loc[v5["LANDSLIDE"] == 0, "latitude"].round(6),
    v5.loc[v5["LANDSLIDE"] == 0, "longitude"].round(6),
    v5.loc[v5["LANDSLIDE"] == 0, "reference_date"].astype(str)
))

print("\nNegative-control locations:")
print("Shared:", len(controls_v4 & controls_v5))
print("Only in V4:", len(controls_v4 - controls_v5))
print("Only in V5:", len(controls_v5 - controls_v4))

# Compare target distributions
for name, df in [("V4", v4), ("V5", v5)]:
    print(f"\n{name} NEXT_1D distribution:")
    print(df["NEXT_1D"].value_counts().sort_index())

V4 shape: (2250, 15)
V5 shape: (2250, 15)

Positive samples:
V4: 1125
V5: 1125
Identical positive samples: True

Negative-control locations:
Shared: 0
Only in V4: 1125
Only in V5: 1125

V4 NEXT_1D distribution:
NEXT_1D
0    1875
1     375
Name: count, dtype: int64

V5 NEXT_1D distribution:
NEXT_1D
0    1875
1     375
Name: count, dtype: int64


In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

datasets = {
    "V4": pd.read_csv("../data/himalayan_temporal_final_v4.csv"),
    "V5": pd.read_csv("../data/himalayan_temporal_final_v5.csv")
}

features = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE", "RAIN_3D"
]

results = []

for version, df in datasets.items():
    X = df[features]
    y = df["NEXT_1D"].astype(int)

    # Same approximate 5 km spatial grouping method
    lat_km = df["latitude"] * 111.32
    lon_km = (
        df["longitude"] * 111.32
        * np.cos(np.radians(df["latitude"]))
    )
    groups = (
        np.floor(lat_km / 5).astype(str) + "_"
        + np.floor(lon_km / 5).astype(str)
    )

    splitter = GroupShuffleSplit(
        n_splits=5, test_size=0.25, random_state=42
    )

    coords = np.radians(
        df[["latitude", "longitude"]].to_numpy()
    )

    for split_no, (train_idx, test_idx) in enumerate(
        splitter.split(X, y, groups), start=1
    ):
        # Remove training points within 5 km of test points
        nn = NearestNeighbors(n_neighbors=1, metric="haversine")
        nn.fit(coords[test_idx])
        distances, _ = nn.kneighbors(coords[train_idx])

        train_idx = train_idx[distances[:, 0] * 6371 >= 5]

        model = RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])

        probabilities = model.predict_proba(X.iloc[test_idx])[:, 1]

        results.append({
            "Version": version,
            "Split": split_no,
            "Train": len(train_idx),
            "Test": len(test_idx),
            "ROC_AUC": roc_auc_score(y.iloc[test_idx], probabilities),
            "PR_AUC": average_precision_score(y.iloc[test_idx], probabilities)
        })

results_df = pd.DataFrame(results)

print("Results by split:")
print(results_df.round(4).to_string(index=False))

print("\nMean and standard deviation:")
print(
    results_df.groupby("Version")[["ROC_AUC", "PR_AUC"]]
    .agg(["mean", "std"]).round(4)
)

Results by split:
Version  Split  Train  Test  ROC_AUC  PR_AUC
     V4      1   1125   624   0.5268  0.1926
     V4      2   1236   552   0.5034  0.1876
     V4      3    921   696   0.5319  0.1817
     V4      4   1143   540   0.5264  0.1845
     V4      5   1002   672   0.5515  0.2001
     V5      1   1410   600   0.6748  0.2961
     V5      2   1395   555   0.6965  0.2746
     V5      3   1362   596   0.6999  0.3159
     V5      4   1437   528   0.6927  0.2660
     V5      5   1431   561   0.6829  0.2619

Mean and standard deviation:
        ROC_AUC          PR_AUC        
           mean     std    mean     std
Version                                
V4       0.5280  0.0171  0.1893  0.0073
V5       0.6893  0.0103  0.2829  0.0227


In [19]:
import pandas as pd
import numpy as np

controls = pd.read_csv("../data/v5_negative_definitions.csv")
events = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

controls["reference_date"] = pd.to_datetime(controls["reference_date"])
events["event_date"] = pd.to_datetime(events["event_date"])

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lon1 = np.radians(np.asarray(lon1, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    a = np.clip(a, 0, 1)
    return 6371 * 2 * np.arcsin(np.sqrt(a))

nearest_distances = []
events_in_windows = []

for _, control in controls.iterrows():
    start = control["reference_date"]
    horizon = int(control["positive_horizon"])
    end = start + pd.Timedelta(days=horizon)

    future_events = events[
        (events["event_date"] > start)
        & (events["event_date"] <= end)
    ]

    events_in_windows.append(len(future_events))

    if future_events.empty:
        nearest_distances.append(np.nan)
        continue

    distances = haversine_km(
        control["latitude"],
        control["longitude"],
        future_events["latitude"].to_numpy(),
        future_events["longitude"].to_numpy()
    )

    nearest_distances.append(float(np.min(distances)))

controls["future_event_count"] = events_in_windows
controls["nearest_future_event_km"] = nearest_distances

print("Total controls:", len(controls))
print("Controls with no recorded events in their time window:",
      (controls["future_event_count"] == 0).sum())
print("Controls with a known event within 5 km:",
      (controls["nearest_future_event_km"] <= 5).sum())
print("Controls with a known event within 10 km:",
      (controls["nearest_future_event_km"] <= 10).sum())
print("Controls with a known event within 25 km:",
      (controls["nearest_future_event_km"] <= 25).sum())

print("\nNearest-event distance summary (km):")
print(controls["nearest_future_event_km"].describe())

print("\nResults by horizon:")
print(
    controls.groupby("positive_horizon")[
        ["future_event_count", "nearest_future_event_km"]
    ].agg(["count", "median", "min"]).round(2)
)

Total controls: 1125
Controls with no recorded events in their time window: 0
Controls with a known event within 5 km: 0
Controls with a known event within 10 km: 12
Controls with a known event within 25 km: 59

Nearest-event distance summary (km):
count    1125.000000
mean      141.639429
std       100.621083
min         5.506159
25%        62.333947
50%       118.453760
75%       203.370712
max       527.390210
Name: nearest_future_event_km, dtype: float64

Results by horizon:
                 future_event_count            nearest_future_event_km  \
                              count median min                   count   
positive_horizon                                                         
1                               375    6.0   3                     375   
2                               375    9.0   3                     375   
3                               375    9.0   3                     375   

                                
                  median   min  
posit

In [20]:
cols = [
    "latitude",
    "longitude",
    "reference_date",
    "positive_horizon",
    "nearest_future_event_km"
]

print(
    controls.loc[
        controls["nearest_future_event_km"] <= 10,
        cols
    ]
    .sort_values("nearest_future_event_km")
    .round(4)
    .to_string(index=False)
)

 latitude  longitude reference_date  positive_horizon  nearest_future_event_km
  30.2073    78.3218     2016-07-16                 1                   5.5062
  30.7251    77.2270     2010-09-16                 3                   5.5254
  30.1992    78.6160     2010-09-17                 3                   6.1570
  30.9429    79.5749     2010-09-17                 2                   6.1617
  30.8687    78.2662     2018-08-06                 2                   6.1700
  29.8689    80.4442     2013-06-15                 1                   8.2490
  30.5635    78.4306     2013-06-15                 1                   8.6033
  30.5916    78.9641     2010-08-17                 3                   8.8850
  32.0408    77.2569     2010-09-16                 3                   8.9758
  32.3728    76.5853     2018-08-31                 2                   8.9993
  29.5357    79.4561     2010-09-17                 2                   9.3552
  30.9868    78.7957     2010-09-18                 

In [22]:
import pandas as pd
import numpy as np

controls_close = controls[
    controls["nearest_future_event_km"] <= 10
].copy()

events = pd.read_csv(
    "../data/multi_horizon_positive_definitions.csv"
)
events["event_date"] = pd.to_datetime(events["event_date"])

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(float(lat1))
    lon1 = np.radians(float(lon1))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    a = np.clip(a, 0, 1)
    return 6371 * 2 * np.arcsin(np.sqrt(a))
rows = []

for _, c in controls_close.iterrows():
    start = c["reference_date"]
    end = start + pd.Timedelta(days=int(c["positive_horizon"]))

    candidates = events[
        (events["event_date"] > start)
        & (events["event_date"] <= end)
    ].copy()

    if candidates.empty:
        continue

    candidates["distance_km"] = haversine_km(
        c["latitude"], c["longitude"],
        candidates["latitude"].to_numpy(),
        candidates["longitude"].to_numpy()
    )

    nearest = candidates.loc[candidates["distance_km"].idxmin()]

    rows.append({
        "control_lat": c["latitude"],
        "control_lon": c["longitude"],
        "reference_date": start.date(),
        "horizon_days": int(c["positive_horizon"]),
        "event_date": nearest["event_date"].date(),
        "event_lat": nearest["latitude"],
        "event_lon": nearest["longitude"],
        "distance_km": nearest["distance_km"]
    })

print(pd.DataFrame(rows).round(4).to_string(index=False))

 control_lat  control_lon reference_date  horizon_days event_date  event_lat  event_lon  distance_km
     30.9429      79.5749     2010-09-17             2 2010-09-19    30.9013    79.5322       6.1617
     30.2073      78.3218     2016-07-16             1 2016-07-17    30.1642    78.2936       5.5062
     30.5916      78.9641     2010-08-17             3 2010-08-20    30.6509    79.0264       8.8850
     29.8689      80.4442     2013-06-15             1 2013-06-16    29.7986    80.4170       8.2490
     29.5357      79.4561     2010-09-17             2 2010-09-19    29.6131    79.4940       9.3552
     30.7251      77.2270     2010-09-16             3 2010-09-19    30.7198    77.1695       5.5254
     32.3728      76.5853     2018-08-31             2 2018-09-02    32.4416    76.5347       8.9993
     30.9868      78.7957     2010-09-18             1 2010-09-19    30.9015    78.7756       9.6784
     32.0408      77.2569     2010-09-16             3 2010-09-19    31.9843    77.1889    

In [23]:
import pandas as pd

controls = pd.read_csv("../data/v5_negative_definitions.csv")
positives = pd.read_csv("../data/multi_horizon_positive_definitions.csv")

keys = ["latitude", "longitude", "reference_date"]

controls["reference_date"] = pd.to_datetime(controls["reference_date"]).dt.strftime("%Y-%m-%d")
positives["reference_date"] = pd.to_datetime(positives["reference_date"]).dt.strftime("%Y-%m-%d")

control_keys = set(zip(
    controls["latitude"].round(5),
    controls["longitude"].round(5),
    controls["reference_date"]
))

positive_keys = set(zip(
    positives["latitude"].round(5),
    positives["longitude"].round(5),
    positives["reference_date"]
))

print("Exact same location/date conflicts:",
      len(control_keys & positive_keys))

print("Unique controls:", len(control_keys))
print("Unique positive location/date samples:", len(positive_keys))

Exact same location/date conflicts: 0
Unique controls: 1125
Unique positive location/date samples: 1125


In [24]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

df = pd.read_csv("../data/himalayan_temporal_final_v5_lithology.csv")

numeric = [
    "latitude", "longitude", "ELEVATION",
    "SLOPE", "LANDCOVER_CODE", "RAIN_3D"
]
all_features = numeric + ["LITHOLOGY"]
y = df["NEXT_1D"].astype(int)

# Same approximate 5 km spatial grouping
lat_km = df["latitude"] * 111.32
lon_km = (
    df["longitude"] * 111.32
    * np.cos(np.radians(df["latitude"]))
)
groups = (
    np.floor(lat_km / 5).astype(str) + "_"
    + np.floor(lon_km / 5).astype(str)
)

coords = np.radians(df[["latitude", "longitude"]].to_numpy())
splitter = GroupShuffleSplit(
    n_splits=5, test_size=0.25, random_state=42
)

results = []

for split_no, (train_idx, test_idx) in enumerate(
    splitter.split(df, y, groups), start=1
):
    # Remove training points within 5 km of test points
    nn = NearestNeighbors(n_neighbors=1, metric="haversine")
    nn.fit(coords[test_idx])
    distances, _ = nn.kneighbors(coords[train_idx])
    train_idx = train_idx[distances[:, 0] * 6371 >= 5]

    for name, features in [
        ("Baseline", numeric),
        ("With lithology", all_features)
    ]:
        if name == "With lithology":
            preprocess = ColumnTransformer([
                ("numeric", "passthrough", numeric),
                ("lithology", OneHotEncoder(
                    handle_unknown="ignore"
                ), ["LITHOLOGY"])
            ])
        else:
            preprocess = "passthrough"

        model = make_pipeline(
            preprocess,
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight="balanced"
            )
        )

        model.fit(df.iloc[train_idx][features], y.iloc[train_idx])

        X_test = df.iloc[test_idx][features]
        y_test = y.iloc[test_idx]
        predictions = model.predict(X_test)  # threshold = 0.5
        probabilities = model.predict_proba(X_test)[:, 1]

        results.append({
            "Split": split_no,
            "Model": name,
            "Accuracy": accuracy_score(y_test, predictions),
            "Precision": precision_score(
                y_test, predictions, zero_division=0
            ),
            "Recall": recall_score(
                y_test, predictions, zero_division=0
            ),
            "F1": f1_score(
                y_test, predictions, zero_division=0
            ),
            "ROC_AUC": roc_auc_score(y_test, probabilities),
            "PR_AUC": average_precision_score(
                y_test, probabilities
            )
        })

results_df = pd.DataFrame(results)

print("METRICS FOR EACH SPLIT")
print(results_df.round(4).to_string(index=False))

metric_cols = [
    "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"
]

print("\nMEAN METRICS ACROSS FIVE SPLITS")
print(results_df.groupby("Model")[metric_cols].mean().round(4))

print("\nSTANDARD DEVIATION ACROSS FIVE SPLITS")
print(results_df.groupby("Model")[metric_cols].std().round(4))

METRICS FOR EACH SPLIT
 Split          Model  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC
     1       Baseline    0.7667     0.3067  0.2072 0.2473   0.6748  0.2961
     1 With lithology    0.7650     0.3256  0.2523 0.2843   0.6794  0.3022
     2       Baseline    0.7910     0.3276  0.1979 0.2468   0.6965  0.2746
     2 With lithology    0.7694     0.2778  0.2083 0.2381   0.6969  0.2677
     3       Baseline    0.7752     0.3511  0.3113 0.3300   0.6999  0.3159
     3 With lithology    0.7768     0.3151  0.2170 0.2570   0.6851  0.2926
     4       Baseline    0.8201     0.3396  0.2308 0.2748   0.6927  0.2660
     4 With lithology    0.8106     0.2963  0.2051 0.2424   0.7053  0.2807
     5       Baseline    0.7807     0.2639  0.2135 0.2360   0.6829  0.2619
     5 With lithology    0.7825     0.2857  0.2472 0.2651   0.6889  0.2615

MEAN METRICS ACROSS FIVE SPLITS
                Accuracy  Precision  Recall      F1  ROC_AUC  PR_AUC
Model                                             

In [25]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")
y = df["NEXT_1D"].astype(int)

rain = ["RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"]
terrain = ["ELEVATION", "SLOPE", "LANDCOVER_CODE"]

experiments = {
    "Rainfall only": rain,
    "Terrain only": terrain,
    "Rainfall + terrain": rain + terrain
}

# Create consistent approximate 5 km spatial groups
lat_km = df["latitude"] * 111.32
lon_km = df["longitude"] * 111.32 * np.cos(np.radians(df["latitude"]))
groups = (
    np.floor(lat_km / 5).astype(str) + "_" +
    np.floor(lon_km / 5).astype(str)
)

coords = np.radians(df[["latitude", "longitude"]].to_numpy())

splitter = GroupShuffleSplit(
    n_splits=5, test_size=0.25, random_state=42
)

# Generate splits once, then reuse them for all experiments
splits = list(splitter.split(df, y, groups))
results = []

for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    # Exclude training points within 5 km of any test point
    nn = NearestNeighbors(n_neighbors=1, metric="haversine")
    nn.fit(coords[test_idx])
    distances, _ = nn.kneighbors(coords[train_idx])
    train_idx = train_idx[distances[:, 0] * 6371 >= 5]

    for name, features in experiments.items():
        numeric_cols = [c for c in features if c != "LANDCOVER_CODE"]
        transformers = []

        if numeric_cols:
            transformers.append(
                ("numeric", "passthrough", numeric_cols)
            )

        if "LANDCOVER_CODE" in features:
            transformers.append(
                ("landcover", OneHotEncoder(handle_unknown="ignore"),
                 ["LANDCOVER_CODE"])
            )

        preprocess = ColumnTransformer(transformers)

        model = make_pipeline(
            preprocess,
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight="balanced"
            )
        )

        model.fit(df.iloc[train_idx][features], y.iloc[train_idx])

        y_test = y.iloc[test_idx]
        pred = model.predict(df.iloc[test_idx][features])
        prob = model.predict_proba(df.iloc[test_idx][features])[:, 1]

        results.append({
            "Fold": fold,
            "Model": name,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "F1": f1_score(y_test, pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, prob),
            "PR_AUC": average_precision_score(y_test, prob)
        })

results_df = pd.DataFrame(results)

print("RESULTS BY FOLD")
print(results_df.round(4).to_string(index=False))

metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"]

print("\nMEAN METRICS")
print(results_df.groupby("Model")[metrics].mean().round(4))

print("\nSTANDARD DEVIATION")
print(results_df.groupby("Model")[metrics].std().round(4))

RESULTS BY FOLD
 Fold              Model  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC
    1      Rainfall only    0.7650     0.2656  0.1532 0.1943   0.5784  0.2355
    1       Terrain only    0.7283     0.2636  0.2613 0.2624   0.6226  0.2506
    1 Rainfall + terrain    0.7850     0.3333  0.1622 0.2182   0.6678  0.2882
    2      Rainfall only    0.7694     0.2500  0.1667 0.2000   0.5377  0.2268
    2       Terrain only    0.7351     0.2571  0.2812 0.2687   0.6520  0.2525
    2 Rainfall + terrain    0.7982     0.3519  0.1979 0.2533   0.6857  0.2811
    3      Rainfall only    0.7617     0.2429  0.1604 0.1932   0.5573  0.2089
    3       Terrain only    0.7081     0.2763  0.3962 0.3256   0.6609  0.2669
    3 Rainfall + terrain    0.7752     0.2879  0.1792 0.2209   0.6718  0.2838
    4      Rainfall only    0.7955     0.2000  0.1282 0.1562   0.5487  0.1810
    4       Terrain only    0.7557     0.2135  0.2436 0.2275   0.6187  0.2112
    4 Rainfall + terrain    0.8011     0.1860  0

In [26]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import NearestNeighbors
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")
y = df["NEXT_1D"].astype(int)

rain = ["RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"]
terrain = ["ELEVATION", "SLOPE", "LANDCOVER_CODE"]

experiments = {
    "Terrain only": terrain,
    "Rainfall + terrain": rain + terrain
}

# Approximate 5 km spatial groups
lat_km = df["latitude"] * 111.32
lon_km = df["longitude"] * 111.32 * np.cos(
    np.radians(df["latitude"])
)
groups = (
    np.floor(lat_km / 5).astype(str) + "_" +
    np.floor(lon_km / 5).astype(str)
)

coords = np.radians(df[["latitude", "longitude"]].to_numpy())

def remove_nearby_training(train_idx, validation_idx):
    """Exclude training samples within 5 km of held-out samples."""
    nn = NearestNeighbors(n_neighbors=1, metric="haversine")
    nn.fit(coords[validation_idx])
    distances, _ = nn.kneighbors(coords[train_idx])
    return train_idx[distances[:, 0] * 6371 >= 5]

def build_model(features):
    numeric_cols = [c for c in features if c != "LANDCOVER_CODE"]
    transformers = []

    if numeric_cols:
        transformers.append(("numeric", "passthrough", numeric_cols))

    if "LANDCOVER_CODE" in features:
        transformers.append((
            "landcover",
            OneHotEncoder(handle_unknown="ignore"),
            ["LANDCOVER_CODE"]
        ))

    prep = ColumnTransformer(transformers)

    return make_pipeline(
        prep,
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        )
    )

def get_metrics(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, probabilities),
        "PR_AUC": average_precision_score(y_true, probabilities)
    }

# Five outer spatial test splits, shared by both models
outer_splitter = GroupShuffleSplit(
    n_splits=5, test_size=0.25, random_state=42
)

results = []

for fold, (outer_train, test_idx) in enumerate(
    outer_splitter.split(df, y, groups), start=1
):
    # Protect outer test from nearby training samples
    outer_train = remove_nearby_training(outer_train, test_idx)

    # Make a separate spatial validation split inside outer training
    inner_splitter = GroupShuffleSplit(
        n_splits=1, test_size=0.25, random_state=100 + fold
    )

    inner_train_local, val_local = next(
        inner_splitter.split(
            df.iloc[outer_train],
            y.iloc[outer_train],
            groups.iloc[outer_train]
        )
    )

    inner_train = outer_train[inner_train_local]
    val_idx = outer_train[val_local]

    inner_train = remove_nearby_training(inner_train, val_idx)

    for name, features in experiments.items():
        # 1. Select threshold on validation data only
        validation_model = build_model(features)
        validation_model.fit(
            df.iloc[inner_train][features],
            y.iloc[inner_train]
        )

        val_prob = validation_model.predict_proba(
            df.iloc[val_idx][features]
        )[:, 1]
        y_val = y.iloc[val_idx]

        thresholds = np.arange(0.05, 0.951, 0.01)
        f1_values = [
            f1_score(
                y_val, (val_prob >= t).astype(int),
                zero_division=0
            )
            for t in thresholds
        ]
        best_threshold = float(thresholds[np.argmax(f1_values)])

        # 2. Refit on all outer training data
        final_model = build_model(features)
        final_model.fit(
            df.iloc[outer_train][features],
            y.iloc[outer_train]
        )

        # 3. Evaluate once on the untouched outer test set
        test_prob = final_model.predict_proba(
            df.iloc[test_idx][features]
        )[:, 1]
        y_test = y.iloc[test_idx]

        for threshold_name, threshold in [
            ("Default 0.50", 0.50),
            ("Validation F1", best_threshold)
        ]:
            metrics = get_metrics(y_test, test_prob, threshold)
            results.append({
                "Fold": fold,
                "Model": name,
                "Threshold type": threshold_name,
                "Threshold": threshold,
                **metrics
            })

results_df = pd.DataFrame(results)

print("RESULTS BY FOLD")
print(results_df.round(4).to_string(index=False))

metric_cols = [
    "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC"
]

print("\nMEAN METRICS")
print(
    results_df.groupby(["Model", "Threshold type"])[metric_cols]
    .mean().round(4)
)

print("\nMEAN SELECTED THRESHOLD BY MODEL")
print(
    results_df[results_df["Threshold type"] == "Validation F1"]
    .groupby("Model")["Threshold"].mean().round(3)
)

RESULTS BY FOLD
 Fold              Model Threshold type  Threshold  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC
    1       Terrain only   Default 0.50       0.50    0.7283     0.2636  0.2613 0.2624   0.6226  0.2506
    1       Terrain only  Validation F1       0.20    0.5217     0.2317  0.6847 0.3462   0.6226  0.2506
    1 Rainfall + terrain   Default 0.50       0.50    0.7833     0.3273  0.1622 0.2169   0.6678  0.2882
    1 Rainfall + terrain  Validation F1       0.17    0.4400     0.2328  0.8829 0.3684   0.6678  0.2882
    2       Terrain only   Default 0.50       0.50    0.7351     0.2571  0.2812 0.2687   0.6520  0.2525
    2       Terrain only  Validation F1       0.17    0.5171     0.2312  0.7708 0.3558   0.6520  0.2525
    2 Rainfall + terrain   Default 0.50       0.50    0.7964     0.3455  0.1979 0.2517   0.6857  0.2811
    2 Rainfall + terrain  Validation F1       0.18    0.4450     0.2268  0.9167 0.3636   0.6857  0.2811
    3       Terrain only   Default 0.50       0.

In [30]:

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score
)

# 1. Load dataset
df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")
df["reference_date"] = pd.to_datetime(df["reference_date"])

TARGET = "NEXT_1D"
MIN_PRECISION = 0.30
BUFFER_KM = 5

rainfall = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D"
]
terrain = ["ELEVATION", "SLOPE", "LANDCOVER_CODE"]

models = {
    "Terrain only": terrain,
    "Rainfall + terrain": rainfall + terrain
}

X_all = df[sorted(set(rainfall + terrain))].copy()
y_all = df[TARGET].astype(int).to_numpy()
coords = df[["latitude", "longitude"]].to_numpy()

# 2. Remove training rows within BUFFER_KM of evaluation rows
def remove_nearby_train_points(train_idx, reference_idx,
                               coords, buffer_km=5):
    train_idx = np.asarray(train_idx, dtype=int)
    reference_idx = np.asarray(reference_idx, dtype=int)

    if len(train_idx) == 0 or len(reference_idx) == 0:
        return train_idx

    train_coords = coords[train_idx]
    reference_coords = coords[reference_idx]
    keep = np.ones(len(train_idx), dtype=bool)

    for i, point in enumerate(train_coords):
        lat_scale = 111.0
        lon_scale = 111.0 * np.cos(np.radians(point[0]))

        dlat = (reference_coords[:, 0] - point[0]) * lat_scale
        dlon = (reference_coords[:, 1] - point[1]) * lon_scale
        distances = np.sqrt(dlat**2 + dlon**2)

        if np.any(distances < buffer_km):
            keep[i] = False

    return train_idx[keep]


# 3. Model pipeline
def make_model(features):
    categorical = ["LANDCOVER_CODE"]
    numeric = [c for c in features if c not in categorical]

    preprocess = ColumnTransformer([
        ("numeric", "passthrough", numeric),
        ("categorical",
         OneHotEncoder(handle_unknown="ignore"),
         categorical)
    ])

    return Pipeline([
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])


# 4. Spatial groups (approximate grouping)
groups = np.column_stack([
    np.round(coords[:, 0], 1),
    np.round(coords[:, 1], 1)
])
groups = pd.Series(list(map(tuple, groups))).to_numpy()

outer = GroupShuffleSplit(
    n_splits=5,
    test_size=0.25,
    random_state=42
)

results = []

# 5. Outer spatial evaluation
for fold, (train_idx, test_idx) in enumerate(
    outer.split(X_all, y_all, groups=groups), start=1
):
    # Keep test rows untouched; buffer outer training from test
    train_idx = remove_nearby_train_points(
        train_idx, test_idx, coords, BUFFER_KM
    )

    # Split outer training into model-training and validation sets
    inner = GroupShuffleSplit(
        n_splits=1,
        test_size=0.25,
        random_state=100 + fold
    )

    inner_train_rel, val_rel = next(
        inner.split(
            X_all.iloc[train_idx],
            y_all[train_idx],
            groups=groups[train_idx]
        )
    )

    inner_train_idx = train_idx[inner_train_rel]
    val_idx = train_idx[val_rel]

    # Remove inner-training rows near validation locations
    inner_train_idx = remove_nearby_train_points(
        inner_train_idx, val_idx, coords, BUFFER_KM
    )

    for name, features in models.items():
        model = make_model(features)

        model.fit(
            X_all.iloc[inner_train_idx][features],
            y_all[inner_train_idx]
        )

        val_probs = model.predict_proba(
            X_all.iloc[val_idx][features]
        )[:, 1]

        # 6. Select threshold using validation data only
        candidates = []

        for threshold in np.arange(0.05, 0.96, 0.01):
            val_preds = (val_probs >= threshold).astype(int)

            precision = precision_score(
                y_all[val_idx], val_preds, zero_division=0
            )
            recall = recall_score(
                y_all[val_idx], val_preds, zero_division=0
            )
            f1 = f1_score(
                y_all[val_idx], val_preds, zero_division=0
            )

            if precision >= MIN_PRECISION and val_preds.sum() > 0:
                candidates.append(
                    (f1, threshold, precision, recall)
                )

        if not candidates:
            results.append({
                "Fold": fold,
                "Model": name,
                "Threshold": np.nan,
                "Status": "No validation threshold met 30% precision"
            })
            continue

        # Maximize validation F1 among thresholds meeting precision target
        _, best_threshold, val_precision, val_recall = max(
            candidates, key=lambda item: item[0]
        )

        # 7. Refit on the full outer training partition
        final_model = make_model(features)
        final_model.fit(
            X_all.iloc[train_idx][features],
            y_all[train_idx]
        )

        # 8. Evaluate once on untouched outer test partition
        test_probs = final_model.predict_proba(
            X_all.iloc[test_idx][features]
        )[:, 1]

        test_preds = (test_probs >= best_threshold).astype(int)

        results.append({
            "Fold": fold,
            "Model": name,
            "Threshold": round(best_threshold, 2),
            "Validation precision": round(val_precision, 3),
            "Validation recall": round(val_recall, 3),
            "Test precision": round(precision_score(
                y_all[test_idx], test_preds, zero_division=0
            ), 3),
            "Test recall": round(recall_score(
                y_all[test_idx], test_preds, zero_division=0
            ), 3),
            "Test F1": round(f1_score(
                y_all[test_idx], test_preds, zero_division=0
            ), 3),
            "Test accuracy": round(accuracy_score(
                y_all[test_idx], test_preds
            ), 3),
            "Test ROC-AUC": round(roc_auc_score(
                y_all[test_idx], test_probs
            ), 3),
            "Test PR-AUC": round(average_precision_score(
                y_all[test_idx], test_probs
            ), 3),
            "Status": "OK"
        })

# 9. Display results
results_df = pd.DataFrame(results)

display(results_df)

metric_cols = [
    "Test precision", "Test recall", "Test F1",
    "Test accuracy", "Test ROC-AUC", "Test PR-AUC"
]

print("Mean test metrics across folds:")
display(
    results_df.groupby("Model")[metric_cols]
    .mean()
    .round(3)
)


,Fold,Model,Threshold,Validation precision,Validation recall,Test precision,Test recall,Test F1,Test accuracy,Test ROC-AUC,Test PR-AUC,Status
0,1,Terrain only,0.62,0.302,0.264,0.293,0.136,0.186,0.806,0.660,0.245,OK
1,1,Rainfall + terrain,0.66,1.000,0.028,0.333,0.011,0.022,0.835,0.700,0.271,OK
2,2,Terrain only,0.64,0.300,0.091,0.308,0.282,0.294,0.782,0.670,0.258,OK
3,2,Rainfall + terrain,0.50,0.318,0.106,0.200,0.141,0.166,0.770,0.696,0.263,OK
4,3,Terrain only,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No validation threshold met 30% precision
5,3,Rainfall + terrain,0.64,0.667,0.029,0.273,0.034,0.060,0.832,0.660,0.233,OK
6,4,Terrain only,0.62,0.302,0.188,0.231,0.080,0.119,0.832,0.661,0.207,OK
7,4,Rainfall + terrain,0.48,0.319,0.217,0.241,0.187,0.211,0.802,0.717,0.235,OK
8,5,Terrain only,0.68,0.333,0.043,0.326,0.151,0.206,0.807,0.650,0.254,OK
9,5,Rainfall + terrain,0.52,0.357,0.106,0.312,0.161,0.213,0.801,0.677,0.264,OK


Mean test metrics across folds:


,Test precision,Test recall,Test F1,Test accuracy,Test ROC-AUC,Test PR-AUC
Model,,,,,,
Rainfall + terrain,0.272,0.107,0.134,0.808,0.69,0.253
Terrain only,0.290,0.162,0.201,0.807,0.66,0.241


In [31]:

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# =========================================================
# 1. Load V5 without modifying the original file
# =========================================================
df = pd.read_csv("../data/himalayan_temporal_final_v5.csv")
df["reference_date"] = pd.to_datetime(
    df["reference_date"], errors="coerce"
)

TARGET = "NEXT_1D"
MIN_PRECISION = 0.30

rain_features = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]
terrain_features = [
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]
features = rain_features + terrain_features

df = df.dropna(
    subset=["reference_date", TARGET]
).copy()
df[TARGET] = df[TARGET].astype(int)

# =========================================================
# 2. Chronological split: earlier train, middle validation,
#    latest dates test. Keep each date in only one split.
# =========================================================
dates = np.sort(df["reference_date"].unique())

if len(dates) < 5:
    raise ValueError(
        "Too few distinct dates for a reliable temporal split."
    )

train_end = dates[int(len(dates) * 0.60) - 1]
val_end = dates[int(len(dates) * 0.80) - 1]

train = df[df["reference_date"] <= train_end].copy()
val = df[
    (df["reference_date"] > train_end) &
    (df["reference_date"] <= val_end)
].copy()
test = df[df["reference_date"] > val_end].copy()

for name, part in [
    ("Train", train), ("Validation", val), ("Test", test)
]:
    print(
        f"{name}: {len(part)} rows | "
        f"{part['reference_date'].min().date()} to "
        f"{part['reference_date'].max().date()} | "
        f"classes={part[TARGET].value_counts().to_dict()}"
    )

if any(part.empty for part in [train, val, test]):
    raise ValueError("One of the chronological splits is empty.")

if any(part[TARGET].nunique() < 2 for part in [train, val, test]):
    raise ValueError(
        "A split has only one target class. "
        "Use date boundaries that leave both classes in each split."
    )

# =========================================================
# 3. Preprocessing: land-cover codes are categorical,
#    not continuous measurements.
# =========================================================
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder)
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, rain_features + ["ELEVATION", "SLOPE"]),
    ("landcover", categorical_pipe, ["LANDCOVER_CODE"])
])

# =========================================================
# 4. Compare three models on identical data
# =========================================================
model_definitions = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Histogram Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    )
}

X_train = train[features]
y_train = train[TARGET].to_numpy()

X_val = val[features]
y_val = val[TARGET].to_numpy()

X_test = test[features]
y_test = test[TARGET].to_numpy()

results = []

for name, estimator in model_definitions.items():
    model = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", clone(estimator))
    ])

    if name == "Histogram Gradient Boosting":
        weights = compute_sample_weight("balanced", y_train)
        model.fit(
            X_train, y_train,
            model__sample_weight=weights
        )
    else:
        model.fit(X_train, y_train)

    # Validation predictions select the threshold.
    val_probs = model.predict_proba(X_val)[:, 1]

    candidates = []
    all_thresholds = []

    for threshold in np.arange(0.05, 0.96, 0.01):
        preds = (val_probs >= threshold).astype(int)

        precision = precision_score(
            y_val, preds, zero_division=0
        )
        recall = recall_score(
            y_val, preds, zero_division=0
        )
        f1 = f1_score(y_val, preds, zero_division=0)

        record = (f1, threshold, precision, recall)

        all_thresholds.append(record)

        if precision >= MIN_PRECISION and preds.sum() > 0:
            candidates.append(record)

    if candidates:
        best = max(candidates, key=lambda row: row[0])
        constraint_met = True
    else:
        best = max(all_thresholds, key=lambda row: row[0])
        constraint_met = False

    _, threshold, val_precision, val_recall = best

    # Test is used only after the threshold has been selected.
    test_probs = model.predict_proba(X_test)[:, 1]

    test_preds = (test_probs >= threshold).astype(int)
    default_preds = (test_probs >= 0.50).astype(int)

    results.append({
        "Model": name,
        "Threshold": round(float(threshold), 2),
        "Validation precision": round(float(val_precision), 3),
        "Validation recall": round(float(val_recall), 3),
        "30% validation target met": constraint_met,
        "Test prevalence": round(float(y_test.mean()), 3),
        "Test precision": round(
            precision_score(y_test, test_preds, zero_division=0), 3
        ),
        "Test recall": round(
            recall_score(y_test, test_preds, zero_division=0), 3
        ),
        "Test F1": round(
            f1_score(y_test, test_preds, zero_division=0), 3
        ),
        "Test accuracy": round(
            accuracy_score(y_test, test_preds), 3
        ),
        "Test ROC-AUC": round(
            roc_auc_score(y_test, test_probs), 3
        ),
        "Test PR-AUC": round(
            average_precision_score(y_test, test_probs), 3
        ),
        "Test precision at 0.50": round(
            precision_score(y_test, default_preds, zero_division=0), 3
        ),
        "Test recall at 0.50": round(
            recall_score(y_test, default_preds, zero_division=0), 3
        )
    })

results_df = pd.DataFrame(results).sort_values(
    "Test PR-AUC", ascending=False
)

print("\nFinal 24-hour experiment:")
display(results_df.reset_index(drop=True))

print(
    "\nRandom-ranking PR-AUC reference:",
    round(float(y_test.mean()), 3)
)


Train: 1560 rows | 2007-07-27 to 2014-07-27 | classes={0: 1300, 1: 260}
Validation: 350 rows | 2014-08-12 to 2016-08-06 | classes={0: 292, 1: 58}
Test: 340 rows | 2016-08-07 to 2018-10-08 | classes={0: 283, 1: 57}

Final 24-hour experiment:


,Model,Threshold,Validation precision,Validation recall,30% validation target met,Test prevalence,Test precision,Test recall,Test F1,Test accuracy,Test ROC-AUC,Test PR-AUC,Test precision at 0.50,Test recall at 0.50
0,Extra Trees,0.55,0.311,0.328,True,0.168,0.297,0.193,0.234,0.788,0.654,0.288,0.247,0.386
1,Random Forest,0.59,0.308,0.138,True,0.168,0.412,0.123,0.189,0.824,0.649,0.254,0.269,0.368
2,Histogram Gradient Boosting,0.62,0.333,0.293,True,0.168,0.220,0.158,0.184,0.765,0.629,0.245,0.240,0.439



Random-ranking PR-AUC reference: 0.168


In [32]:

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    HistGradientBoostingClassifier
)
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# ======================================================
# 1. LOAD DATA: prefer the separate lithology dataset
# ======================================================
data_dir = Path("../data")
lithology_path = data_dir / "himalayan_temporal_final_v5_lithology.csv"
base_path = data_dir / "himalayan_temporal_final_v5.csv"

if lithology_path.exists():
    df = pd.read_csv(lithology_path)
    print("Using V5 with lithology.")
else:
    df = pd.read_csv(base_path)
    print("Lithology file unavailable; using original V5.")

df["reference_date"] = pd.to_datetime(
    df["reference_date"], errors="coerce"
)

TARGET = "NEXT_1D"
TARGET_RECALL = 0.80

df = df.dropna(subset=["reference_date", TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

# ======================================================
# 2. RAINFALL FEATURE ENGINEERING
# Assumes rainfall columns are cumulative totals over
# the named preceding windows. Verify this in your data.
# ======================================================
required_rain = [
    "RAIN_1D", "RAIN_3D", "RAIN_7D",
    "RAIN_15D", "RAIN_30D"
]

missing = [c for c in required_rain if c not in df.columns]
if missing:
    raise ValueError(f"Missing rainfall columns: {missing}")

# Approximate rainfall in the intervals between windows
df["RAIN_PREV_2D"] = (df["RAIN_3D"] - df["RAIN_1D"]).clip(lower=0)
df["RAIN_PREV_4D"] = (df["RAIN_7D"] - df["RAIN_3D"]).clip(lower=0)
df["RAIN_PREV_8D"] = (df["RAIN_15D"] - df["RAIN_7D"]).clip(lower=0)
df["RAIN_PREV_15D"] = (df["RAIN_30D"] - df["RAIN_15D"]).clip(lower=0)

# Recent rainfall relative to the longer-term average
df["RAIN_1D_VS_3D"] = (
    df["RAIN_1D"] - df["RAIN_3D"] / 3
)
df["RAIN_3D_VS_7D"] = (
    df["RAIN_3D"] / 3 - df["RAIN_7D"] / 7
)

rain_features = required_rain + [
    "RAIN_PREV_2D", "RAIN_PREV_4D",
    "RAIN_PREV_8D", "RAIN_PREV_15D",
    "RAIN_1D_VS_3D", "RAIN_3D_VS_7D"
]

# Include coordinates and terrain; lithology is optional.
numeric_features = rain_features + [
    c for c in [
        "latitude", "longitude", "ELEVATION", "SLOPE"
    ] if c in df.columns
]

categorical_features = [
    c for c in ["LANDCOVER_CODE", "LITHOLOGY"]
    if c in df.columns
]

features = numeric_features + categorical_features

# Explicitly exclude target-derived fields, dates and identifiers.
for forbidden in [
    "LANDSLIDE", "NEXT_1D", "NEXT_2D", "NEXT_3D",
    "reference_date", "event_date", "positive_horizon",
    "future_event_count"
]:
    if forbidden in features:
        features.remove(forbidden)

print("Features used:", features)

# ======================================================
# 3. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ======================================================
dates = np.sort(df["reference_date"].unique())

if len(dates) < 5:
    raise ValueError("Not enough distinct dates for temporal splitting.")

train_end = dates[int(len(dates) * 0.60) - 1]
val_end = dates[int(len(dates) * 0.80) - 1]

train = df[df["reference_date"] <= train_end].copy()
val = df[
    (df["reference_date"] > train_end) &
    (df["reference_date"] <= val_end)
].copy()
test = df[df["reference_date"] > val_end].copy()

for name, part in [
    ("Train", train), ("Validation", val), ("Test", test)
]:
    print(
        f"{name}: {len(part)} rows, "
        f"positive rate={part[TARGET].mean():.3f}, "
        f"dates={part['reference_date'].min().date()} "
        f"to {part['reference_date'].max().date()}"
    )

if any(part.empty or part[TARGET].nunique() < 2
       for part in [train, val, test]):
    raise ValueError(
        "A split is empty or contains only one class. "
        "Check the chronological boundaries."
    )

X_train, y_train = train[features], train[TARGET].to_numpy()
X_val, y_val = val[features], val[TARGET].to_numpy()
X_test, y_test = test[features], test[TARGET].to_numpy()

# ======================================================
# 4. PREPROCESSING
# ======================================================
try:
    encoder = OneHotEncoder(
        handle_unknown="ignore", sparse_output=False
    )
except TypeError:
    encoder = OneHotEncoder(
        handle_unknown="ignore", sparse=False
    )

transformers = []

if numeric_features:
    transformers.append((
        "numeric",
        SimpleImputer(strategy="median"),
        numeric_features
    ))

if categorical_features:
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", encoder)
    ])
    transformers.append((
        "categorical", categorical_pipe, categorical_features
    ))

preprocessor = ColumnTransformer(
    transformers, sparse_threshold=0
)

# ======================================================
# 5. CANDIDATE MODELS WITH DIFFERENT POSITIVE WEIGHTS
# Higher weight tells the model to pay more attention
# to missed positive examples.
# ======================================================
candidates = []

for weight in [2, 4, 8]:
    candidates.append((
        f"Extra Trees, positive weight {weight}",
        ExtraTreesClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        ),
        weight
    ))

    candidates.append((
        f"Random Forest, positive weight {weight}",
        RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            max_features="sqrt",
            random_state=42,
            n_jobs=-1
        ),
        weight
    ))

candidates.append((
    "Histogram Gradient Boosting, positive weight 4",
    HistGradientBoostingClassifier(
        max_iter=200,
        learning_rate=0.05,
        max_leaf_nodes=15,
        l2_regularization=1.0,
        random_state=42
    ),
    4
))

# ======================================================
# 6. SELECT THRESHOLD AND MODEL USING VALIDATION ONLY
# Choose the highest threshold that achieves at least
# 80% validation recall. This favors precision among
# thresholds satisfying the recall goal.
# ======================================================
validation_results = []
fitted_models = {}

for name, estimator, positive_weight in candidates:
    model = Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", clone(estimator))
    ])

    sample_weights = np.where(
        y_train == 1, positive_weight, 1.0
    )

    model.fit(
        X_train, y_train,
        model__sample_weight=sample_weights
    )

    val_probs = model.predict_proba(X_val)[:, 1]

    thresholds = np.unique(
        np.concatenate(([0.0], val_probs, [1.0]))
    )

    valid_choices = []

    for threshold in thresholds:
        preds = (val_probs >= threshold).astype(int)

        rec = recall_score(y_val, preds, zero_division=0)
        prec = precision_score(y_val, preds, zero_division=0)

        if rec >= TARGET_RECALL and preds.sum() > 0:
            valid_choices.append((threshold, prec, rec))

    if valid_choices:
        threshold, val_precision, val_recall = max(
            valid_choices, key=lambda row: row[0]
        )
        goal_met = True
    else:
        threshold = 0.0
        val_precision = precision_score(
            y_val, np.ones(len(y_val)), zero_division=0
        )
        val_recall = 1.0
        goal_met = False

    val_preds = (val_probs >= threshold).astype(int)

    val_pr_auc = average_precision_score(y_val, val_probs)

    validation_results.append({
        "Model": name,
        "Positive weight": positive_weight,
        "Threshold": float(threshold),
        "Validation precision": val_precision,
        "Validation recall": val_recall,
        "Validation F1": f1_score(
            y_val, val_preds, zero_division=0
        ),
        "Validation PR-AUC": val_pr_auc,
        "Recall target met": goal_met
    })

    fitted_models[name] = (model, threshold)

validation_df = pd.DataFrame(validation_results)

# Select by validation precision among models meeting recall goal;
# use validation PR-AUC as a tie-breaker.
eligible = validation_df[validation_df["Recall target met"]]

if len(eligible):
    winner_name = eligible.sort_values(
        ["Validation precision", "Validation PR-AUC"],
        ascending=False
    ).iloc[0]["Model"]
else:
    winner_name = validation_df.sort_values(
        ["Validation recall", "Validation PR-AUC"],
        ascending=False
    ).iloc[0]["Model"]

winner, chosen_threshold = fitted_models[winner_name]

print("\nValidation comparison (no test metrics used for selection):")
display(
    validation_df.sort_values(
        ["Recall target met", "Validation precision"],
        ascending=False
    ).round(3).reset_index(drop=True)
)

print("\nSelected model:", winner_name)
print("Selected threshold:", round(chosen_threshold, 4))

# ======================================================
# 7. FINAL EVALUATION ON THE HELD-OUT TEST PERIOD
# Only the selected model is evaluated here.
# ======================================================
test_probs = winner.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= chosen_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, test_preds, labels=[0, 1]
).ravel()

print("\nFINAL 24-HOUR EXPERIMENT")
print("Test period:",
      test["reference_date"].min().date(), "to",
      test["reference_date"].max().date())
print("Threshold:", round(chosen_threshold, 4))
print("Test positive rate:", round(y_test.mean(), 3))
print("Test precision:", round(
    precision_score(y_test, test_preds, zero_division=0), 3
))
print("Test recall:", round(
    recall_score(y_test, test_preds, zero_division=0), 3
))
print("Test F1:", round(
    f1_score(y_test, test_preds, zero_division=0), 3
))
print("Test accuracy:", round(
    accuracy_score(y_test, test_preds), 3
))
print("Test ROC-AUC:", round(
    roc_auc_score(y_test, test_probs), 3
))
print("Test PR-AUC:", round(
    average_precision_score(y_test, test_probs), 3
))

print("\nTest confusion counts:")
print("True positives (events caught):", tp)
print("False negatives (events missed):", fn)
print("False positives (false alarms):", fp)
print("True negatives:", tn)
print("Predicted positive rate:", round(test_preds.mean(), 3))

# Save experiment summary separately; original datasets untouched.
summary = pd.DataFrame([{
    "Model": winner_name,
    "Threshold": chosen_threshold,
    "Test precision": precision_score(
        y_test, test_preds, zero_division=0
    ),
    "Test recall": recall_score(
        y_test, test_preds, zero_division=0
    ),
    "Test F1": f1_score(
        y_test, test_preds, zero_division=0
    ),
    "Test ROC-AUC": roc_auc_score(y_test, test_probs),
    "Test PR-AUC": average_precision_score(y_test, test_probs),
    "TP": tp, "FN": fn, "FP": fp, "TN": tn
}])

summary.to_csv(
    "../data/last_24h_experiment_results.csv", index=False
)

print("\nSaved summary to data/last_24h_experiment_results.csv")


Using V5 with lithology.
Features used: ['RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'RAIN_PREV_2D', 'RAIN_PREV_4D', 'RAIN_PREV_8D', 'RAIN_PREV_15D', 'RAIN_1D_VS_3D', 'RAIN_3D_VS_7D', 'latitude', 'longitude', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE', 'LITHOLOGY']
Train: 1560 rows, positive rate=0.167, dates=2007-07-27 to 2014-07-27
Validation: 350 rows, positive rate=0.166, dates=2014-08-12 to 2016-08-06
Test: 340 rows, positive rate=0.168, dates=2016-08-07 to 2018-10-08

Validation comparison (no test metrics used for selection):


,Model,Positive weight,Threshold,Validation precision,Validation recall,Validation F1,Validation PR-AUC,Recall target met
0,"Random Forest, positive weight 4",4,0.314,0.288,0.81,0.425,0.285,True
1,"Random Forest, positive weight 8",8,0.390,0.281,0.81,0.418,0.279,True
2,"Random Forest, positive weight 2",2,0.232,0.276,0.81,0.412,0.282,True
3,"Histogram Gradient Boosting, positive weight 4",4,0.230,0.272,0.81,0.407,0.262,True
4,"Extra Trees, positive weight 8",8,0.183,0.242,0.81,0.373,0.277,True
5,"Extra Trees, positive weight 4",4,0.175,0.240,0.81,0.370,0.283,True
6,"Extra Trees, positive weight 2",2,0.150,0.232,0.81,0.360,0.296,True



Selected model: Random Forest, positive weight 4
Selected threshold: 0.3135

FINAL 24-HOUR EXPERIMENT
Test period: 2016-08-07 to 2018-10-08
Threshold: 0.3135
Test positive rate: 0.168
Test precision: 0.261
Test recall: 0.649
Test F1: 0.372
Test accuracy: 0.632
Test ROC-AUC: 0.664
Test PR-AUC: 0.254

Test confusion counts:
True positives (events caught): 37
False negatives (events missed): 20
False positives (false alarms): 105
True negatives: 178
Predicted positive rate: 0.418

Saved summary to data/last_24h_experiment_results.csv


In [33]:

import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

# Negative controls
neg = pd.read_csv(DATA_DIR / "v5_negative_definitions.csv")

# Known positive event definitions
pos = pd.read_csv(DATA_DIR / "multi_horizon_positive_definitions.csv")

print("NEGATIVE CONTROLS")
print("Shape:", neg.shape)
print("Columns:", neg.columns.tolist())
display(neg.head())

print("\nPOSITIVE EVENT DEFINITIONS")
print("Shape:", pos.shape)
print("Columns:", pos.columns.tolist())
display(pos.head())

# Check date coverage
for name, df in [("Negative controls", neg), ("Positive definitions", pos)]:
    print(f"\n{name}")
    if "reference_date" in df.columns:
        dates = pd.to_datetime(df["reference_date"], errors="coerce")
        print("Reference date range:", dates.min(), "to", dates.max())
    if "event_date" in df.columns:
        dates = pd.to_datetime(df["event_date"], errors="coerce")
        print("Event date range:", dates.min(), "to", dates.max())

# Check available inventory files
print("\nPOSSIBLE EVENT INVENTORY FILES")
for p in DATA_DIR.glob("*"):
    if any(word in p.name.lower() for word in ["landslide", "inventory", "event"]):
        print(p.name)


NEGATIVE CONTROLS
Shape: (1125, 7)
Columns: ['control_id', 'latitude', 'longitude', 'reference_date', 'positive_horizon', 'nearest_event_km', 'LANDSLIDE']


,control_id,latitude,longitude,reference_date,positive_horizon,nearest_event_km,LANDSLIDE
0,0,29.286069,80.202177,2011-08-11,1,11.046148,0
1,1,30.376208,77.716244,2011-08-10,2,22.707925,0
2,2,31.628616,78.265249,2011-08-09,3,6.603815,0
3,3,32.269574,76.442974,2014-07-22,1,10.193653,0
4,4,31.672466,76.424893,2014-07-21,2,8.347840,0



POSITIVE EVENT DEFINITIONS
Shape: (1125, 6)
Columns: ['latitude', 'longitude', 'reference_date', 'event_date', 'positive_horizon', 'future_event_count']


,latitude,longitude,reference_date,event_date,positive_horizon,future_event_count
0,28.8665,79.5682,2011-08-11,2011-08-12,1,1
1,28.8665,79.5682,2011-08-10,2011-08-12,2,1
2,28.8665,79.5682,2011-08-09,2011-08-12,3,1
3,29.1585,79.1464,2014-07-22,2014-07-23,1,1
4,29.1585,79.1464,2014-07-21,2014-07-23,2,1



Negative controls
Reference date range: 2007-07-27 00:00:00 to 2018-10-08 00:00:00

Positive definitions
Reference date range: 2007-07-27 00:00:00 to 2018-10-08 00:00:00
Event date range: 2007-07-30 00:00:00 to 2018-10-09 00:00:00

POSSIBLE EVENT INVENTORY FILES
himalayan_event_rainfall.csv
himalayan_landslide_v1_dataset.csv


In [34]:

import pandas as pd
from pathlib import Path

path = Path("../data/himalayan_event_rainfall.csv")
events = pd.read_csv(path)

print("Shape:", events.shape)
print("\nColumns:")
print(events.columns.tolist())

display(events.head())

print("\nData types:")
print(events.dtypes)

print("\nMissing values:")
print(events.isnull().sum())


Shape: (377, 10)

Columns:
['latitude', 'longitude', 'event_date', 'state', 'trigger', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D']


,latitude,longitude,event_date,state,trigger,RAIN_1D,RAIN_3D,RAIN_7D,RAIN_15D,RAIN_30D
0,30.7666,78.7963,2011-06-29,Uttarakhand,downpour,0.000000,0.000000,351.082520,351.082520,351.082520
1,30.7652,78.5925,2011-07-09,Uttarakhand,downpour,0.000000,0.000000,0.000000,89.277454,165.996467
2,30.7331,79.0685,2011-07-09,Uttarakhand,downpour,0.000000,0.000000,0.000000,0.000000,394.746674
3,31.0188,78.4129,2011-07-09,Uttarakhand,downpour,0.000000,0.000000,0.000000,156.171574,372.439198
4,29.7334,79.6367,2011-07-21,Uttarakhand,rain,49.256058,65.674744,65.674744,152.320995,365.862450



Data types:
latitude      float64
longitude     float64
event_date     object
state          object
trigger        object
RAIN_1D       float64
RAIN_3D       float64
RAIN_7D       float64
RAIN_15D      float64
RAIN_30D      float64
dtype: object

Missing values:
latitude      0
longitude     0
event_date    0
state         0
trigger       0
RAIN_1D       0
RAIN_3D       0
RAIN_7D       0
RAIN_15D      0
RAIN_30D      0
dtype: int64


In [35]:

events["event_date"] = pd.to_datetime(events["event_date"], errors="coerce")
neg["reference_date"] = pd.to_datetime(neg["reference_date"], errors="coerce")
pos["event_date"] = pd.to_datetime(pos["event_date"], errors="coerce")

print("EVENT FILE")
print("Rows:", len(events))
print("Unique event dates:", events["event_date"].nunique())
print("Unique coordinates:", events[["latitude", "longitude"]].drop_duplicates().shape[0])
print("Date range:", events["event_date"].min(), "to", events["event_date"].max())

print("\nNEGATIVE CONTROLS")
print("Rows:", len(neg))
print("Horizon distribution:")
print(neg["positive_horizon"].value_counts(dropna=False).sort_index())

print("\nPOSITIVE DEFINITIONS")
print("Rows:", len(pos))
print("Horizon distribution:")
print(pos["positive_horizon"].value_counts(dropna=False).sort_index())

print("\nEVENT DATE MATCH")
print(
    "Positive event dates found in event file:",
    pos["event_date"].isin(events["event_date"]).sum(),
    "out of", len(pos)
)


EVENT FILE
Rows: 377
Unique event dates: 212
Unique coordinates: 374
Date range: 2007-07-30 00:00:00 to 2018-10-09 00:00:00

NEGATIVE CONTROLS
Rows: 1125
Horizon distribution:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64

POSITIVE DEFINITIONS
Rows: 1125
Horizon distribution:
positive_horizon
1    375
2    375
3    375
Name: count, dtype: int64

EVENT DATE MATCH
Positive event dates found in event file: 1125 out of 1125


In [36]:

# Round coordinates to 5 decimal places to avoid tiny floating-point differences
event_keys = events.assign(
    latitude=events["latitude"].round(5),
    longitude=events["longitude"].round(5)
)[["latitude", "longitude", "event_date"]].drop_duplicates()

positive_keys = pos.assign(
    latitude=pos["latitude"].round(5),
    longitude=pos["longitude"].round(5)
)[["latitude", "longitude", "event_date"]].drop_duplicates()

matched = positive_keys.merge(
    event_keys,
    on=["latitude", "longitude", "event_date"],
    how="inner"
)

print("Unique positive location-date-event combinations:", len(positive_keys))
print("Matching combinations in event file:", len(matched))
print("Not matched:", len(positive_keys) - len(matched))


Unique positive location-date-event combinations: 375
Matching combinations in event file: 375
Not matched: 0


In [37]:

# Compare unique events and duplicate records
event_cols = ["latitude", "longitude", "event_date"]

print("Total event rows:", len(events))
print("Unique location-date combinations:",
      events[event_cols].drop_duplicates().shape[0])

print("\nDuplicate location-date rows:")
display(
    events[
        events.duplicated(subset=event_cols, keep=False)
    ].sort_values(event_cols)
)

print("\nState distribution:")
print(events["state"].value_counts(dropna=False))

print("\nDate counts (most frequent):")
print(events["event_date"].value_counts().head(10))


Total event rows: 377
Unique location-date combinations: 375

Duplicate location-date rows:


,latitude,longitude,event_date,state,trigger,RAIN_1D,RAIN_3D,RAIN_7D,RAIN_15D,RAIN_30D
159,29.3802,79.4633,2013-07-12,Uttarakhand,downpour,34.95256,189.954453,228.031696,433.065125,863.889584
164,29.3802,79.4633,2013-07-12,Uttarakhand,rain,34.95256,189.954453,228.031696,433.065125,863.889584
160,30.8928,78.4800,2013-07-25,Uttarakhand,continuous_rain,0.00000,50.103104,308.883331,308.883331,507.739288
280,30.8928,78.4800,2013-07-25,Uttarakhand,continuous_rain,0.00000,50.103104,308.883331,308.883331,507.739288



State distribution:
state
Uttarakhand         204
Himachal Pradesh    173
Name: count, dtype: int64

Date counts (most frequent):
event_date
2010-09-19    19
2013-06-16    10
2011-06-29     8
2013-07-07     7
2010-07-07     6
2013-07-24     6
2010-09-20     6
2015-07-26     6
2013-07-25     5
2011-08-07     5
Name: count, dtype: int64


In [38]:

from pathlib import Path

data_dir = Path("../data")

for file in sorted(data_dir.iterdir()):
    if file.is_file():
        print(f"{file.name:45} {file.stat().st_size / 1024:.1f} KB")


background_locations.csv                      6.6 KB
background_locations_v2.csv                   17.4 KB
himalayan_event_rainfall.csv                  47.7 KB
himalayan_landslide_v1_dataset.csv            1659.3 KB
himalayan_lithology.geojson                   3588.9 KB
himalayan_temporal_dataset.csv                91.6 KB
himalayan_temporal_dataset_v2.csv             103.0 KB
himalayan_temporal_final_v3.csv               320.2 KB
himalayan_temporal_final_v4.csv               321.7 KB
himalayan_temporal_final_v5.csv               318.9 KB
himalayan_temporal_final_v5_lithology.csv     384.6 KB
himalayan_temporal_model_dataset_v1.csv       320.4 KB
HMA_LS_Cat_point_v02.0.zip                    820.3 KB
last_24h_experiment_results.csv               0.2 KB
matched_negative_temporal.csv                 54.8 KB
multi_horizon_negative_definitions.csv        18.5 KB
multi_horizon_negative_definitions_audited.csv 26.2 KB
multi_horizon_negative_definitions_cleaned.csv 18.4 KB
multi_horizon_neg

In [39]:

from pathlib import Path
import pandas as pd
import zipfile

data_dir = Path("../data")

# 1. Inspect the existing audited negatives
audited_path = data_dir / "multi_horizon_negative_definitions_audited.csv"

if audited_path.exists():
    audited = pd.read_csv(audited_path)
    print("AUDITED NEGATIVES")
    print("Shape:", audited.shape)
    print("Columns:", audited.columns.tolist())
    display(audited.head())

# 2. Inspect the contents of the landslide inventory ZIP
zip_path = data_dir / "HMA_LS_Cat_point_v02.0.zip"

if zip_path.exists():
    with zipfile.ZipFile(zip_path) as z:
        print("\nZIP CONTENTS")
        for item in z.infolist():
            print(item.filename, f"({item.file_size / 1024:.1f} KB)")


AUDITED NEGATIVES
Shape: (477, 6)
Columns: ['latitude', 'longitude', 'reference_date', 'LANDSLIDE', 'nearest_event_km', 'event_within_1km_next_3d']


,latitude,longitude,reference_date,LANDSLIDE,nearest_event_km,event_within_1km_next_3d
0,32.301650,77.934624,2007-07-27,0,11.364974,False
1,29.841000,79.775800,2007-07-28,0,0.000000,False
2,31.598600,78.438400,2007-07-29,0,0.000000,False
3,30.653877,79.597045,2007-08-08,0,3.549211,False
4,31.050818,77.185099,2007-08-09,0,4.203525,False



ZIP CONTENTS
HMA_LS_Cat_point_v02.0.cpg (0.0 KB)
HMA_LS_Cat_point_v02.0.dbf (6872.1 KB)
HMA_LS_Cat_point_v02.0.prj (0.1 KB)
HMA_LS_Cat_point_v02.0.sbn (26.9 KB)
HMA_LS_Cat_point_v02.0.sbx (1.4 KB)
HMA_LS_Cat_point_v02.0.shp (76.7 KB)
HMA_LS_Cat_point_v02.0.shp.xml (4617.5 KB)
HMA_LS_Cat_point_v02.0.shx (22.0 KB)
HMA_LS_Cat_point_v02.0.xml (5.2 KB)


In [40]:

from pathlib import Path
import pandas as pd

data_dir = Path("../data")

# Check existing geographic files that might contain state boundaries
print("GEOGRAPHIC FILES")
for p in sorted(data_dir.iterdir()):
    if p.suffix.lower() in [".shp", ".geojson", ".gpkg", ".json"]:
        print(p.name)

# Check whether state labels already exist in the main datasets
for filename in [
    "himalayan_temporal_final_v5.csv",
    "himalayan_temporal_final_v5_lithology.csv",
    "v5_negative_definitions.csv",
    "background_locations.csv",
    "background_locations_v2.csv",
    "himalayan_event_rainfall.csv"
]:
    path = data_dir / filename
    if path.exists():
        df = pd.read_csv(path, nrows=5)
        print(f"\n{filename}")
        print("Columns:", df.columns.tolist())


GEOGRAPHIC FILES
himalayan_lithology.geojson

himalayan_temporal_final_v5.csv
Columns: ['latitude', 'longitude', 'reference_date', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D']

himalayan_temporal_final_v5_lithology.csv
Columns: ['latitude', 'longitude', 'reference_date', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D', 'ELEVATION', 'SLOPE', 'LANDCOVER_CODE', 'LANDSLIDE', 'NEXT_1D', 'NEXT_2D', 'NEXT_3D', 'LITHO_CODE', 'LITHOLOGY']

v5_negative_definitions.csv
Columns: ['control_id', 'latitude', 'longitude', 'reference_date', 'positive_horizon', 'nearest_event_km', 'LANDSLIDE']

background_locations.csv
Columns: ['latitude', 'longitude']

background_locations_v2.csv
Columns: ['latitude', 'longitude']

himalayan_event_rainfall.csv
Columns: ['latitude', 'longitude', 'event_date', 'state', 'trigger', 'RAIN_1D', 'RAIN_3D', 'RAIN_7D', 'RAIN_15D', 'RAIN_30D']


In [41]:

import pandas as pd
import json
from pathlib import Path

data_dir = Path("../data")

# Inspect coordinates of known events by state
events = pd.read_csv(data_dir / "himalayan_event_rainfall.csv")

print("EVENT COORDINATE RANGES")
display(
    events.groupby("state").agg(
        count=("latitude", "size"),
        min_lat=("latitude", "min"),
        max_lat=("latitude", "max"),
        min_lon=("longitude", "min"),
        max_lon=("longitude", "max")
    )
)

# Inspect the existing lithology GeoJSON
with open(data_dir / "himalayan_lithology.geojson", encoding="utf-8") as f:
    geo = json.load(f)

print("\nGEOJSON feature count:", len(geo.get("features", [])))

if geo.get("features"):
    print("First feature properties:")
    print(geo["features"][0].get("properties", {}))


EVENT COORDINATE RANGES


,count,min_lat,max_lat,min_lon,max_lon
state,,,,,
Himachal Pradesh,173,30.5589,32.763909,75.9568,78.6167
Uttarakhand,204,28.8665,31.035800,77.8644,80.8576



GEOJSON feature count: 753
First feature properties:
{'OBJECTID': 994671, 'IDENTITY_': 'HIMAL450', 'Litho': 'pa____', 'xx': 'pa', 'yy': '__', 'zz': '__', 'xx_Description': 'Acid plutonic rocks', 'yy_Description': None, 'zz_Description': None, 'Shape__Area': 63760242.1953125, 'Shape__Length': 35196.19584695297}


In [43]:

# Inspect points that fall outside both state boundaries
unclassified = df[
    df["STATE_EXPERIMENT"] == "Unclassified"
].copy()

print("Unclassified points:", len(unclassified))
print("\nCoordinate ranges:")
display(
    unclassified[["latitude", "longitude"]].agg(
        ["min", "max", "mean"]
    )
)

print("\nFirst 20 unclassified points:")
display(
    unclassified[
        ["latitude", "longitude", "reference_date", "NEXT_1D"]
    ].head(20)
)

print("\nOriginal state-assignment counts:")
print(df["STATE_EXPERIMENT"].value_counts(dropna=False))


Unclassified points: 24

Coordinate ranges:


,latitude,longitude
min,28.866500,75.664927
max,32.941748,80.565222
mean,30.580794,77.990087



First 20 unclassified points:


,latitude,longitude,reference_date,NEXT_1D
0,28.866500,79.568200,2011-08-11,1
1,28.866500,79.568200,2011-08-10,0
2,28.866500,79.568200,2011-08-09,0
102,29.888847,80.565222,2017-07-04,1
103,29.888847,80.565222,2017-07-03,0
104,29.888847,80.565222,2017-07-02,0
186,30.239361,77.955817,2017-07-12,1
187,30.239361,77.955817,2017-07-11,0
188,30.239361,77.955817,2017-07-10,0
630,30.914100,76.818000,2010-10-02,1



Original state-assignment counts:
STATE_EXPERIMENT
Uttarakhand         1137
Himachal Pradesh    1089
Unclassified          24
Name: count, dtype: int64


In [45]:

# Count unique locations among unclassified rows
unique_unclassified = (
    unclassified[["latitude", "longitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unclassified rows:", len(unclassified))
print("Unique unclassified locations:", len(unique_unclassified))

# Show every unique location and how many records use it
location_counts = (
    unclassified.groupby(["latitude", "longitude"])
    .agg(
        rows=("reference_date", "size"),
        first_date=("reference_date", "min"),
        last_date=("reference_date", "max"),
        positive_next_1d=("NEXT_1D", "sum")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(location_counts)


Unclassified rows: 24
Unique unclassified locations: 16


,latitude,longitude,rows,first_date,last_date,positive_next_1d
0,28.866500,79.568200,3,2011-08-09,2011-08-11,1
3,29.888847,80.565222,3,2017-07-02,2017-07-04,1
6,30.239361,77.955817,3,2017-07-10,2017-07-12,1
8,30.914100,76.818000,3,2010-09-30,2010-10-02,1
1,29.379017,78.834652,1,2013-07-05,2013-07-05,0
2,29.616323,78.540152,1,2009-09-02,2009-09-02,0
4,29.987106,77.742535,1,2013-06-14,2013-06-14,0
5,30.207800,77.905851,1,2011-07-07,2011-07-07,0
7,30.673582,77.154751,1,2015-07-08,2015-07-08,0
9,31.072415,76.615185,1,2009-09-10,2009-09-10,0


In [46]:

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

# 1. Keep only confidently assigned Uttarakhand / Himachal rows.
# This does not modify the original CSV.
data = df[
    df["STATE_EXPERIMENT"].isin(["Uttarakhand", "Himachal Pradesh"])
].copy()

data["reference_date"] = pd.to_datetime(data["reference_date"])
data = data.dropna(subset=["reference_date", "NEXT_1D"])
data["NEXT_1D"] = data["NEXT_1D"].astype(int)

uk_mask = data["STATE_EXPERIMENT"].eq("Uttarakhand")

print("Rows excluded:", len(df) - len(data))
print("Rows retained:", len(data))
print("State counts:")
print(data["STATE_EXPERIMENT"].value_counts())

# 2. Make chronological splits once, shared by both experiments.
dates = np.array(sorted(data["reference_date"].unique()))
if len(dates) < 10:
    raise ValueError("Too few distinct dates to create reliable splits.")

train_end = dates[int(len(dates) * 0.60) - 1]
val_end = dates[int(len(dates) * 0.80) - 1]

train_period = data["reference_date"] <= train_end
val_period = (
    (data["reference_date"] > train_end)
    & (data["reference_date"] <= val_end)
)
test_period = data["reference_date"] > val_end

train_uk = data[train_period & uk_mask].copy()
train_both = data[train_period].copy()
val_uk = data[val_period & uk_mask].copy()
test_uk = data[test_period & uk_mask].copy()

print("\nDate cutoffs:")
print("Training through:", pd.Timestamp(train_end).date())
print("Validation through:", pd.Timestamp(val_end).date())
print("\nTraining samples — Uttarakhand:", len(train_uk))
print("Training samples — both states:", len(train_both))
print("Shared UK validation samples:", len(val_uk))
print("Shared UK test samples:", len(test_uk))

if min(len(train_uk), len(train_both), len(val_uk), len(test_uk)) == 0:
    raise ValueError("A split is empty; inspect the date/state distribution.")

if val_uk["NEXT_1D"].nunique() < 2 or test_uk["NEXT_1D"].nunique() < 2:
    raise ValueError("Validation or test set has only one target class.")

# 3. Identical features and model settings.
numeric = [
    "latitude", "longitude",
    "RAIN_1D", "RAIN_3D", "RAIN_7D", "RAIN_15D", "RAIN_30D",
    "ELEVATION", "SLOPE", "LANDCOVER_CODE"
]
categorical = ["LITHOLOGY"]
features = numeric + categorical
target = "NEXT_1D"

missing = [c for c in features + [target] if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

def build_model():
    preprocessing = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical)
    ])

    return Pipeline([
        ("preprocess", preprocessing),
        ("rf", RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight={0: 1, 1: 4},
            random_state=42,
            n_jobs=-1
        ))
    ])

# 4. Select threshold on the same UK validation data.
def choose_threshold(y, probabilities, target_recall=0.80):
    candidates = np.unique(np.r_[0.0, probabilities])
    eligible = []

    for threshold in candidates:
        predictions = (probabilities >= threshold).astype(int)
        rec = recall_score(y, predictions, zero_division=0)
        if rec >= target_recall:
            eligible.append(threshold)

    return max(eligible) if eligible else 0.0

def run_experiment(name, training_data):
    model = build_model()
    model.fit(training_data[features], training_data[target])

    val_prob = model.predict_proba(val_uk[features])[:, 1]
    threshold = choose_threshold(val_uk[target].to_numpy(), val_prob)

    test_prob = model.predict_proba(test_uk[features])[:, 1]
    y = test_uk[target].to_numpy()
    pred = (test_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, pred, labels=[0, 1]
    ).ravel()

    return {
        "experiment": name,
        "training_rows": len(training_data),
        "UK_test_rows": len(test_uk),
        "validation_threshold": threshold,
        "test_positive_rate": y.mean(),
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y, test_prob),
        "PR_AUC": average_precision_score(y, test_prob),
        "TP": int(tp),
        "FN": int(fn),
        "FP": int(fp),
        "TN": int(tn),
        "predicted_alert_rate": pred.mean()
    }

# 5. Run both experiments on the same UK validation/test samples.
results = [
    run_experiment("Uttarakhand only", train_uk),
    run_experiment("Uttarakhand + Himachal Pradesh", train_both)
]

results_df = pd.DataFrame(results)
display(results_df.round(4))

# Save to a NEW file only.
output_path = Path("../data/state_model_comparison_experiment.csv")
results_df.to_csv(output_path, index=False)
print("Saved comparison to:", output_path)
print("No original dataset was overwritten.")


Rows excluded: 24
Rows retained: 2226
State counts:
STATE_EXPERIMENT
Uttarakhand         1137
Himachal Pradesh    1089
Name: count, dtype: int64

Date cutoffs:
Training through: 2014-07-27
Validation through: 2016-08-06

Training samples — Uttarakhand: 797
Training samples — both states: 1546
Shared UK validation samples: 191
Shared UK test samples: 149


,experiment,training_rows,UK_test_rows,validation_threshold,test_positive_rate,accuracy,precision,recall,f1,ROC_AUC,PR_AUC,TP,FN,FP,TN,predicted_alert_rate
0,Uttarakhand only,797,149,0.3440,0.1342,0.6846,0.2453,0.65,0.3562,0.7554,0.3051,13,7,40,89,0.3557
1,Uttarakhand + Himachal Pradesh,1546,149,0.3504,0.1342,0.7248,0.2941,0.75,0.4225,0.7849,0.3469,15,5,36,93,0.3423


Saved comparison to: ..\data\state_model_comparison_experiment.csv
No original dataset was overwritten.
